#I) Tiền xử lý dữ liệu

##1\. Đọc dữ liệu vào




In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
!pip install ucimlrepo
warnings.filterwarnings('ignore')
from scipy.stats import pearsonr
from sklearn.decomposition import PCA

SEP = "_" * 72

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 130)
pd.set_option('display.float_format', '{:.3f}'.format)
print(f"\n{SEP}")

from ucimlrepo import fetch_ucirepo
# Tải dữ liệu trực tiếp từ repo UCI
support2 = fetch_ucirepo(id=880)
df_raw = pd.concat([support2.data.features, support2.data.targets], axis=1)

# In ra cấu trúc tổng quan gồm số hàng, cột, danh sách cột
N_ROWS, N_COLS = df_raw.shape
print(f"Number of Observations: {N_ROWS}")
print(f"Number of Features: {N_COLS}")
print(f"Features: \n{df_raw.columns.tolist()}\n")

ValueError: mount failed

Mô tả cấu trúc dữ liệu

In [ ]:
import subprocess, os, shutil
from google.colab import drive

# Unmount FUSE trước
subprocess.run(['fusermount', '-uz', '/content/drive'], capture_output=True)

# Xóa thư mục
shutil.rmtree('/content/drive', ignore_errors=True)

# Mount lại
drive.mount('/content/drive')

In [ ]:
# Định nghĩa N_ROWS, N_COLS và in ra cấu trúc tổng quan
N_ROWS, N_COLS = df_raw.shape
print(f"Tổng số bản ghi (Rows): {N_ROWS}")
print(f"Tổng số trường (Columns): {N_COLS}")
print(f"Danh sách các trường:\n{df_raw.columns.tolist()}\n")

Thống kê sơ bộ

In [ ]:
# Thống kê sơ bộ
num_cols = df_raw.select_dtypes(include='number').columns.tolist()
obj_cols = df_raw.select_dtypes(include='object').columns.tolist()

print(f"\n THỐNG KÊ MÔ TẢ — NUMERICAL ({len(num_cols)} cột)")
desc = df_raw[num_cols].describe().T
desc['skew']   = df_raw[num_cols].skew().round(3)
desc['null%']  = (df_raw[num_cols].isna().sum() / N_ROWS * 100).round(1)
print(desc[['count','mean','std','min','50%','max','skew','null%']].to_string())

print(f"\n THỐNG KÊ MÔ TẢ — CATEGORY ({len(obj_cols)} cột)")
for col in obj_cols:
    vc = df_raw[col].value_counts(dropna=False)
    print(f"\n  ▸ {col}  (unique={df_raw[col].nunique()}, null={df_raw[col].isna().sum()})")
    print(vc.head(6).to_string())

##2\. Làm sạch dữ liệu và xử lý dữ liệu lỗi

In [ ]:
print(f"\n{SEP}")
print("  PHẦN 2 — LÀM SẠCH VÀ XỬ LÝ DỮ LIỆU LỖI")
print(SEP)

# Tạo bản sao để làm sạch
df = df_raw.copy()

# [2.0] Xóa các hàng thiếu trường 'charge'
n_missing_target = df['charges'].isna().sum()
df.dropna(subset=['charges'], inplace=True)
print(f"[2.0] Đã xóa {n_missing_target} bản ghi bị thiếu biến mục tiêu 'charges'")

# [2.1] Xóa các biến gây rò rỉ dữ liệu
# surv2m, surv6m, prg2m, prg6m là kết quả dự đoán của chính mô hình dataset SUPPORT2 hoặc của bác sĩ, không có ý nghĩa huấn luyện
# totcst, totmcst là tổng chi phí, nếu sử dụng chính 2 biến này để dự đoán chi phí sẽ gây rò rỉ dữ liệu
# death, hospdead là kết quả kết quả cuối cùng, cho biết bệnh nhân sống hay chết
# sfdm2 cho biết tình trạng bệnh nhân trong và sau 2 tháng, cũng là kết quả cuối cùng
leakage_cols = ['death', 'hospdead', 'sfdm2', 'totcst', 'totmcst', 'surv2m', 'surv6m', 'prg2m', 'prg6m']
cols_to_drop = [c for c in leakage_cols if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)
print(f"\n[2.1] Đã xóa {len(cols_to_drop)} cột gây rò rỉ dữ liệu")

# [2.2] Bản ghi trùng lặp
n_dup = df.duplicated().sum()
df.drop_duplicates(keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"\n[2.2] Bản ghi trùng lặp: phát hiện {n_dup:,}, xóa → còn {len(df):,} hàng")

# [2.3] Chuẩn hóa giá trị cột phân loại (inconsistency)
SEX_MAP = {'male':'male','Male':'male','MALE':'male','m':'male','M':'male',
           'female':'female','Female':'female','FEMALE':'female','f':'female','F':'female'}
CA_MAP  = {'yes':'yes','Yes':'yes','YES':'yes',
           'no':'no','No':'no','NO':'no',
           'metastatic':'metastatic','Metastatic':'metastatic'}

if 'sex' in df.columns: df['sex'] = df['sex'].astype(str).str.strip().map(SEX_MAP)
if 'ca' in df.columns:  df['ca']  = df['ca'].astype(str).str.strip().map(CA_MAP)

for col in ['race','income','dzgroup','dzclass']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
        df[col] = df[col].replace({'nan':np.nan,'none':np.nan,'':np.nan})

print(f"\n[2.3] Chuẩn hóa phân loại (xử lý hoa/thường, khoảng trắng)")

# [2.4] Giá trị ngoài khoảng y khoa (không đúng về mặt giới hạn cơ thể con người)
VALID_RANGE = {
    'age':    (18, 120),  'meanbp': (10, 300),
    'temp':   (25,  42),  'hrt':    (20, 300),
    'resp':   ( 4,  60),  'ph':     (6.5, 8.0),
    'sod':    (100, 180)
}
print(f"\n[2.4] Giá trị ngoài khoảng hợp lệ → NaN:")
for col, (lo, hi) in VALID_RANGE.items():
    if col not in df.columns: continue
    df[col] = pd.to_numeric(df[col], errors='coerce')
    mask = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
    n = mask.sum()
    if n: df.loc[mask, col] = np.nan
    print(f"      {col:<10}: {n:4d} giá trị ngoài [{lo}, {hi}] → NaN")

# [2.5] Xử lý Missing Values
print(f"\n[2.5] Xử lý giá trị thiếu:")

# a. Điền theo khuyến nghị y khoa (Prof. Frank Harrell)
HARRELL_FILL = {
    'alb': 3.5, 'pafi': 333.3, 'bili': 1.01,
    'crea': 1.01, 'bun': 6.51, 'wblc': 9.0, 'urine': 2502.0
}
for col, val in HARRELL_FILL.items():
    if col in df.columns:
        n_fill = df[col].isna().sum()
        if n_fill > 0:
            df[col].fillna(val, inplace=True)
            print(f"      '{col:<5}': {n_fill:5,} NaN → domain expert = {val}")

# b. Xử lý các cột còn lại
THRESHOLD_DROP = 0.50
SKEW_THRESHOLD = 1.0
cols_dropped_na = []

for col in df.columns:
    if col == 'charges': continue # không xử lý cột của biến mục tiêu

    miss_pct = df[col].isna().mean()
    if miss_pct == 0: continue

    # Xóa cột nếu thiếu quá nhiều
    if miss_pct > THRESHOLD_DROP:
        cols_dropped_na.append(col)
        print(f"      xóa cột '{col}': thiếu {miss_pct*100:.1f}%")
        continue

    # Impute số
    if df[col].dtype in ['float64','int64']:
        skew = df[col].skew()
        if abs(skew) > SKEW_THRESHOLD:
            fill_val = df[col].median()
            method   = f'median={fill_val:.3g}'
        else:
            fill_val = df[col].mean()
            method   = f'mean={fill_val:.3g}'
    # Impute category
    else:
        mode_s = df[col].mode()
        fill_val = mode_s[0] if len(mode_s) else 'unknown'
        method   = f'mode="{fill_val}"'

    n_fill = df[col].isna().sum()
    df[col].fillna(fill_val, inplace=True)
    print(f"      '{col:<5}': {n_fill:5,} NaN → {method}")

df.drop(columns=cols_dropped_na, inplace=True, errors='ignore')

# Ép kiểu sau khi làm sạch
for col in ['age','num.co']:
    if col in df.columns:
        df[col] = df[col].fillna(0).round().astype(int)

for col in ['sex','race','income','dzgroup','dzclass','ca']:
    if col in df.columns:
        df[col] = df[col].astype('category')

print(f"\n      - Sau làm sạch: {df.shape[0]:,} hàng × {df.shape[1]} cột")
print(f"      - Giá trị thiếu trên toàn bộ Dataset: {df.isna().sum().sum()}")

## 3\. Encoding

In [ ]:
print(f"\n{SEP}")
print("  PHẦN 3 — ENCODING")
print(SEP)

"""
'sex': 2 giá trị → ánh xạ về 0/1.
'ca' (Ung thư): Có thứ tự tiến triển y khoa (Không < Có < Di căn) → 0, 1, 2
"""
df_enc = df.copy()
BINARY_MAP = {
    'sex':   {'male': 0, 'female': 1},
    'ca':    {'no': 0, 'yes': 1, 'metastatic': 2},
}

print("\n[3.1] Binary / Ordinal Encoding (Sex, CA):")
for col, mapping in BINARY_MAP.items():
    if col in df_enc.columns:
        # Ánh xạ và ép kiểu ép buộc về dạng số (float/int)
        df_enc[col] = pd.to_numeric(df_enc[col].map(mapping), errors='coerce')
        print(f"      {col}: {mapping}")
# ── 3.2 Ordinal Encoding (Thu nhập) ──────────────────────────────────
# Cột thu nhập có thứ tự tự nhiên.
ORDINAL_MAP = {
    'income': {'under $11k': 0, '$11-$25k': 1, '$25-$50k': 2, '>$50k': 3},
}
print("\n[3.2] Ordinal Encoding (Income):")
for col, mapping in ORDINAL_MAP.items():
    if col in df_enc.columns:
      # Ánh xạ và ép kiểu ép buộc về dạng số (float/int)
        df_enc[col] = pd.to_numeric(df_enc[col].map(mapping), errors='coerce')
        print(f"      {col}: {mapping}")

# ── 3.3 One-Hot Encoding (Nominal không có thứ tự) ───────────────────
"""
Các cột phân loại không có thứ tự (dzgroup, dzclass, race).
Sử dụng drop_first=True để tránh đa cộng tuyến.
"""
OHE_COLS = ['race', 'dzgroup', 'dzclass', 'dnr']
OHE_COLS = [c for c in OHE_COLS if c in df_enc.columns]

print(f"\n[3.3] One-Hot Encoding: {OHE_COLS}")
df_before_ohe = df_enc.shape[1]

for col in OHE_COLS:
    dummies = pd.get_dummies(df_enc[col], prefix=col, drop_first=True, dtype=int)
    df_enc  = pd.concat([df_enc.drop(columns=[col]), dummies], axis=1)
    print(f"      '{col}' → {dummies.shape[1]} cột mới")

print(f"\n      Số cột: {df_before_ohe} → {df_enc.shape[1]} (thêm {df_enc.shape[1]-df_before_ohe} cột OHE)")

# ── 3.4 Kiểm tra kết quả encoding ────────────────────────────────────
remaining_obj = df_enc.select_dtypes(include=['object','category']).columns.tolist()
print(f"\n[3.4] Kiểm tra: còn {len(remaining_obj)} cột object/category")
if remaining_obj:
    print(f"      - Cần xử lý các cột sau: {remaining_obj}")
else:
    print(f"      - Tất cả cột đã là dạng số.")

print(f"\n      Kích thước sau khi encoding: {df_enc.shape}")

## 4\. Chuẩn hóa dữ liệu

In [ ]:
import numpy as np
import json

print(f"\n{SEP}")
print("  PHẦN 4 — CHUẨN HÓA DỮ LIỆU")
print(SEP)

df_scaled = df_enc.copy()
TARGET_COLS  = ['charges']

# Tìm các cột nhị phân (0/1) gốc và các cột sinh ra từ One-Hot Encoding
BINARY_FLAGS = [c for c in df_scaled.columns if df_scaled[c].dtype not in ["object","category"] and df_scaled[c].nunique() == 2 and df_scaled[c].min() == 0 and df_scaled[c].max() == 1]
OHE_NEW_COLS = [c for c in df_scaled.columns if any(c.startswith(p + '_') for p in OHE_COLS)]
ORDINAL_COLS = ['income', 'ca']

# Tập hợp các cột KHÔNG DÙNG Z-score
SKIP_SCALE   = set(TARGET_COLS + BINARY_FLAGS + OHE_NEW_COLS + ORDINAL_COLS)
SCALE_COLS   = [c for c in df_scaled.select_dtypes(include='number').columns if c not in SKIP_SCALE]

print(f"\n▸ Cột cần scale (Z-score): {len(SCALE_COLS)}")
print(f"▸ Cột bỏ qua (Không Z-score): {len(SKIP_SCALE)} (target + Categorical)")

# 4.1 Log transform cho biến mục tiêu và các cột lệch phải
"""
Log1p(x) = log(x + 1) an toàn với x=0, giúp giảm độ lệch phải (skewness).
Áp dụng cho charges (Target) và các biến sinh lý có độ lệch lớn.
"""
LOG_COLS = ['charges', 'bili', 'crea', 'bun', 'wblc', 'urine', 'sps', 'aps']
LOG_COLS = [c for c in LOG_COLS if c in df_scaled.columns]

print(f"\n[4.1] Log1p transform (giảm lệch phải):")
for col in LOG_COLS:
    skew_before = df_scaled[col].skew()
    df_scaled[col] = np.log1p(df_scaled[col].clip(lower=0))
    skew_after = df_scaled[col].skew()
    print(f"      {col:<14}: skew {skew_before:+.2f} → {skew_after:+.2f}")

# 4.2 Z-score Standardization
"""
Z-score: (x - mean) / std → Đưa về trung bình 0, độ lệch chuẩn 1.
Chỉ áp dụng cho các cột liên tục trong SCALE_COLS (không scale y).
"""
print(f"\n[4.2] Z-score Standardization:")
scale_params = {}
for col in SCALE_COLS:
    if col not in df_scaled.columns: continue

    mu  = df_scaled[col].mean()
    sig = df_scaled[col].std()

    if sig == 0: continue   # tránh chia 0 cho cột hằng số

    df_scaled[col] = (df_scaled[col] - mu) / sig
    scale_params[col] = {'mean': round(mu,4), 'std': round(sig,4)}
    print(f"      {col:<14}: mean={mu:8.3f}, std={sig:7.3f} → z-score")

# 4.3 Lưu thông số scale
try:
    with open('/content/drive/MyDrive/project_ml/data/scale_params.json', 'w') as f:
        json.dump(scale_params, f, indent=2)
    print(f"\n      ✔ Lưu scale_params.json thành công ({len(scale_params)} cột)")
except FileNotFoundError:
    print(f"\n      [CẢNH BÁO] Không tìm thấy đường dẫn lưu file json. Vui lòng kiểm tra lại Google Drive.")

##5\. Mô tả dữ liệu sau chuẩn hóa

In [ ]:
print(f"\n{SEP}")
print("  PHẦN 5 — MÔ TẢ DỮ LIỆU SAU CHUẨN HÓA")
print(SEP)

print(f"\n▸ Shape  : {df_scaled.shape[0]:,} hàng × {df_scaled.shape[1]} cột")
print(f"▸ Missing: {df_scaled.isna().sum().sum()} (phải = 0)")
print(f"▸ Dtypes : {dict(df_scaled.dtypes.value_counts())}")

# Thống kê mô tả sau scale (chỉ cột numeric đã scale)
SAMPLE_SCALED = [c for c in SCALE_COLS if c in df_scaled.columns][:12]
print(f"\n Thống kê mô tả sau chuẩn hóa (chọn {len(SAMPLE_SCALED)} cột đại diện):")
desc_scaled = df_scaled[SAMPLE_SCALED].describe().T
print(desc_scaled[['mean','std','min','25%','50%','75%','max']].round(3).to_string())

print("\n  → Sau Z-score: mean ≈ 0, std ≈ 1 với tất cả cột số (đặc trưng X)")
print("  → OHE columns vẫn là 0/1 nguyên")
print("  → ORDINAL columns giữ nguyên")
print("  → Target ('charges') ở dạng Log1p")

# ── Thống kê cột One-hot Encoding ─────────────────────────────────────────────────
print(f"\n Cột One-Hot Encoding (tỷ lệ = 1):")
ohe_existing = [c for c in OHE_NEW_COLS if c in df_scaled.columns]
for col in ohe_existing:
    pct = df_scaled[col].mean() * 100
    print(f"  {col:<40}: {pct:5.1f}%")

# ── Phân phối biến target (Charges dạng Log) ────────────────────────
print(f"\n Phân phối biến mục tiêu (charges - dạng Log1p):")
if 'charges' in df_scaled.columns:
    bins = pd.cut(df_scaled['charges'], bins=5)
    print(bins.value_counts().sort_index().to_string())

##6\. Tổng hợp toàn bộ quy trình

In [ ]:
C = {'blue':'#2563EB','red':'#DC2626','green':'#16A34A',
     'amber':'#D97706','gray':'#6B7280','bg':'#F8FAFC','dark':'#1E293B',
     'purple':'#7C3AED','teal':'#0D9488'}

plt.rcParams.update({'figure.facecolor':C['bg'],'axes.facecolor':C['bg'],
                     'axes.spines.top':False,'axes.spines.right':False,
                     'axes.titlesize':11,'axes.labelsize':9,'font.family':'DejaVu Sans'})

fig = plt.figure(figsize=(22, 26), facecolor=C['bg'])
fig.suptitle("SUPPORT2 — Phân tích & Chuẩn hóa Dữ liệu (Mục tiêu: charges)", fontsize=17,
             fontweight='bold', color=C['dark'], y=0.99)

gs = gridspec.GridSpec(5, 3, figure=fig, hspace=0.50, wspace=0.38)

# (1) Tỷ lệ missing gốc
ax1 = fig.add_subplot(gs[0, :2])
miss = (df_raw.isna().sum() / len(df_raw) * 100).sort_values(ascending=False)
miss = miss[miss > 0]
colors_bar = [C['red'] if v > 30 else C['amber'] if v > 10 else C['blue'] for v in miss]
bars = ax1.barh(miss.index[::-1], miss.values[::-1], color=colors_bar[::-1], alpha=0.85)
ax1.axvline(50, color=C['red'], ls='--', lw=1.2, alpha=0.7, label='Ngưỡng xóa cột (50%)')
ax1.set_xlabel("Tỷ lệ thiếu (%)")
ax1.set_title("Tỷ lệ giá trị thiếu (dữ liệu gốc)", fontweight='bold')
ax1.legend(fontsize=8)
for bar, val in zip(bars, miss.values[::-1]):
    ax1.text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=7.5)

# (2) Phân phối Ung thư (ca) thay vì death
ax2 = fig.add_subplot(gs[0, 2])
if 'ca' in df.columns:
    ca_vc = df['ca'].value_counts()
    ax2.pie(ca_vc, labels=ca_vc.index, colors=[C['green'], C['amber'], C['red']],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2}, textprops={'fontsize':10})
    ax2.set_title("Phân phối ung thư (ca)", fontweight='bold')

# (3) Trước/Sau log transform cho charges
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])
raw_ch = df_raw['charges'].dropna()
log_ch = np.log1p(raw_ch.clip(lower=0))
ax3.hist(raw_ch, bins=60, color=C['amber'], edgecolor='white', alpha=0.85)
ax3.set_title(f"charges (gốc, skew={raw_ch.skew():.2f})", fontweight='bold')
ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
ax4.hist(log_ch, bins=60, color=C['teal'], edgecolor='white', alpha=0.85)
ax4.set_title(f"charges sau log1p (skew={log_ch.skew():.2f})", fontweight='bold')

# (4) Trước/Sau Z-score (age)
ax5 = fig.add_subplot(gs[1, 2])
age_raw = df['age'].dropna()
age_z   = (age_raw - age_raw.mean()) / age_raw.std()
ax5.hist(age_raw, bins=35, color=C['purple'], edgecolor='white', alpha=0.6, label='Gốc')
ax5_r = ax5.twinx()
ax5_r.hist(age_z, bins=35, color=C['blue'], edgecolor='white', alpha=0.5, label='Z-score')
ax5.set_title("age: gốc vs Z-score", fontweight='bold')
ax5.set_xlabel("Tuổi / Z-score")
lines1,_ = ax5.get_legend_handles_labels()
lines2,_ = ax5_r.get_legend_handles_labels()
ax5.legend(lines1+lines2, ['Gốc','Z-score'], fontsize=8, loc='upper left')

# (5) Correlation heatmap (Đã dọn dẹp các biến leakage)
ax6 = fig.add_subplot(gs[2, :2])
corr_cols = ['age','meanbp','hrt','resp','temp','charges','los','bun','crea','wblc']
corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax6, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5,
            annot_kws={'size':8}, cbar_kws={'shrink':0.8})
ax6.set_title("Ma trận tương quan (Loại bỏ rò rỉ dữ liệu)", fontweight='bold')
ax6.tick_params(axis='x', rotation=40, labelsize=8)
ax6.tick_params(axis='y', rotation=0,  labelsize=8)

# (6) Phân phối nhóm bệnh
ax7 = fig.add_subplot(gs[2, 2])
if 'dzgroup' in df.columns:
    dz = df['dzgroup'].value_counts()
    ax7.barh(dz.index[::-1], dz.values[::-1], color=C['purple'], alpha=0.8)
    ax7.set_title("Phân phối nhóm bệnh (dzgroup)", fontweight='bold')
    ax7.set_xlabel("Số bệnh nhân")

# (7) Boxplot Z-score sau scale
ax8 = fig.add_subplot(gs[3, :])
box_cols = [c for c in ['age','meanbp','hrt','resp','alb','sod','ph'] if c in df_scaled.columns]
box_data = [df_scaled[c].dropna().values for c in box_cols]
bp = ax8.boxplot(box_data, labels=box_cols, patch_artist=True,
                 medianprops={'color':C['red'],'linewidth':2},
                 flierprops={'marker':'o','markersize':2,'alpha':0.3})
palette = [C['blue'],C['teal'],C['purple'],C['amber'],C['green'],C['red'],C['gray']]
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color); patch.set_alpha(0.65)
ax8.axhline(0, color=C['dark'], ls='--', lw=0.8, alpha=0.5, label='mean=0')
ax8.axhline(1, color=C['gray'], ls=':', lw=0.8, alpha=0.5, label='±1 std')
ax8.axhline(-1,color=C['gray'], ls=':', lw=0.8, alpha=0.5)
ax8.set_title("Phân phối sau Z-score Standardization (mean≈0, std≈1)", fontweight='bold')
ax8.set_ylabel("Z-score"); ax8.legend(fontsize=8)

# (8) OHE proportion bar chart
ax9 = fig.add_subplot(gs[4, :])
ohe_pct = {c: df_scaled[c].mean()*100 for c in ohe_existing if c in df_scaled.columns}
if ohe_pct:
    ohe_series = pd.Series(ohe_pct).sort_values(ascending=False)
    ax9.bar(range(len(ohe_series)), ohe_series.values, color=C['teal'], alpha=0.8, edgecolor='white')
    ax9.set_xticks(range(len(ohe_series)))
    ax9.set_xticklabels(ohe_series.index, rotation=40, ha='right', fontsize=8)
    ax9.set_ylabel("Tỷ lệ = 1 (%)")
    ax9.set_title("Tỷ lệ giá trị 1 trong các cột One-Hot Encoding", fontweight='bold')

# Xử lý lưu file ảnh an toàn (tránh lỗi nếu chưa có thư mục)
# Xử lý lưu file ảnh an toàn (tránh lỗi nếu chưa có thư mục)
import os
save_dir = '/content/drive/MyDrive/project_ml/'
data_dir = os.path.join(save_dir, 'data') # Thư mục con chứa data
os.makedirs(save_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)      # Tạo thư mục data

plt.savefig(os.path.join(save_dir, 'support2_full_analysis.png'), dpi=120, bbox_inches='tight', facecolor=C['bg'])
plt.close()
print(f"\n✔ Lưu biểu đồ: support2_full_analysis.png")

# ── Lưu file ─────────────────────────────────────────────────────────
df.to_csv(os.path.join(data_dir, 'support2_cleaned.csv'), index=False)
df_enc.to_csv(os.path.join(data_dir, 'support2_encoded.csv'), index=False)
df_scaled.to_csv(os.path.join(data_dir, 'support2_final.csv'), index=False)
print(f"✔ support2_cleaned.csv  — {df.shape}")
print(f"✔ support2_encoded.csv  — {df_enc.shape}")
print(f"✔ support2_final.csv    — {df_scaled.shape}")

print(f"""
{SEP}
  QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU
{SEP}
  1. Đọc dữ liệu     : {N_ROWS:,} hàng × {N_COLS} cột gốc.
  2. Làm sạch        : - Đã xóa dòng thiếu 'charges' và xóa các cột rò rỉ dữ liệu.
                       - Xóa {n_dup} trùng lặp, chuẩn hóa categorical.
                       - Xử lý invalid range và Impute missing (y khoa + thống kê).
  3. Encoding        : Binary (sex, ca), Ordinal (income), One-Hot (race, dzgroup, dzclass).
  4. Scaling         : Log1p cho charges và {len(LOG_COLS)-1} cột lệch, Z-score cho {len(scale_params)} cột số.
  5. Kết quả cuối    : {df_scaled.shape[0]:,} hàng × {df_scaled.shape[1]} cột.
{SEP}
""")

#II) Phân tích và trực quan hóa dữ liệu

Khai báo thư viện và style toàn cục

In [ ]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import gaussian_kde
import umap
import itertools

# Style toàn cục
plt.rcParams.update({
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor':   '#FFFFFF',
    'axes.grid':        True,
    'grid.alpha':       0.25,
    'grid.linestyle':   '--',
    'font.size':        10,
    'axes.titlesize':   12,
    'axes.titleweight': 'bold',
    'axes.labelsize':   10,
    'xtick.labelsize':  8,
    'ytick.labelsize':  8,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
PALETTE  = ['#2E86AB', '#E84855', '#3BB273', '#F18F01', '#9B5DE5', '#00B4D8']
CMAP_DIV = 'RdBu_r'
CMAP_SEQ = 'viridis'
SEP      = "=" * 65
SEP_THIN = "-" * 65

df = df_scaled

X_scaled = df_scaled.drop(columns=["charges"])
y = df_scaled["charges"]
feat_names = list(X_scaled.columns)
N, D       = X_scaled.shape

## 1\. PCAmix

Địng nghĩa các hàm cần thiết

In [ ]:
import numpy as np
import pandas as pd

# Hàm PCA thủ công
class PCA_Manual:
    def __init__(self, n_components=None):
        """
        Khởi tạo hàm PCA thủ công chuẩn cấu trúc Scikit-Learn
        :param n_components: Số lượng chiều muốn giữ lại (Nếu None, giữ lại toàn bộ)
        """
        self.n_components = n_components
        self.components_ = None            # Ma trận loadings (n_components, n_features)
        self.explained_variance_ = None     # Các trị riêng (Eigenvalues) tương ứng
        self.explained_variance_ratio_ = None # Tỷ lệ thông tin bảo tồn
        self.mean_ = None                  # Giá trị trung bình của các biến gốc
        self.n_components_ = None          # Số chiều thực tế được chọn

    def fit(self, X, y=None):
        X_arr = np.asarray(X, dtype=float)
        n_samples, n_features = X_arr.shape

        # BƯỚC 1: Tính Mean và Chuẩn tâm dữ liệu (Centering)
        self.mean_ = np.mean(X_arr, axis=0)
        X_centered = X_arr - self.mean_

        # BƯỚC 2: Tính ma trận hiệp phương sai (Covariance Matrix)
        cov_matrix = np.cov(X_centered, rowvar=False)

        # BƯỚC 3: Tìm Trị riêng (Eigenvalues) và Vectơ riêng (Eigenvectors)
        # Sử dụng eigh vì ma trận đối xứng, giúp tính toán ổn định hơn
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # BƯỚC 4: Sắp xếp Trị riêng & Vectơ riêng giảm dần
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # BƯỚC 5: Xác định số lượng thành phần chính cần lấy
        if self.n_components is None:
            self.n_components_ = n_features
        else:
            self.n_components_ = min(self.n_components, n_features)

        # BƯỚC 6: Đóng gói thuộc tính đầu ra
        self.components_ = eigenvectors.T[:self.n_components_]
        self.explained_variance_ = eigenvalues[:self.n_components_]

        total_variance = np.sum(eigenvalues)
        self.explained_variance_ratio_ = eigenvalues[:self.n_components_] / total_variance

        return self

    def transform(self, X):
        X_arr = np.asarray(X, dtype=float)
        X_centered = X_arr - self.mean_
        return X_centered @ self.components_.T

    def fit_transform(self, X, y=None):
        self.fit(X, y)
        return self.transform(X)

# Hàm PCAmix thủ công: Sử dụng kết hợp PCA gốc cho dữ liệu numeric và MCA cho dữ liệu categeory
class PCAmix_Manual:
    def __init__(self, n_components=None, discrete_features=None, continuous_features=None):
        """
        Khởi tạo bộ lọc PCAmix tận dụng lõi PCA thủ công cho dữ liệu hỗn hợp
        :param discrete_features: List chứa index hoặc tên cột dữ liệu rời rạc
        :param continuous_features: List chứa index hoặc tên cột dữ liệu liên tục
        """
        self.n_components = n_components
        self.discrete_features = list(discrete_features) if discrete_features is not None else []
        self.continuous_features = list(continuous_features) if continuous_features is not None else []
        self.pca_core = PCA_Manual(n_components=n_components)

        self.stds_con_ = None
        self.categories_per_feature_ = {}
        self.p_k_per_feature_ = {}
    def _preprocess(self, X, is_fit=True):
        """
        Hàm tiền xử lý:
        - Biến liên tục: Giữ nguyên (vì đã được chuẩn hóa từ trước).
        - Biến phân loại: Xử lý trọng số MCA trực tiếp trên cột đã OHE.
        """
        n_samples = X.shape[0]
        X_trans = []

        if self.continuous_features is not None and len(self.continuous_features) > 0:
            X_cont = X[:, self.continuous_features].astype(float)
            X_trans.append(X_cont)

        if self.discrete_features is not None and len(self.discrete_features) > 0:
            X_disc = X[:, self.discrete_features].astype(float)

            # tính toán tần suất p_k
            if is_fit:
                self.p_k_ = np.mean(X_disc, axis=0)
                self.p_k_[self.p_k_ == 0] = 1e-8
                # Khởi tạo nhãn
                self.categories_per_feature_ = {idx: ['active'] for idx in self.discrete_features}

            # Áp dụng chuẩn hóa trọng số MCA cho cả Fit và Transform
            X_disc_scaled = (X_disc / np.sqrt(self.p_k_)) - np.sqrt(self.p_k_)
            X_trans.append(X_disc_scaled)

        # Ghép lại thành ma trận tổng hợp
        return np.hstack(X_trans)

    def fit(self, X, y=None):
        X_preprocessed = self._preprocess(X, is_fit=True)
        self.pca_core.fit(X_preprocessed)
        self.components_ = self.pca_core.components_
        self.explained_variance_ = self.pca_core.explained_variance_
        self.explained_variance_ratio_ = self.pca_core.explained_variance_ratio_
        return self

    def transform(self, X):
        X_preprocessed = self._preprocess(X, is_fit=False)
        return self.pca_core.transform(X_preprocessed)

    def fit_transform(self, X, y=None):
        self.fit(X, y)
        return self.transform(X)

Thực hiện giảm chiều

In [ ]:
import pickle
import os
import pandas as pd
import numpy as np

# 1. Chuẩn bị dữ liệu tổng hợp (Gộp tất cả các cột đặc trưng lại)
ALL_FEATURES = SCALE_COLS + list(set(BINARY_FLAGS + OHE_NEW_COLS + ORDINAL_COLS))
X_mixed = df_scaled[ALL_FEATURES].values

discrete_indices = [ALL_FEATURES.index(col) for col in set(BINARY_FLAGS + OHE_NEW_COLS + ORDINAL_COLS)]
continuous_indices = [ALL_FEATURES.index(col) for col in SCALE_COLS]

# chạy PCAmix trên toàn bộ dữ liệu
pcamix_model = PCAmix_Manual(n_components=25,
                             discrete_features=discrete_indices,
                             continuous_features=continuous_indices)

# Thực hiện giảm chiều
mixed_features = pcamix_model.fit_transform(X_mixed)

# --- TÍNH TOÁN VÀ IN THÔNG SỐ ĐẦU RA ---
total_variance_retained = np.sum(pcamix_model.explained_variance_ratio_) * 100

print(f"✔ Đã giảm số lượng đặc trưng từ {X_mixed.shape[1]} chiều xuống còn {mixed_features.shape[1]} chiều.")
print(f"✔ Tổng lượng thông tin (phương sai) được giữ lại: {total_variance_retained:.2f}%")
print("-" * 60)

# 3. Tạo DataFrame kết quả đồng nhất
pc_cols = [f'PC_mix_{i+1}' for i in range(mixed_features.shape[1])]
df_pcamix_res = pd.DataFrame(mixed_features, columns=pc_cols, index=df_scaled.index)
df_pcamix_final = pd.concat([df_pcamix_res, df_scaled[['charges']]], axis=1)

save_dir = '/content/drive/MyDrive/project_ml/data/'
os.makedirs(save_dir, exist_ok=True)

# Lưu file CSV kết quả và Model
df_pcamix_final.to_csv(os.path.join(save_dir, 'support2_PCAmix_25.csv'), index=False)
with open(os.path.join(save_dir, 'fitted_pcamix_25_model.pkl'), 'wb') as f:
    pickle.dump(pcamix_model, f)

print("✔ Đã lưu file CSV và lưu model thành công bằng pickle!")

Trực quan hóa PCAmix

In [ ]:
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
print("\n" + "="*60)
print("KHẢO SÁT VÀ TRỰC QUAN PCAMIX (TRÊN TOÀN BỘ BIẾN HỖN HỢP)")
print("="*60)

CMAP_SEQ = 'viridis'
y = df_scaled['charges'].values
X_scaled = df_scaled.drop(columns=['charges']) if 'charges' in df_scaled.columns else df_scaled

X_pca = mixed_features
explained = pcamix_model.explained_variance_ratio_
ev_m = pcamix_model.explained_variance_
cum_m = np.cumsum(explained)
D_num = X_pca.shape[1]
N = X_pca.shape[0]

# Định danh tên cột để vẽ Loadings
continuous_names = [ALL_FEATURES[i] for i in continuous_indices]
discrete_names = [ALL_FEATURES[j] for j in discrete_indices]
PREPROCESSED_COLS = continuous_names + discrete_names

# BẢNG 1: Thống kê phương sai dạng bảng ký tự đẹp
print(f"\n  {'PC':<6} {'Eigenvalue':>12} {'Var %':>10} {'Cộng dồn %':>13}")
print(f"  {'-'*45}")
for i in range(min(25, len(ev_m))):
    bar = '▓' * max(1, int(explained[i]*100/2))
    print(f"  PC{i+1:<4} {ev_m[i]:>12.4f} {explained[i]*100:>9.2f}%  {cum_m[i]*100:>11.2f}%  {bar}")

# BẢNG 2: Xuất Ma trận Tải (Loadings) từ cấu trúc hỗn hợp của PCAmix
loadings = pd.DataFrame(pcamix_model.components_,
                        index=[f'PC_mix_{i+1}' for i in range(D_num)],
                        columns=PREPROCESSED_COLS).T

print("\nTop 5 chiều gốc đóng góp lớn nhất vào mỗi PC:")
for col in loadings.columns:
    top5 = loadings[col].abs().nlargest(5)
    row = ' | '.join([f"{k}({loadings.at[k,col]:+.3f})" for k in top5.index])
    print(f"  {col}: {row}")

# BẢNG 3: Khảo sát tích lũy thông tin theo k cấu hình cố định
print("\nLượng thông tin bảo tồn (Explained Variance) — theo từng cấu hình k:")
print(f"  {'k':>4}  {'Var %':>10}  {'Nhận xét'}")
print(f"  {'-'*40}")
for k in [1, 2, 3, 4, 5, 6, 10, 15, 20, 25, min(30, N-1)]:
    if k > min(D_num, N): continue
    info = explained[:k].sum() * 100
    note = ("✓ Tốt (>=80%)" if info >= 80 else "△ TB (60-80%)" if info >= 60 else "✗ Thấp (<60%)")
    print(f"  {k:>4}  {info:>9.2f}%  {note}")


# =========================================================================
# BIỂU ĐỒ 1: SCREE PLOT DUAL AXIS + KHẢO SÁT K LÝ TƯỞNG
# =========================================================================
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
fig.suptitle("PCAmix — Phân tích thành phần phương sai giải thích", fontsize=13, fontweight='bold')

# Subplot 1: Scree Plot tích hợp trục kép
ax1 = axes[0]
color_bar = '#2E86AB'
ax1.set_xlabel('Thành phần chính (Principal Components)', fontweight='bold')
ax1.set_ylabel('Tỷ lệ phương sai giải thích (%)', color=color_bar, fontweight='bold')
ax1.bar(range(1, D_num + 1), explained * 100, color=color_bar, alpha=0.7, label='Phương sai riêng lẻ')
ax1.tick_params(axis='y', labelcolor=color_bar)
ax1.set_xticks(range(1, D_num + 1))
ax1.set_xticklabels(range(1, D_num + 1), fontsize=8, rotation=45)

ax2 = ax1.twinx()
color_line = '#E84855'
ax2.set_ylabel('Phương sai cộng dồn (%)', color=color_line, fontweight='bold')
ax2.plot(range(1, D_num + 1), cum_m * 100, color=color_line, marker='o', linewidth=2, markersize=4, label='Phương sai cộng dồn')
ax2.tick_params(axis='y', labelcolor=color_line)
ax2.axhline(60, color='orange', linestyle='--', alpha=0.6, label='Ngưỡng 60%')
ax2.axhline(80, color='green', linestyle='--', alpha=0.6, label='Ngưỡng 80%')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=8)
ax1.grid(axis='y', alpha=0.3, linestyle=':')
ax1.set_title("Scree Plot + Cumulative")

# Subplot 2: Biểu đồ cột phân bố màu sắc theo chất lượng thông tin k
ax = axes[1]
ks = [k for k in [1,2,3,4,5,6,10,15,20,25,min(30,N-1)] if k <= min(D_num,N)]
infos = [explained[:k].sum() * 100 for k in ks]
bar_colors = ['#E84855' if v<60 else '#F18F01' if v<80 else '#3BB273' for v in infos]
ax.bar([str(k) for k in ks], infos, color=bar_colors, alpha=0.85)
ax.axhline(80, color='#3BB273', linestyle='--', linewidth=1.5, label='Ngưỡng 80%')
ax.axhline(95, color='#9B5DE5', linestyle='--', linewidth=1.5, label='Ngưỡng 95%')
ax.set_xlabel("Số lượng thành phần lựa chọn (k)")
ax.set_ylabel("Tổng phương sai tích lũy (%)")
ax.set_title("Khảo sát lượng thông tin bảo tồn")
ax.legend(fontsize=8)

axes[2].set_visible(False)
plt.tight_layout()
plt.show()

# =========================================================================
# BIỂU ĐỒ 2: LOADINGS HEATMAP (Bản đồ nhiệt đóng góp - Trích xuất Top 15 PC)
# =========================================================================
fig, ax = plt.subplots(figsize=(14, 7))
loadings_top15 = loadings.iloc[:, :15]
sns.heatmap(loadings_top15, cmap='RdBu_r', vmax=1.0, vmin=-1.0, annot=False,
            linewidths=0.3, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title("Loadings Heatmap\n(Cơ cấu đóng góp của các biến vào Top 15 PC)", fontsize=12, fontweight='bold')
ax.set_xlabel("Thành phần chính (Principal Components)")
ax.set_ylabel("Các biến số gốc")
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

# =========================================================================
# BIỂU ĐỒ 3: TRỰC QUAN HÓA PHÂN TÁN TỪNG CẶP PCA (PC1 -> PC6 kèm Ellipse)
# =========================================================================
print("\nTrực quan hóa phân tán ma trận các cặp thành phần PCA...")
pairs_all = list(itertools.combinations(range(min(6, D_num)), 2))
ncols_p = 5
nrows_p = int(np.ceil(len(pairs_all) / ncols_p))

fig, axes = plt.subplots(nrows_p, ncols_p, figsize=(ncols_p*3.8, nrows_p*3.3))
fig.suptitle(f"PCAmix — Scatter Ellipse (PC1–PC{min(6, D_num)})", fontsize=13, fontweight='bold')
axes = axes.flatten()
sc_ref = None

for idx, (i, j) in enumerate(pairs_all):
    ax = axes[idx]
    sc = ax.scatter(X_pca[:, i], X_pca[:, j], c=y, cmap=CMAP_SEQ, alpha=0.65, s=15, edgecolors='none')
    sc_ref = sc
    pts = X_pca[:, [i, j]]
    mu = pts.mean(axis=0)
    cov2 = np.cov(pts.T)
    if cov2.ndim == 2 and not np.isnan(cov2).any():
        vals2, vecs2 = np.linalg.eigh(cov2)
        angle = np.degrees(np.arctan2(*vecs2[:, 1][::-1]))
        w, h = 2 * np.sqrt(np.abs(vals2))
        for nsig, al in [(1, 0.12), (2, 0.06)]:
            ell = Ellipse(mu, nsig*w, nsig*h, angle=angle, color='gray', alpha=al, fill=True, linewidth=0)
            ax.add_patch(ell)
        ell2 = Ellipse(mu, w, h, angle=angle, color='#2E86AB', fill=False, linewidth=1.2, linestyle='--')
        ax.add_patch(ell2)
    ax.set_xlabel(f"PC{i+1} ({explained[i]*100:.1f}%)", fontsize=8)
    ax.set_ylabel(f"PC{j+1} ({explained[j]*100:.1f}%)", fontsize=8)
    ax.set_title(f"PC{i+1} vs PC{j+1}", fontsize=9)

for idx in range(len(pairs_all), len(axes)):
    axes[idx].set_visible(False)

fig.colorbar(sc_ref, ax=axes[:len(pairs_all)], label='charges', shrink=0.4)
plt.tight_layout()
plt.show()

# =========================================================================
# BIỂU ĐỒ 4: PHÂN TÍCH TƯƠNG QUAN PCA VỚI BIẾN MỤC TIÊU (CHARGES)
# =========================================================================
print("\n" + "="*60)
print("PHÂN TÍCH TƯƠNG QUAN TUYẾN TÍNH VỚI ĐẦU RA (CHARGES)")
print("="*60)

corr_pca = []
for i in range(min(6, D_num)):
    r = np.corrcoef(X_pca[:, i], y)[0, 1]
    corr_pca.append(r)

raw_corrs = X_scaled.corrwith(pd.Series(y, name='charges')).sort_values(key=abs, ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
fig.suptitle("Mối tương quan hệ thống với đầu ra (charges)", fontsize=13, fontweight='bold')

# Trục 1: Tương quan của hệ thống PC đầu tiên với charges
ax = axes[0]
bar_c = ['#3BB273' if r>0 else '#E84855' for r in corr_pca]
bars = ax.bar([f'PC_mix_{i+1}' for i in range(len(corr_pca))], corr_pca, color=bar_c, alpha=0.85)
for bar, val in zip(bars, corr_pca):
    ax.text(bar.get_x()+bar.get_width()/2, val+(0.01 if val>=0 else -0.03), f'{val:.3f}', ha='center', va='bottom', fontsize=8)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(0.3, color='gray', linestyle='--', linewidth=1, label='±0.3')
ax.axhline(-0.3, color='gray', linestyle='--', linewidth=1)
ax.set_title("PCA components vs charges (Pearson r)")
ax.set_ylabel("Pearson r")
ax.legend(fontsize=8)

# Trục 2: Đường xu hướng Scatter đặc trưng PC1 vs charges
ax = axes[1]
ax.scatter(X_pca[:, 0], y, alpha=0.6, s=18, color='#2E86AB', edgecolors='none')
z = np.polyfit(X_pca[:, 0], y, 1)
xl = np.linspace(X_pca[:, 0].min(), X_pca[:, 0].max(), 100)
ax.plot(xl, np.poly1d(z)(xl), color='#E84855', linewidth=2, linestyle='--', label=f'Trendline (r={corr_pca[0]:.3f})')
ax.set_xlabel("PC1_mix_1")
ax.set_ylabel("charges")
ax.set_title(f"Scatter: PC1_mix_1 vs charges")
ax.legend(fontsize=8)

# Trục 3: Top 10 biến gốc tương quan mạnh nhất
ax = axes[2]
top10 = raw_corrs.head(10)
colors_fc = ['#3BB273' if v>0 else '#E84855' for v in top10.values]
ax.barh(range(len(top10)), top10.values, color=colors_fc, alpha=0.85)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10.index, fontsize=8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title("Top 10 chiều gốc tương quan charges")
ax.set_xlabel("Pearson r")

plt.tight_layout()
plt.show()

# =========================================================================
# BIỂU ĐỒ 5: BIỂU ĐỒ VECTƠ LỰC PCA BIPLOT (Scores + Loadings Arrows)
# =========================================================================
if D_num >= 2:
    fig, ax = plt.subplots(figsize=(9, 7))
    sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap=CMAP_SEQ, alpha=0.6, s=18, edgecolors='none')
    plt.colorbar(sc, ax=ax, label='charges')

    top8 = loadings.pow(2).sum(axis=1).nlargest(8).index
    scale = min(X_pca[:, 0].std(), X_pca[:, 1].std()) * 3
    for feat in top8:
        lx, ly = loadings.at[feat,'PC_mix_1']*scale, loadings.at[feat,'PC_mix_2']*scale
        ax.annotate('', xy=(lx, ly), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='#E84855', lw=1.8, alpha=0.85))
        ax.text(lx*1.08, ly*1.08, feat, fontsize=8, color='#E84855', alpha=0.9)

    ax.set_xlabel(f"PC1 ({explained[0]*100:.2f}%)")
    ax.set_ylabel(f"PC2 ({explained[1]*100:.2f}%)")
    ax.set_title("PCAmix Biplot — Scores + Loadings arrows (Top 8 đặc trưng gốc)", fontweight='bold')
    ax.axhline(0, color='gray', linewidth=0.6)
    ax.axvline(0, color='gray', linewidth=0.6)
    plt.tight_layout()
    plt.show()

##2\. UMAP

In [ ]:
print("\n" + "="*60)
print("UMAP ANALYSIS")
print("="*60)
m2_2d = umap.UMAP(n_components=2, random_state=42).fit_transform(X_scaled)
m2_3d = umap.UMAP(n_components=3, random_state=42).fit_transform(X_scaled)
lbl   = "UMAP"
kl_info = ""

#sub(f"Thống kê {lbl} components")
m2_df = pd.DataFrame(m2_2d, columns=[f'{lbl}1', f'{lbl}2'])
print(m2_df.describe().round(4).to_string())

#sub(f"Tương quan {lbl} với charges")
for i, col in enumerate(m2_df.columns):
    r = np.corrcoef(m2_df.iloc[:, i], y)[0, 1]
    print(f"  {col} vs charges: r = {r:+.4f}")

# Phân nhóm charges
q33    = np.percentile(y, 33)
q66    = np.percentile(y, 66)
groups = np.where(y < q33, 0, np.where(y < q66, 1, 2))
g_lbls = ['Thấp', 'Trung bình', 'Cao']
g_cols = ['#2E86AB', '#F18F01', '#E84855']

# Biểu đồ
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"{lbl} — Trực quan hóa phi tuyến", fontsize=13, fontweight='bold')

ax = axes[0]
sc = ax.scatter(m2_2d[:, 0], m2_2d[:, 1], c=y, cmap=CMAP_SEQ,
                alpha=0.7, s=20, edgecolors='none')
plt.colorbar(sc, ax=ax, label='charges')
ax.set_xlabel(f"{lbl}1")
ax.set_ylabel(f"{lbl}2")
ax.set_title(f"{lbl} 2D — tô màu theo charges")

ax = axes[1]
for g, (gl, gc) in enumerate(zip(g_lbls, g_cols)):
    mask = groups == g
    ax.scatter(m2_2d[mask, 0], m2_2d[mask, 1], label=f'{gl} (n={mask.sum()})',
               color=gc, alpha=0.65, s=20, edgecolors='none')
ax.set_xlabel(f"{lbl}1")
ax.set_ylabel(f"{lbl}2")
ax.set_title(f"{lbl} 2D — 3 nhóm charges")
ax.legend(fontsize=8)

ax = axes[2]
if N > 5:
    kde  = gaussian_kde(m2_2d.T)
    dens = kde(m2_2d.T)
    sc2  = ax.scatter(m2_2d[:, 0], m2_2d[:, 1], c=dens,
                      cmap='hot_r', alpha=0.7, s=20, edgecolors='none')
    plt.colorbar(sc2, ax=ax, label='Mật độ')
else:
    ax.scatter(m2_2d[:, 0], m2_2d[:, 1], color='#2E86AB', s=40)
ax.set_xlabel(f"{lbl}1")
ax.set_ylabel(f"{lbl}2")
ax.set_title(f"{lbl} — Mật độ KDE")

plt.tight_layout()
plt.show()

# Biểu đồ: UMAP pair plots
pair_data   = m2_3d[:, :3]
pair_labels = [f'{lbl}{i+1}' for i in range(3)]


pairs_m2 = list(itertools.combinations(range(3), 2))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"{lbl} — Scatter Plot cặp thành phần (3 components)",
             fontsize=13, fontweight='bold')
for idx, (i, j) in enumerate(pairs_m2):
    ax  = axes[idx]
    sc2 = ax.scatter(pair_data[:, i], pair_data[:, j], c=y,
                     cmap=CMAP_SEQ, alpha=0.65, s=15, edgecolors='none')
    plt.colorbar(sc2, ax=ax, shrink=0.85)
    ax.set_xlabel(pair_labels[i], fontsize=9)
    ax.set_ylabel(pair_labels[j], fontsize=9)
    ax.set_title(f"{pair_labels[i]} vs {pair_labels[j]}")
plt.tight_layout()
plt.show()

print("\n[Giải thích]")
print("- UMAP không có explained variance")
print("- Dùng để biểu diễn cấu trúc phi tuyến")
print("- PCA giữ ít thông tin → dữ liệu có thể phi tuyến")
print("- UMAP giúp biểu diễn cấu trúc phi tuyến tốt hơn")
print("- UMAP phù hợp hơn PCA để trực quan dataset này")

## 3\. So sánh 2 phương pháp

In [ ]:
import numpy as np
import pandas as pd

# ==============================================================================
# TỰ ĐỘNG KHỞI TẠO VÀ TÍNH CUM_EVR NẾU CHƯA CÓ ĐỂ TRÁNH LỖI NAMEERROR
# ==============================================================================
try:
    # Nếu hệ thống đã chạy qua bước PCA trước đó và có sẵn biến 'pca_full_manual'
    cum_evr = np.cumsum(pca_full_manual.explained_variance_ratio_)
    # Chuẩn hóa về dạng tỷ lệ phần trăm từ 0 -> 100%
    if cum_evr[0] <= 1.0:
        cum_evr = cum_evr * 100
except NameError:
    # Trường hợp bạn chạy cell này độc lập mà chưa chạy các cell PCA trước:
    # Tiến hành tính toán nhanh lại mảng phương sai giải thích tích lũy
    print("⚠ Cảnh báo: Không tìm thấy biến 'pca_full_manual'. Hệ thống đang tự động tính toán lại...")
    ALL_FEATURES = SCALE_COLS + list(set(BINARY_FLAGS + OHE_NEW_COLS + ORDINAL_COLS))
    X_input_temp = df_scaled[ALL_FEATURES].values

    pca_temp = PCA_Manual(n_components=17)
    pca_temp.fit(X_input_temp)
    cum_evr = np.cumsum(pca_temp.explained_variance_ratio_) * 100

# ==============================================================================
# IN BẢNG SO SÁNH PHƯƠNG PHÁP (ĐÃ ĐƯỢC ĐÓNG GÓI CHUẨN DATAFRAME)
# ==============================================================================
print("="*80)
print("BẢNG ĐỐI CHIẾU PHƯƠNG PHÁP GIẢM CHIỀU DỮ LIỆU")
print("="*80)

# Cấu trúc lại danh sách thành DataFrame của Pandas để hiển thị đẹp mắt dưới dạng bảng
summary_methods = pd.DataFrame({
    "Tiêu chí so sánh": [
        "Loại mô hình",
        "Cấu trúc hình học giữ lại",
        "Phương sai giải thích (Explained Variance)",
        "Khả năng diễn giải đặc trưng (Interpretability)",
        "Khả năng tái tạo gốc (Reconstruction)"
    ],
    "Phương pháp Tuyến tính (PCA / PCA thô)": [
        "Tuyến tính (Linear)",
        "Toàn cục - Toàn bộ phương sai lớn nhất của dữ liệu",
        f"{cum_evr[5]:.1f}% (Tích lũy tại 6 PC đầu)",  # Sử dụng an toàn biến cum_evr
        "Dễ dàng - Dựa trên ma trận trọng số Loadings rõ ràng",
        "Rất tốt - Có thể nghịch đảo ma trận để phục hồi dữ liệu gốc"
    ],
    "Phương pháp Phi tuyến (t-SNE / UMAP / Isomap)": [
        "Phi tuyến tính (Non-linear)",
        "Cục bộ - Bảo toàn khoảng cách lân cận gần nhất (Topology)",
        "Không đo trực tiếp - Không tồn tại khái niệm Trị riêng",
        "Rất khó - Các trục không có ý nghĩa vật lý trực tiếp",
        "Không thể - Chỉ phục vụ mục đích nén để trực quan hóa đồ thị"
    ]
})

# Hiển thị bảng dạng giao diện đẹp trong Jupyter Notebook/Colab
display(summary_methods)

#III) Phân cụm dữ liệu

### 1. Các độ đo định lượng
- Đầu vào:
    - **Silhouette Score (SH)**
    - **Davies-Bouldin Index (DB)**
    - **Calinski-Harabasz Index (CH)**
- Đầu ra:
    - **Adjusted Rand Index (ARI)**
    - **Fowlkes-Mallows Index (FMI)**

In [ ]:
"""
=============================================================================
PHÂN CỤM DỮ LIỆU — CLUSTERING ANALYSIS
(i)  K-Means
(ii) GMM  — Gaussian Mixture Model (EM Algorithm)
(iii) DBSCAN — Density-Based Spatial Clustering
=============================================================================
"""

# ==========================================
# 0. IMPORTS
# ==========================================
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from sklearn.neighbors import NearestNeighbors
from scipy.stats import f_oneway, kruskal
import itertools

warnings.filterwarnings('ignore')

# ─── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#F8F9FA', 'axes.facecolor': '#FFFFFF',
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linestyle': '--',
    'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.labelsize': 10, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
})

OUT_DIR  = "/mnt/user-data/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Bảng màu phân biệt tối đa 8 cụm + nhiễu (-1)
CLUSTER_COLORS = ['#2E86AB','#E84855','#3BB273','#F18F01',
                  '#9B5DE5','#00B4D8','#FF6B6B','#4ECDC4']
NOISE_COLOR    = '#AAAAAA'
CMAP_SEQ       = 'viridis'
SEP            = "=" * 65
SEP_THIN       = "-" * 65

def header(t):
    print(f"\n{SEP}\n  {t}\n{SEP}")

def sub(t):
    print(f"\n  ▶ {t}\n  {SEP_THIN}")

def cluster_color(label):
    if label == -1:
        return NOISE_COLOR
    return CLUSTER_COLORS[label % len(CLUSTER_COLORS)]


# ==========================================
# 1. LOAD & CHUẨN BỊ DỮ LIỆU
# ==========================================
header("1. LOAD & CHUẨN BỊ DỮ LIỆU")

CSV_PATH = "support2_final.csv"
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f"  Đã load: {CSV_PATH}")
else:
    print("  [INFO] Không tìm thấy CSV → tạo dataset demo (300 mẫu × 47 chiều)")
    np.random.seed(42)
    n = 300
    continuous = ['age','num_co','edu','scoma','avtisst','sps','aps','hday',
                  'dnrday','meanbp','wblc','hrt','resp','temp','pafi','alb',
                  'bili','crea','sod','ph','glucose','bun','urine','adls','adlsc']
    binary = ['sex','income','diabetes','dementia','ca',
              'race_black','race_hispanic','race_other','race_white',
              'dzgroup_chf','dzgroup_cirrhosis','dzgroup_colon_cancer',
              'dzgroup_coma','dzgroup_copd','dzgroup_lung_cancer',
              'dzgroup_mosf_w_malig','dzclass_cancer','dzclass_coma',
              'dzclass_copd_chf_cirrhosis','dnr_before_sadm','dnr_no_dnr']
    data = {}
    # Tạo 3 cụm rõ ràng để demo
    centers = [[-1.5, -1.5], [1.5, 1.5], [0, -2.5]]
    cluster_id = np.repeat([0,1,2], [100,100,100])
    for i, c in enumerate(continuous):
        cx = centers[cluster_id % 3][0] if i == 0 else 0
        data[c] = (np.random.randn(n) + np.array([centers[k][i%2] for k in cluster_id]))
    data.update({c: np.random.randint(0, 2, n) for c in binary})
    df = pd.DataFrame(data)
    df['charges'] = (5000 + 300*df['age'] + 200*df['aps']
                     - 150*df['meanbp'] + 400*df['sps']
                     + np.array([cluster_id[i]*1500 for i in range(n)])
                     + np.random.randn(n)*1500)

if df.isnull().sum().sum() > 0:
    df = df.fillna(df.median(numeric_only=True))

X_raw      = df.drop(columns=["charges"])
y_target   = df["charges"].values
feat_names = list(X_raw.columns)
N, D       = X_raw.shape

print(f"  Shape dữ liệu đầu vào (bỏ 'charges'): {N} mẫu × {D} chiều")
print(f"  Target 'charges': mean={y_target.mean():.1f}, std={y_target.std():.1f}, "
      f"min={y_target.min():.1f}, max={y_target.max():.1f}")

# Chuẩn hóa
scaler   = StandardScaler()
#X_scaled = scaler.fit_transform(X_raw.values)
X_scaled = X_raw
# PCA 2D & 3D để trực quan hóa
pca2      = PCA(n_components=2, random_state=42)
X_pca2    = pca2.fit_transform(X_scaled)
pca3      = PCA(n_components=3, random_state=42)
X_pca3    = pca3.fit_transform(X_scaled)
var2      = pca2.explained_variance_ratio_ * 100
var3      = pca3.explained_variance_ratio_ * 100

print(f"\n  PCA 2D giải thích: {var2.sum():.1f}%  "
      f"(PC1={var2[0]:.1f}%, PC2={var2[1]:.1f}%)")
print(f"  PCA 3D giải thích: {var3.sum():.1f}%")


# ==========================================
# 2. CHỌN SỐ CỤM TỐI ƯU (Elbow + Silhouette + BIC)
# ==========================================
header("2. CHỌN SỐ CỤM TỐI ƯU")

k_range    = range(2, min(11, N//10 + 2))
inertias   = []
sil_km     = []
ch_km      = []
db_km      = []
bic_gmm    = []
aic_gmm    = []
sil_gmm    = []

print(f"\n  {'k':>4}  {'Inertia':>12}  {'Silhouette':>12}  {'CH Score':>10}  "
      f"{'DB Score':>10}  {'GMM BIC':>12}  {'GMM AIC':>12}")
print(f"  {'-'*76}")

for k in k_range:
    # K-Means
    km   = KMeans(n_clusters=k, random_state=42, n_init=10)
    lkm  = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_km.append(silhouette_score(X_scaled, lkm))
    ch_km.append(calinski_harabasz_score(X_scaled, lkm))
    db_km.append(davies_bouldin_score(X_scaled, lkm))
    # GMM
    gmm  = GaussianMixture(n_components=k, random_state=42, n_init=3)
    gmm.fit(X_scaled)
    lgmm = gmm.predict(X_scaled)
    bic_gmm.append(gmm.bic(X_scaled))
    aic_gmm.append(gmm.aic(X_scaled))
    sil_gmm.append(silhouette_score(X_scaled, lgmm))
    print(f"  {k:>4}  {inertias[-1]:>12.1f}  {sil_km[-1]:>12.4f}  "
          f"{ch_km[-1]:>10.1f}  {db_km[-1]:>10.4f}  "
          f"{bic_gmm[-1]:>12.1f}  {aic_gmm[-1]:>12.1f}")

# Chọn k tối ưu
best_k_sil = list(k_range)[np.argmax(sil_km)]
best_k_bic = list(k_range)[np.argmin(bic_gmm)]
best_k = best_k_sil
print(f"\n  → K tối ưu (Silhouette KMeans): k = {best_k_sil}")
print(f"  → K tối ưu (BIC GMM)           : k = {best_k_bic}")
print(f"  → Chọn k = {best_k} cho tất cả phương pháp")

# ─── Biểu đồ 1: Chọn k ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Chọn số cụm tối ưu (k)", fontsize=13, fontweight='bold')
ks = list(k_range)

ax = axes[0, 0]
ax.plot(ks, inertias, 'o-', color='#2E86AB', linewidth=2, markersize=6)
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5,
           label=f'k={best_k}')
ax.set_xlabel("Số cụm k"); ax.set_ylabel("Inertia (WCSS)")
ax.set_title("Elbow Method (K-Means Inertia)")
ax.legend(fontsize=9)

ax = axes[0, 1]
ax.plot(ks, sil_km, 'o-', color='#3BB273', linewidth=2, markersize=6, label='K-Means')
ax.plot(ks, sil_gmm, 's--', color='#F18F01', linewidth=2, markersize=6, label='GMM')
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score (cao hơn = tốt hơn)")
ax.legend(fontsize=9)

ax = axes[0, 2]
ax.plot(ks, ch_km, 'o-', color='#9B5DE5', linewidth=2, markersize=6)
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("CH Score")
ax.set_title("Calinski-Harabasz Score (cao hơn = tốt hơn)")

ax = axes[1, 0]
ax.plot(ks, db_km, 'o-', color='#E84855', linewidth=2, markersize=6)
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("DB Score")
ax.set_title("Davies-Bouldin Score (thấp hơn = tốt hơn)")

ax = axes[1, 1]
ax.plot(ks, bic_gmm, 'o-', color='#00B4D8', linewidth=2, markersize=6, label='BIC')
ax.plot(ks, aic_gmm, 's--', color='#FF6B6B', linewidth=2, markersize=6, label='AIC')
ax.axvline(best_k_bic, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("Score")
ax.set_title("GMM: BIC & AIC (thấp hơn = tốt hơn)")
ax.legend(fontsize=9)

# k-distance plot cho DBSCAN
ax = axes[1, 2]
nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled)
dists, _ = nbrs.kneighbors(X_scaled)
k_dists  = np.sort(dists[:, -1])[::-1]
ax.plot(range(len(k_dists)), k_dists, color='#3BB273', linewidth=1.5)
ax.set_xlabel("Điểm dữ liệu (sắp xếp theo khoảng cách)")
ax.set_ylabel("Khoảng cách đến điểm láng giềng thứ 5")
ax.set_title("k-Distance Plot (chọn eps cho DBSCAN)")
# Tìm elbow
diffs  = np.diff(k_dists)
knee   = np.argmin(diffs) + 1
eps_suggest = k_dists[knee]
ax.axhline(eps_suggest, color='#E84855', linestyle='--',
           label=f'eps ≈ {eps_suggest:.3f}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/C01_choose_k.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"\n  [Đã lưu] C01_choose_k.png")
print(f"  eps gợi ý cho DBSCAN (từ k-distance): {eps_suggest:.4f}")


# ==========================================
# 3. K-MEANS CLUSTERING
# ==========================================
header(f"3. K-MEANS CLUSTERING  (k = {best_k})")

km_final   = KMeans(n_clusters=best_k, random_state=42, n_init=20, max_iter=500)
labels_km  = km_final.fit_predict(X_scaled)
centers_km = km_final.cluster_centers_

sub("3a. Thông tin cụm")
print(f"  {'Cụm':>5}  {'Số mẫu':>8}  {'% mẫu':>8}  "
      f"{'charges mean':>14}  {'charges std':>13}  {'charges median':>15}")
print(f"  {'-'*68}")
for c in sorted(set(labels_km)):
    mask = labels_km == c
    print(f"  {c:>5}  {mask.sum():>8}  {mask.mean()*100:>7.1f}%  "
          f"{y_target[mask].mean():>14.1f}  {y_target[mask].std():>13.1f}  "
          f"{np.median(y_target[mask]):>15.1f}")

sub("3b. Độ đo đánh giá chất lượng phân cụm")
sil_km_f   = silhouette_score(X_scaled, labels_km)
ch_km_f    = calinski_harabasz_score(X_scaled, labels_km)
db_km_f    = davies_bouldin_score(X_scaled, labels_km)
inertia_km = km_final.inertia_

print(f"  Silhouette Score       : {sil_km_f:.4f}  ([-1,1], cao hơn = tốt hơn)")
print(f"  Calinski-Harabasz Score: {ch_km_f:.2f}  (cao hơn = tốt hơn)")
print(f"  Davies-Bouldin Score   : {db_km_f:.4f}  (thấp hơn = tốt hơn)")
print(f"  Inertia (WCSS)         : {inertia_km:.2f}")

sub("3c. Silhouette mỗi cụm")
sil_samples_km = silhouette_samples(X_scaled, labels_km)
for c in sorted(set(labels_km)):
    mask = labels_km == c
    print(f"  Cụm {c}: mean sil = {sil_samples_km[mask].mean():.4f}, "
          f"min = {sil_samples_km[mask].min():.4f}, "
          f"max = {sil_samples_km[mask].max():.4f}")

sub("3d. Kiểm định ANOVA / Kruskal-Wallis (charges giữa các cụm)")
groups_km = [y_target[labels_km == c] for c in sorted(set(labels_km))]
f_stat, p_anova = f_oneway(*groups_km)
h_stat, p_krus  = kruskal(*groups_km)
print(f"  ANOVA     : F = {f_stat:.4f},  p = {p_anova:.6f}  "
      f"{'→ Có sự khác biệt đáng kể' if p_anova < 0.05 else '→ Không đáng kể'}")
print(f"  Kruskal-W : H = {h_stat:.4f},  p = {p_krus:.6f}  "
      f"{'→ Có sự khác biệt đáng kể' if p_krus < 0.05 else '→ Không đáng kể'}")

sub("3e. Đặc trưng trung bình của từng cụm (5 chiều đầu)")
km_profile = pd.DataFrame(
    #scaler.inverse_transform(centers_km),
    centers_km,
    columns=feat_names,
    index=[f'Cụm_{c}' for c in range(best_k)]
)
print(km_profile.iloc[:, :10].round(3).to_string())

# ─── Biểu đồ 2: K-Means ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f"K-Means Clustering (k={best_k})", fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)
colors_km = [cluster_color(l) for l in labels_km]

# PCA 2D scatter
ax = fig.add_subplot(gs[0, :2])
for c in sorted(set(labels_km)):
    mask = labels_km == c
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
               color=cluster_color(c), alpha=0.7, s=20, label=f'Cụm {c}')
# Vẽ centroids
centers_pca = pca2.transform(centers_km)
ax.scatter(centers_pca[:, 0], centers_pca[:, 1],
           marker='*', s=300, color='black', zorder=5, label='Centroid')
for c, (cx, cy) in enumerate(centers_pca):
    ax.annotate(f'C{c}', (cx, cy), textcoords="offset points",
                xytext=(6, 6), fontsize=9, fontweight='bold')
ax.set_xlabel(f"PC1 ({var2[0]:.1f}%)"); ax.set_ylabel(f"PC2 ({var2[1]:.1f}%)")
ax.set_title("K-Means — Scatter PCA 2D + Centroids")
ax.legend(fontsize=8, markerscale=1.5)

# Silhouette plot
ax = fig.add_subplot(gs[0, 2:])
y_lower = 10
for c in sorted(set(labels_km)):
    c_sil = np.sort(sil_samples_km[labels_km == c])
    y_upper = y_lower + len(c_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     alpha=0.75, color=cluster_color(c), label=f'Cụm {c}')
    ax.text(-0.05, (y_lower + y_upper)/2, str(c), fontsize=8)
    y_lower = y_upper + 10
ax.axvline(sil_km_f, color='#E84855', linestyle='--', linewidth=1.5,
           label=f'Mean Sil = {sil_km_f:.3f}')
ax.set_xlabel("Silhouette Coefficient"); ax.set_ylabel("Mẫu (theo cụm)")
ax.set_title("Silhouette Plot — K-Means")
ax.legend(fontsize=8)

# Box plot charges theo cụm
ax = fig.add_subplot(gs[1, :2])
bp_data = [y_target[labels_km == c] for c in sorted(set(labels_km))]
bp = ax.boxplot(bp_data, patch_artist=True, notch=True,
                medianprops=dict(color='black', linewidth=2))
for patch, c in zip(bp['boxes'], sorted(set(labels_km))):
    patch.set_facecolor(cluster_color(c)); patch.set_alpha(0.75)
ax.set_xticklabels([f'Cụm {c}' for c in sorted(set(labels_km))])
ax.set_ylabel("charges")
ax.set_title("Phân phối charges theo K-Means cụm")

# Heatmap đặc trưng trung bình cụm
ax = fig.add_subplot(gs[1, 2:])
km_heat = pd.DataFrame(centers_km,
                        columns=feat_names,
                        index=[f'C{c}' for c in range(best_k)])
show_feat = km_heat.std().nlargest(15).index
sns.heatmap(km_heat[show_feat].T, ax=ax, cmap='RdBu_r', center=0,
            annot=True if best_k <= 5 else False,
            fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, cbar_kws={'shrink': 0.8})
ax.set_title("Heatmap đặc trưng trung bình (Z-score, 15 chiều biến thiên nhất)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)

# Parallel coordinates (top 8 features)
ax = fig.add_subplot(gs[2, :])
top8 = km_heat.std().nlargest(8).index
km_norm = (km_heat[top8] - km_heat[top8].min()) / \
          (km_heat[top8].max() - km_heat[top8].min() + 1e-9)
for c in range(best_k):
    ax.plot(range(len(top8)), km_norm.iloc[c], 'o-',
            color=cluster_color(c), linewidth=2.5, markersize=7,
            label=f'Cụm {c} (n={( labels_km==c).sum()})')
ax.set_xticks(range(len(top8)))
ax.set_xticklabels(top8, rotation=20, ha='right', fontsize=9)
ax.set_ylabel("Giá trị chuẩn hóa (0–1)")
ax.set_title("Parallel Coordinates — Profile các cụm K-Means (Top 8 chiều)")
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(-0.1, 1.1)

plt.savefig(f"{OUT_DIR}/C02_kmeans.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"\n  [Đã lưu] C02_kmeans.png")


# ==========================================
# 4. GMM — GAUSSIAN MIXTURE MODEL (EM)
# ==========================================
header(f"4. GMM — GAUSSIAN MIXTURE MODEL  (k = {best_k})")

gmm_final  = GaussianMixture(n_components=best_k, covariance_type='full',
                               random_state=42, n_init=5, max_iter=200)
gmm_final.fit(X_scaled)
labels_gmm = gmm_final.predict(X_scaled)
proba_gmm  = gmm_final.predict_proba(X_scaled)
means_gmm  = gmm_final.means_
covs_gmm   = gmm_final.covariances_
weights_gmm = gmm_final.weights_

sub("4a. Thông tin cụm")
print(f"  {'Cụm':>5}  {'Số mẫu':>8}  {'% mẫu':>8}  {'Weight GMM':>12}  "
      f"{'charges mean':>14}  {'charges std':>13}")
print(f"  {'-'*66}")
for c in sorted(set(labels_gmm)):
    mask = labels_gmm == c
    print(f"  {c:>5}  {mask.sum():>8}  {mask.mean()*100:>7.1f}%  "
          f"{weights_gmm[c]:>12.4f}  {y_target[mask].mean():>14.1f}  "
          f"{y_target[mask].std():>13.1f}")

sub("4b. Tham số GMM (EM Algorithm)")
print(f"  Log-likelihood cuối  : {gmm_final.score(X_scaled)*N:.2f}")
print(f"  BIC                  : {gmm_final.bic(X_scaled):.2f}")
print(f"  AIC                  : {gmm_final.aic(X_scaled):.2f}")
print(f"  Số vòng lặp EM       : {gmm_final.n_iter_}")
print(f"  Covariance type      : full")
print(f"\n  Trọng số (mixing weights) các Gaussian:")
for c in range(best_k):
    print(f"    Component {c}: π = {weights_gmm[c]:.4f}")

sub("4c. Độ đo đánh giá")
sil_gmm_f = silhouette_score(X_scaled, labels_gmm)
ch_gmm_f  = calinski_harabasz_score(X_scaled, labels_gmm)
db_gmm_f  = davies_bouldin_score(X_scaled, labels_gmm)
sil_samples_gmm = silhouette_samples(X_scaled, labels_gmm)

print(f"  Silhouette Score       : {sil_gmm_f:.4f}")
print(f"  Calinski-Harabasz Score: {ch_gmm_f:.2f}")
print(f"  Davies-Bouldin Score   : {db_gmm_f:.4f}")

sub("4d. Xác suất thuộc cụm (5 mẫu đầu)")
prob_df = pd.DataFrame(proba_gmm, columns=[f'P(Cụm_{c})' for c in range(best_k)])
prob_df['Cụm gán'] = labels_gmm
prob_df['Max prob'] = proba_gmm.max(axis=1)
print(prob_df.head(10).round(4).to_string())
print(f"\n  Trung bình Max probability: {prob_df['Max prob'].mean():.4f}")
print(f"  (Giá trị gần 1.0 → mẫu nằm rõ ràng trong 1 cụm)")

sub("4e. Kiểm định ANOVA / Kruskal-Wallis")
groups_gmm = [y_target[labels_gmm == c] for c in sorted(set(labels_gmm))]
f_g, p_g   = f_oneway(*groups_gmm)
h_g, p_kg  = kruskal(*groups_gmm)
print(f"  ANOVA     : F = {f_g:.4f},  p = {p_g:.6f}  "
      f"{'→ Có sự khác biệt đáng kể' if p_g < 0.05 else '→ Không đáng kể'}")
print(f"  Kruskal-W : H = {h_g:.4f},  p = {p_kg:.6f}  "
      f"{'→ Có sự khác biệt đáng kể' if p_kg < 0.05 else '→ Không đáng kể'}")

sub("4f. So sánh nhãn K-Means vs GMM")
ari_km_gmm = adjusted_rand_score(labels_km, labels_gmm)
nmi_km_gmm = normalized_mutual_info_score(labels_km, labels_gmm)
print(f"  Adjusted Rand Index (ARI)         : {ari_km_gmm:.4f}  (1=giống hệt, 0=ngẫu nhiên)")
print(f"  Normalized Mutual Information (NMI): {nmi_km_gmm:.4f}")

# ─── Biểu đồ 3: GMM ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f"GMM Clustering (k={best_k}, covariance=full)", fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)

# PCA 2D
ax = fig.add_subplot(gs[0, :2])
for c in sorted(set(labels_gmm)):
    mask = labels_gmm == c
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
               color=cluster_color(c), alpha=0.7, s=20, label=f'Cụm {c}')
means_pca = pca2.transform(means_gmm)
ax.scatter(means_pca[:, 0], means_pca[:, 1],
           marker='D', s=200, color='black', zorder=5, label='Mean')
for c, (cx, cy) in enumerate(means_pca):
    ax.annotate(f'G{c}', (cx, cy), textcoords="offset points",
                xytext=(6, 6), fontsize=9, fontweight='bold')
ax.set_xlabel(f"PC1 ({var2[0]:.1f}%)"); ax.set_ylabel(f"PC2 ({var2[1]:.1f}%)")
ax.set_title("GMM — Scatter PCA 2D + Means")
ax.legend(fontsize=8, markerscale=1.5)

# Uncertainty (1 - max proba)
ax = fig.add_subplot(gs[0, 2:])
uncertainty = 1 - proba_gmm.max(axis=1)
sc = ax.scatter(X_pca2[:, 0], X_pca2[:, 1], c=uncertainty,
                cmap='hot_r', alpha=0.8, s=20)
plt.colorbar(sc, ax=ax, label='Uncertainty (1 - max P)')
ax.set_xlabel(f"PC1"); ax.set_ylabel(f"PC2")
ax.set_title("GMM — Uncertainty (đỏ = không chắc chắn)")

# Silhouette
ax = fig.add_subplot(gs[1, :2])
y_lower = 10
for c in sorted(set(labels_gmm)):
    c_sil = np.sort(sil_samples_gmm[labels_gmm == c])
    y_upper = y_lower + len(c_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     alpha=0.75, color=cluster_color(c))
    ax.text(-0.05, (y_lower + y_upper)/2, str(c), fontsize=8)
    y_lower = y_upper + 10
ax.axvline(sil_gmm_f, color='#E84855', linestyle='--', linewidth=1.5,
           label=f'Mean Sil = {sil_gmm_f:.3f}')
ax.set_xlabel("Silhouette Coefficient")
ax.set_title("Silhouette Plot — GMM")
ax.legend(fontsize=8)

# Box plot charges
ax = fig.add_subplot(gs[1, 2:])
bp_data2 = [y_target[labels_gmm == c] for c in sorted(set(labels_gmm))]
bp2 = ax.boxplot(bp_data2, patch_artist=True, notch=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, c in zip(bp2['boxes'], sorted(set(labels_gmm))):
    patch.set_facecolor(cluster_color(c)); patch.set_alpha(0.75)
ax.set_xticklabels([f'Cụm {c}' for c in sorted(set(labels_gmm))])
ax.set_ylabel("charges")
ax.set_title("Phân phối charges theo GMM cụm")

# Probability heatmap
ax = fig.add_subplot(gs[2, :2])
prob_sorted = prob_df.sort_values('Cụm gán').reset_index(drop=True)
prob_mat    = prob_sorted[[f'P(Cụm_{c})' for c in range(best_k)]].values
im = ax.imshow(prob_mat.T, aspect='auto', cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_yticks(range(best_k))
ax.set_yticklabels([f'P(Cụm_{c})' for c in range(best_k)])
ax.set_xlabel("Mẫu dữ liệu (sắp xếp theo cụm)")
ax.set_title("GMM — Ma trận xác suất thuộc cụm (soft assignment)")

# Heatmap đặc trưng
ax = fig.add_subplot(gs[2, 2:])
gmm_heat = pd.DataFrame(means_gmm, columns=feat_names,
                          index=[f'G{c}' for c in range(best_k)])
show_feat2 = gmm_heat.std().nlargest(15).index
sns.heatmap(gmm_heat[show_feat2].T, ax=ax, cmap='RdBu_r', center=0,
            annot=True if best_k <= 5 else False,
            fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, cbar_kws={'shrink': 0.8})
ax.set_title("Heatmap đặc trưng trung bình — GMM (Z-score)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)

plt.savefig(f"{OUT_DIR}/C03_gmm.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"\n  [Đã lưu] C03_gmm.png")


# ==========================================
# 5. DBSCAN
# ==========================================
header("5. DBSCAN — DENSITY-BASED CLUSTERING")

# Thử nhiều eps
sub("5a. Tìm tham số eps & min_samples tối ưu")
eps_candidates     = np.linspace(eps_suggest*0.5, eps_suggest*2.5, 8)
min_samples_list   = [3, 5, 10]
best_eps, best_ms  = eps_suggest, 5
best_sil_db        = -999
best_n_clusters_db = 0

print(f"  {'eps':>8}  {'min_s':>7}  {'n_cụm':>7}  {'n_noise':>8}  {'Silhouette':>12}")
print(f"  {'-'*50}")
for eps_t in eps_candidates:
    for ms in min_samples_list:
        db_t = DBSCAN(eps=eps_t, min_samples=ms).fit(X_scaled)
        lbl_t = db_t.labels_
        nc    = len(set(lbl_t)) - (1 if -1 in lbl_t else 0)
        nn    = (lbl_t == -1).sum()
        if nc >= 2 and (N - nn) > 10:
            sil_t = silhouette_score(X_scaled[lbl_t != -1], lbl_t[lbl_t != -1])
            mark  = " ← best" if (sil_t > best_sil_db) else ""
            print(f"  {eps_t:>8.4f}  {ms:>7}  {nc:>7}  {nn:>8}  {sil_t:>12.4f}{mark}")
            if sil_t > best_sil_db:
                best_sil_db = sil_t
                best_eps, best_ms = eps_t, ms
                best_n_clusters_db = nc
        else:
            print(f"  {eps_t:>8.4f}  {ms:>7}  {nc:>7}  {nn:>8}  {'N/A':>12}")

print(f"\n  → Tham số tối ưu: eps = {best_eps:.4f}, min_samples = {best_ms}")

sub("5b. Chạy DBSCAN với tham số tối ưu")
db_final   = DBSCAN(eps=best_eps, min_samples=best_ms)
labels_db  = db_final.fit_predict(X_scaled)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db    = (labels_db == -1).sum()

print(f"  Số cụm tìm được  : {n_clusters_db}")
print(f"  Số điểm nhiễu    : {n_noise_db} ({n_noise_db/N*100:.1f}%)")

print(f"\n  {'Cụm':>5}  {'Số mẫu':>8}  {'% mẫu':>8}  "
      f"{'charges mean':>14}  {'charges std':>13}  {'charges median':>15}")
print(f"  {'-'*70}")
for c in sorted(set(labels_db)):
    mask = labels_db == c
    cname = f"Nhiễu(-1)" if c == -1 else str(c)
    print(f"  {cname:>9}  {mask.sum():>8}  {mask.mean()*100:>7.1f}%  "
          f"{y_target[mask].mean():>14.1f}  {y_target[mask].std():>13.1f}  "
          f"{np.median(y_target[mask]):>15.1f}")

sub("5c. Độ đo đánh giá (bỏ qua điểm nhiễu)")
non_noise = labels_db != -1
if non_noise.sum() > 10 and n_clusters_db >= 2:
    sil_db_f = silhouette_score(X_scaled[non_noise], labels_db[non_noise])
    ch_db_f  = calinski_harabasz_score(X_scaled[non_noise], labels_db[non_noise])
    db_db_f  = davies_bouldin_score(X_scaled[non_noise], labels_db[non_noise])
    sil_samples_db = silhouette_samples(X_scaled[non_noise], labels_db[non_noise])
    print(f"  Silhouette Score       : {sil_db_f:.4f}")
    print(f"  Calinski-Harabasz Score: {ch_db_f:.2f}")
    print(f"  Davies-Bouldin Score   : {db_db_f:.4f}")
    print(f"  (Tính trên {non_noise.sum()} điểm, loại bỏ {n_noise_db} điểm nhiễu)")
else:
    sil_db_f = ch_db_f = db_db_f = np.nan
    sil_samples_db = np.array([])
    print(f"  Không đủ cụm hoặc điểm để tính độ đo.")

sub("5d. Kiểm định ANOVA / Kruskal-Wallis")
if n_clusters_db >= 2:
    groups_db = [y_target[labels_db == c] for c in sorted(set(labels_db)) if c != -1]
    if len(groups_db) >= 2 and all(len(g) >= 2 for g in groups_db):
        f_d, p_d   = f_oneway(*groups_db)
        h_d, p_kd  = kruskal(*groups_db)
        print(f"  ANOVA     : F = {f_d:.4f},  p = {p_d:.6f}  "
              f"{'→ Có sự khác biệt đáng kể' if p_d < 0.05 else '→ Không đáng kể'}")
        print(f"  Kruskal-W : H = {h_d:.4f},  p = {p_kd:.6f}  "
              f"{'→ Có sự khác biệt đáng kể' if p_kd < 0.05 else '→ Không đáng kể'}")

sub("5e. So sánh nhãn DBSCAN với K-Means và GMM (bỏ điểm nhiễu)")
mask_nn = labels_db != -1
if mask_nn.sum() > 10 and n_clusters_db >= 2:
    ari_db_km  = adjusted_rand_score(labels_km[mask_nn], labels_db[mask_nn])
    ari_db_gmm = adjusted_rand_score(labels_gmm[mask_nn], labels_db[mask_nn])
    nmi_db_km  = normalized_mutual_info_score(labels_km[mask_nn], labels_db[mask_nn])
    print(f"  ARI (DBSCAN vs K-Means) : {ari_db_km:.4f}")
    print(f"  ARI (DBSCAN vs GMM)     : {ari_db_gmm:.4f}")
    print(f"  NMI (DBSCAN vs K-Means) : {nmi_db_km:.4f}")

# ─── Biểu đồ 4: DBSCAN ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f"DBSCAN Clustering (eps={best_eps:.3f}, min_samples={best_ms})",
             fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)

# PCA 2D scatter
ax = fig.add_subplot(gs[0, :2])
unique_lbls = sorted(set(labels_db))
for c in unique_lbls:
    mask  = labels_db == c
    cname = "Nhiễu" if c == -1 else f"Cụm {c}"
    col   = NOISE_COLOR if c == -1 else cluster_color(c)
    size  = 10 if c == -1 else 25
    alpha = 0.3 if c == -1 else 0.75
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
               color=col, alpha=alpha, s=size, label=f'{cname} (n={mask.sum()})')
ax.set_xlabel(f"PC1 ({var2[0]:.1f}%)"); ax.set_ylabel(f"PC2 ({var2[1]:.1f}%)")
ax.set_title("DBSCAN — Scatter PCA 2D\n(xám = nhiễu)")
ax.legend(fontsize=7, markerscale=1.5)

# Highlight noise
ax = fig.add_subplot(gs[0, 2:])
ax.scatter(X_pca2[labels_db != -1, 0], X_pca2[labels_db != -1, 1],
           c=[cluster_color(l) for l in labels_db[labels_db != -1]],
           alpha=0.7, s=20, label='Core/Border')
ax.scatter(X_pca2[labels_db == -1, 0], X_pca2[labels_db == -1, 1],
           color=NOISE_COLOR, alpha=0.4, s=12, marker='x', label=f'Nhiễu (n={n_noise_db})')
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title(f"DBSCAN — Highlight điểm nhiễu\n({n_noise_db}/{N} = {n_noise_db/N*100:.1f}%)")
ax.legend(fontsize=8)

# k-distance plot với eps đã chọn
ax = fig.add_subplot(gs[1, :2])
ax.plot(range(len(k_dists)), k_dists, color='#3BB273', linewidth=1.2)
ax.axhline(best_eps, color='#E84855', linestyle='--', linewidth=2,
           label=f'eps = {best_eps:.4f}')
ax.set_xlabel("Điểm (sắp giảm dần)")
ax.set_ylabel("k-distance (k=5)")
ax.set_title("k-Distance Plot — Xác nhận eps")
ax.legend(fontsize=9)

# Box plot charges
ax = fig.add_subplot(gs[1, 2:])
db_groups_plot = [(c, y_target[labels_db == c])
                  for c in sorted(set(labels_db))]
bp_data3 = [g for _, g in db_groups_plot]
bp3 = ax.boxplot(bp_data3, patch_artist=True, notch=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, (c, _) in zip(bp3['boxes'], db_groups_plot):
    patch.set_facecolor(NOISE_COLOR if c == -1 else cluster_color(c))
    patch.set_alpha(0.75)
ax.set_xticklabels(['Nhiễu' if c == -1 else f'Cụm {c}' for c, _ in db_groups_plot],
                   fontsize=8)
ax.set_ylabel("charges")
ax.set_title("Phân phối charges theo DBSCAN cụm")

# Silhouette (nếu có)
ax = fig.add_subplot(gs[2, :2])
if len(sil_samples_db) > 0 and n_clusters_db >= 2:
    labels_nn = labels_db[non_noise]
    y_lower = 10
    for c in sorted(set(labels_nn)):
        c_sil = np.sort(sil_samples_db[labels_nn == c])
        y_upper = y_lower + len(c_sil)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                         alpha=0.75, color=cluster_color(c))
        ax.text(-0.05, (y_lower + y_upper)/2, str(c), fontsize=8)
        y_lower = y_upper + 10
    ax.axvline(sil_db_f, color='#E84855', linestyle='--', linewidth=1.5,
               label=f'Mean Sil = {sil_db_f:.3f}')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'Không đủ dữ liệu\nđể vẽ Silhouette',
            ha='center', va='center', transform=ax.transAxes, fontsize=11)
ax.set_xlabel("Silhouette Coefficient")
ax.set_title("Silhouette Plot — DBSCAN (bỏ nhiễu)")

# Core/border/noise density
ax = fig.add_subplot(gs[2, 2:])
sc = ax.scatter(X_pca2[:, 0], X_pca2[:, 1],
               c=[0 if l == -1 else 1 for l in labels_db],
               cmap=ListedColormap([NOISE_COLOR, '#3BB273']),
               alpha=0.6, s=15)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("DBSCAN — Core/Border (xanh) vs Noise (xám)")
patches = [mpatches.Patch(color='#3BB273', label='Core/Border'),
           mpatches.Patch(color=NOISE_COLOR, label='Noise')]
ax.legend(handles=patches, fontsize=9)

plt.savefig(f"{OUT_DIR}/C04_dbscan.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"\n  [Đã lưu] C04_dbscan.png")


# ==========================================
# 6. SO SÁNH 3 PHƯƠNG PHÁP
# ==========================================
header("6. SO SÁNH 3 PHƯƠNG PHÁP PHÂN CỤM")

sub("6a. Bảng độ đo định lượng")
print(f"\n  {'Độ đo':<30}  {'K-Means':>12}  {'GMM':>12}  {'DBSCAN':>12}")
print(f"  {'-'*70}")
metrics = [
    ("Silhouette Score",        sil_km_f, sil_gmm_f, sil_db_f),
    ("Calinski-Harabasz Score", ch_km_f,  ch_gmm_f,  ch_db_f),
    ("Davies-Bouldin Score",    db_km_f,  db_gmm_f,  db_db_f),
    ("Số cụm",                 best_k,   best_k,    n_clusters_db),
]
# for name, v1, v2, v3 in metrics:
#     def fmt(v): return f"{v:.4f}" if isinstance(v, float) and not np.isnan(v) else ("N/A" if np.isnan(v) if isinstance(v, float) else str(int(v)))
#     print(f"  {name:<30}  {fmt(v1):>12}  {fmt(v2):>12}  {fmt(v3):>12}")
for name, v1, v2, v3 in metrics:
    def fmt(v):
        if isinstance(v, float):
            return "N/A" if np.isnan(v) else f"{v:.4f}"
        return str(int(v))
    print(f"  {name:<30}  {fmt(v1):>12}  {fmt(v2):>12}  {fmt(v3):>12}")
sub("6b. Tương đồng nhãn giữa các phương pháp (ARI)")
ari_km_gmm2 = adjusted_rand_score(labels_km, labels_gmm)
print(f"  ARI (K-Means vs GMM)    : {ari_km_gmm2:.4f}")
if non_noise.sum() > 10 and n_clusters_db >= 2:
    print(f"  ARI (K-Means vs DBSCAN) : {ari_db_km:.4f}")
    print(f"  ARI (GMM vs DBSCAN)     : {ari_db_gmm:.4f}")

sub("6c. Mối quan hệ mẫu đầu vào trong cụm")
print("""
  K-Means:
  - Giả định cụm có hình dạng cầu, kích thước tương đương
  - Phân chia không gian theo khoảng cách Euclidean đến centroid
  - Nhạy cảm với outlier (centroid bị kéo)

  GMM (EM):
  - Mô hình xác suất, mỗi mẫu có xác suất thuộc mỗi cụm
  - Cho phép cụm có hình elipsoid, kích thước và hướng khác nhau
  - Soft assignment: 1 mẫu có thể "thuộc" nhiều cụm với xác suất khác nhau
  - Thích hợp khi ranh giới cụm mờ (overlapping clusters)

  DBSCAN:
  - Phân cụm theo mật độ, không cần chỉ định số cụm
  - Tự động phát hiện điểm nhiễu (outlier)
  - Tốt với cụm hình dạng tùy ý, không đều
  - Nhạy cảm với tham số eps và min_samples
  """)

sub("6d. Mối quan hệ đầu ra (charges) trong các cụm")
print(f"  Charges thống kê theo cụm:")
print(f"\n  {'':>10}  {'':>10}  {'KM mean':>10}  {'GMM mean':>10}  {'DB mean':>10}")
for c_km in sorted(set(labels_km)):
    mask_km  = labels_km == c_km
    km_m     = y_target[mask_km].mean()
    # Find best matching GMM cluster
    gmm_m = y_target[labels_gmm == c_km].mean() if c_km in labels_gmm else np.nan
    db_m  = y_target[labels_db == c_km].mean() if c_km in labels_db else np.nan
    print(f"  Cụm {c_km:<5}          {' ':>10}  {km_m:>10.1f}  {gmm_m if not np.isnan(gmm_m) else 'N/A':>10}  {db_m if not np.isnan(db_m) else 'N/A':>10}")

# ─── Biểu đồ 5: So sánh ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("So sánh 3 phương pháp phân cụm", fontsize=14, fontweight='bold')

titles   = ['K-Means', 'GMM', 'DBSCAN']
all_lbls = [labels_km, labels_gmm, labels_db]

for col, (lbl, ttl) in enumerate(zip(all_lbls, titles)):
    # Hàng 1: Scatter PCA 2D
    ax = axes[0, col]
    unique_c = sorted(set(lbl))
    for c in unique_c:
        mask  = lbl == c
        col_c = NOISE_COLOR if c == -1 else cluster_color(c)
        cname = "Noise" if c == -1 else f'C{c}'
        ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                   color=col_c, alpha=0.65, s=18, label=f'{cname}(n={mask.sum()})')
    ax.set_xlabel(f"PC1 ({var2[0]:.1f}%)", fontsize=8)
    ax.set_ylabel(f"PC2 ({var2[1]:.1f}%)", fontsize=8)
    ax.set_title(f"{ttl} — Scatter PCA 2D")
    ax.legend(fontsize=7, markerscale=1.3)

    # Hàng 2: Violin + Strip charges theo cụm
    ax = axes[1, col]
    data_vio  = [y_target[lbl == c] for c in unique_c if c != -1]
    labels_vio = [f'C{c}' for c in unique_c if c != -1]
    colors_vio = [cluster_color(c) for c in unique_c if c != -1]
    parts = ax.violinplot(data_vio, positions=range(len(data_vio)),
                          showmedians=True, showextrema=True)
    for pc, col_c in zip(parts['bodies'], colors_vio):
        pc.set_facecolor(col_c); pc.set_alpha(0.6)
    # Stripplot overlay
    for i, (d, col_c) in enumerate(zip(data_vio, colors_vio)):
        jitter = np.random.randn(len(d)) * 0.06
        ax.scatter(i + jitter, d, color=col_c, alpha=0.3, s=6, edgecolors='none')
    ax.set_xticks(range(len(labels_vio)))
    ax.set_xticklabels(labels_vio, fontsize=8)
    ax.set_ylabel("charges")
    ax.set_title(f"{ttl} — Violin + Strip (charges)")
    if -1 in lbl:
        n_ns = (lbl == -1).sum()
        ax.set_xlabel(f"(+ {n_ns} điểm nhiễu không hiển thị)")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/C05_comparison.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"\n  [Đã lưu] C05_comparison.png")

# ─── Biểu đồ 6: Dashboard tổng hợp ──────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.suptitle("Dashboard Phân cụm Tổng hợp — K-Means | GMM | DBSCAN",
             fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.5, wspace=0.4)

# Hàng 0: PCA 2D cho 3 PP
for col, (lbl, ttl) in enumerate(zip(all_lbls, titles)):
    ax   = fig.add_subplot(gs[0, col])
    for c in sorted(set(lbl)):
        mask = lbl == c
        ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                   color=NOISE_COLOR if c==-1 else cluster_color(c),
                   alpha=0.6 if c!=-1 else 0.25,
                   s=15 if c!=-1 else 8,
                   label=f'{"Noise" if c==-1 else f"C{c}"}')
    ax.set_title(f"{ttl}")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.legend(fontsize=7, markerscale=1.2)

# Hàng 1: Silhouette bars
ax = fig.add_subplot(gs[1, :])
methods_sil  = ['K-Means', 'GMM', 'DBSCAN']
values_sil   = [sil_km_f, sil_gmm_f, sil_db_f if not np.isnan(sil_db_f) else 0]
bar_c_sil    = ['#2E86AB', '#3BB273', '#E84855']
bars_s = ax.bar(methods_sil, values_sil, color=bar_c_sil, alpha=0.85, width=0.4)
for bar, val in zip(bars_s, values_sil):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel("Silhouette Score")
ax.set_title("So sánh Silhouette Score (cao hơn = tốt hơn)")
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylim(bottom=min(0, min(values_sil)-0.05))

# Hàng 2: Radar chart
ax = fig.add_subplot(gs[2, :2])
metrics_compare = ['Silhouette\n(↑)', 'CH/1000\n(↑)', '1/DB\n(↑)']
ch_norm  = [ch_km_f/1000, ch_gmm_f/1000, ch_db_f/1000 if not np.isnan(ch_db_f) else 0]
idb_norm = [1/db_km_f if db_km_f > 0 else 0,
            1/db_gmm_f if db_gmm_f > 0 else 0,
            1/db_db_f if (not np.isnan(db_db_f) and db_db_f > 0) else 0]
sil_norm = [max(0, v) for v in values_sil]
for i, (m, bc, lbl_m) in enumerate(zip(
    [sil_norm, ch_norm, idb_norm],
    ['#2E86AB','#3BB273','#E84855'],
    ['Silhouette', 'CH/1000', '1/DB']
)):
    ax.bar([f'{methods_sil[j]}\n{lbl_m}' for j in range(3)],
           m, bottom=i*0.5, color=bc, alpha=0.7, width=0.4)
ax.set_title("Đa chiều độ đo (stacked bar)")
ax.set_ylabel("Giá trị (scale khác nhau)")

# Hàng 2-3: Charges distribution all methods
ax = fig.add_subplot(gs[2:, 2])
all_colors_flat = []
all_y_flat      = []
method_labels   = []
for lbl, ttl in zip(all_lbls, titles):
    for c in sorted(set(lbl)):
        if c == -1: continue
        mask = lbl == c
        all_y_flat.extend(y_target[mask])
        all_colors_flat.extend([cluster_color(c)] * mask.sum())
        method_labels.extend([f'{ttl}\nC{c}'] * mask.sum())
plot_df = pd.DataFrame({'y': all_y_flat, 'method': method_labels})
unique_m = plot_df['method'].unique()
ym = [plot_df[plot_df['method']==m]['y'].values for m in unique_m]
bpf = ax.boxplot(ym, patch_artist=True,
                  medianprops=dict(color='black', linewidth=1.5))
for patch, m in zip(bpf['boxes'], unique_m):
    pp  = m.split('\n')
    c_n = int(pp[1][1]) if len(pp) > 1 else 0
    patch.set_facecolor(cluster_color(c_n)); patch.set_alpha(0.7)
ax.set_xticks(range(1, len(unique_m)+1))
ax.set_xticklabels(unique_m, fontsize=6, rotation=45, ha='right')
ax.set_title("Charges: tất cả cụm × phương pháp")
ax.set_ylabel("charges")

# Hàng 3: Comparison table
ax = fig.add_subplot(gs[3, :2])
ax.axis('off')
table_data = [
    ['Tiêu chí', 'K-Means', 'GMM', 'DBSCAN'],
    ['Loại', 'Centroid-based', 'Probabilistic', 'Density-based'],
    ['Số cụm', str(best_k), str(best_k), str(n_clusters_db)],
    ['Điểm nhiễu', 'Không', 'Không', f'{n_noise_db} ({n_noise_db/N*100:.0f}%)'],
    [f'Silhouette', f'{sil_km_f:.4f}', f'{sil_gmm_f:.4f}',
     f'{sil_db_f:.4f}' if not np.isnan(sil_db_f) else 'N/A'],
    ['CH Score', f'{ch_km_f:.1f}', f'{ch_gmm_f:.1f}',
     f'{ch_db_f:.1f}' if not np.isnan(ch_db_f) else 'N/A'],
    ['DB Score', f'{db_km_f:.4f}', f'{db_gmm_f:.4f}',
     f'{db_db_f:.4f}' if not np.isnan(db_db_f) else 'N/A'],
    ['ARI vs KMeans', '1.0000', f'{ari_km_gmm2:.4f}',
     f'{ari_db_km:.4f}' if non_noise.sum()>10 else 'N/A'],
]
tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0],
               cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
for (r, c_), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2E86AB'); cell.set_text_props(color='white', weight='bold')
    elif c_ == 0:
        cell.set_facecolor('#E8F4FD')
    cell.set_edgecolor('#CCCCCC')
ax.set_title("Bảng so sánh tổng hợp", fontsize=10, fontweight='bold', pad=10)

plt.savefig(f"{OUT_DIR}/C06_dashboard.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"  [Đã lưu] C06_dashboard.png")


# ==========================================
# 7. TÓM TẮT
# ==========================================
header("7. TÓM TẮT & NHẬN XÉT")

print(f"""
  ╔══════════════════════════════════════════════════════════════╗
  ║              KẾT QUẢ PHÂN CỤM DỮ LIỆU                     ║
  ╠══════════════════════════════════════════════════════════════╣
  ║  Dataset : {N} mẫu × {D} chiều (bỏ cột 'charges')
  ║  K tối ưu: {best_k} (Silhouette KMeans), {best_k_bic} (BIC GMM)
  ╠══════════════════════════════════════════════════════════════╣
  ║  K-MEANS (k={best_k}):                                     ║
  ║    Silhouette : {sil_km_f:.4f}
  ║    CH Score   : {ch_km_f:.1f}
  ║    DB Score   : {db_km_f:.4f}
  ╠══════════════════════════════════════════════════════════════╣
  ║  GMM (k={best_k}, EM, covariance=full):                    ║
  ║    Silhouette : {sil_gmm_f:.4f}
  ║    CH Score   : {ch_gmm_f:.1f}
  ║    DB Score   : {db_gmm_f:.4f}
  ║    BIC        : {gmm_final.bic(X_scaled):.1f}
  ╠══════════════════════════════════════════════════════════════╣
  ║  DBSCAN (eps={best_eps:.3f}, min_s={best_ms}):            ║
  ║    Số cụm     : {n_clusters_db}
  ║    Điểm nhiễu : {n_noise_db} ({n_noise_db/N*100:.1f}%)
  ║    Silhouette : {'N/A' if np.isnan(sil_db_f) else f'{sil_db_f:.4f}'}
  ╠══════════════════════════════════════════════════════════════╣
  ║  Nhận xét quan hệ mẫu đầu vào trong cụm:                  ║
  ║  • K-Means / GMM: cụm hình thành dựa trên khoảng cách     ║
  ║    toàn cục — các mẫu cùng cụm có đặc trưng gần nhau      ║
  ║  • DBSCAN: cụm hình thành theo mật độ — tìm vùng đặc      ║
  ║    phân tách bởi vùng thưa; điểm nhiễu = outlier thực sự  ║
  ║  Nhận xét quan hệ đầu ra (charges) trong cụm:             ║
  ║  • Nếu p_anova < 0.05: charges khác nhau đáng kể giữa cụm║
  ║  • Cụm có charges cao → nhóm bệnh nhân nặng/tốn kém hơn  ║
  ╚══════════════════════════════════════════════════════════════╝

  Files đã xuất:
    C01_choose_k.png      — Elbow, Silhouette, CH, DB, BIC/AIC, k-distance
    C02_kmeans.png        — K-Means: scatter, silhouette, boxplot, heatmap, parallel
    C03_gmm.png           — GMM: scatter, uncertainty, soft assignment, heatmap
    C04_dbscan.png        — DBSCAN: scatter, noise highlight, k-dist, violin
    C05_comparison.png    — Scatter + violin 3 PP side-by-side
    C06_dashboard.png     — Dashboard tổng hợp + bảng so sánh
""")
print(f"  {SEP}\n  HOÀN TẤT\n  {SEP}")

## 1\. Dữ liệu gốc

### 1\. KMeans

**Pipeline Phân Tích Cụm (KMeans)**
---

**Dữ liệu đầu vào:** Data gốc (47D) — Đã qua bước tiền xử lý (Preprocessed Data).

**1. Giai Đoạn Phân Cụm (Clustering)**

*Biểu đồ **Elbow** (kết hợp hàm Inertia hoặc Cost) để xác định số lượng cụm tối ưu cho từng phương pháp.*

**1.1. Thực hiện trên Dữ liệu gốc**
- **KMeans** (Chạy trực tiếp trên bộ dữ liệu).
- **K-Prototypes** (Tách feature thành numerical feature và discrete feature, sau đó áp dụng K-Prototypes ).

**1.2. Thực hiện trên Dữ liệu giảm chiều PCA**
- **KMeans** (chạy trên không gian PCA_mixed của toàn bộ tập dữ liệu).
- **K-Prototypes** (chỉ áp dụng PCA để giảm chiều phần dữ liệu liên tục, sau đó ghép lại với phần biến phân loại gốc để phân cụm).

**1.3. Thực hiện trên Dữ liệu giảm chiều UMAP**
- **KMeans** (chạy trên không gian UMAP đã giữ lại cấu trúc cục bộ của dữ liệu).

- **K-Prototypes** (Không áp dụng vì Umap đã biến đổi dữ liệu thành một không gian số thực liên tục. Lúc này dữ liệu không còn các đặc trưng phân loại).

---

**2. Đánh Giá & Phân Tích**

**2.1. Bộ chỉ số đánh giá (Metrics)**
- Silhouette Score.
- Davies-Bouldin Index.
- Calinski-Harabasz Index.
- Tỷ lệ phân bố kích thước cụm (Cluster Size Distribution).

**2.2. Phân tích nội tại cụm**
Phân tích sâu mối quan hệ trong từng cụm giữa:
- Các đặc trưng đầu vào (Input features).
- Output mục tiêu (biến `charges` hoặc các biến đầu ra tương đương).

**2.3. Dashboard Trực quan hóa**
Bố cục bao gồm 3 thành phần chính:
1. **Biểu đồ kích thước cụm:** Pie chart hoặc Bar chart thể hiện tỷ trọng mẫu phân bổ vào từng cụm.
2. **Biểu đồ phân phối mục tiêu:** Boxplot thể hiện sự phân tán của biến `charges` theo từng cụm.
3. **Heatmap đặc trưng:** Thể hiện mức độ phân bổ và giá trị trung bình của các features cấu thành nên từng Cluster.
4. **Biểu đồ trực quan, phân biệt các mẫu dữ liệu thuộc mỗi cụm**

### 2\. GMM

Định nghĩa các hàm

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from scipy.stats import f_oneway, kruskal
import itertools

warnings.filterwarnings('ignore')

# ─── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#F8F9FA', 'axes.facecolor': '#FFFFFF',
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linestyle': '--',
    'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.labelsize': 10, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# Bảng màu phân biệt tối đa 8 cụm + nhiễu (-1)
CLUSTER_COLORS = ['#2E86AB','#E84855','#3BB273','#F18F01',
                  '#9B5DE5','#00B4D8','#FF6B6B','#4ECDC4']
NOISE_COLOR    = '#AAAAAA'
CMAP_SEQ       = 'viridis'
SEP            = "=" * 65
SEP_THIN       = "-" * 65

def header(t):
    print(f"\n{SEP}\n  {t}\n{SEP}")

def sub(t):
    print(f"\n  ▶ {t}\n  {SEP_THIN}")

def cluster_color(label):
    if label == -1:
        return NOISE_COLOR
    return CLUSTER_COLORS[label % len(CLUSTER_COLORS)]
# 1a. GMM thủ công (EM Algorithm)
class GMMManual:
    """
    Gaussian Mixture Model cài thủ công bằng thuật toán EM.
    Hỗ trợ covariance_type = 'full' | 'diag' | 'spherical'
    """
    def __init__(self, n_components=3, covariance_type='full',
                 max_iter=200, tol=1e-4, n_init=3, random_state=None):
        self.K              = n_components
        self.covariance_type = covariance_type
        self.max_iter       = max_iter
        self.tol            = tol
        self.n_init         = n_init
        self.random_state   = random_state
        # Kết quả sau fit
        self.weights_       = None   # (K,)      mixing weights π_k
        self.means_         = None   # (K, D)    vector trung bình μ_k
        self.covariances_   = None   # (K, D, D) ma trận hiệp phương sai Σ_k
        self.n_iter_        = 0
        self._log_likelihood_history = []

    # Helpers
    def _init_params(self, X, rng):
        """Khởi tạo tham số bằng KMeans++."""
        N, D = X.shape
        # Dùng KMeans để khởi tạo means
        from sklearn.cluster import KMeans
        km = KMeans(n_clusters=self.K, init='k-means++',
                    n_init=1, random_state=rng.integers(0, 9999))
        km.fit(X)
        labels = km.labels_

        weights = np.bincount(labels, minlength=self.K) / N  # π_k
        means   = km.cluster_centers_.copy()                  # μ_k

        covs = []
        for k in range(self.K):
            Xk = X[labels == k]
            if len(Xk) < 2:
                Xk = X  # fallback nếu cụm rỗng
            C = np.cov(Xk.T) if D > 1 else np.array([[Xk.var()]])
            C = self._regularize_cov(C, D)
            covs.append(C)

        return weights, means, np.array(covs)

    def _regularize_cov(self, C, D, reg=1e-6):
        """Thêm regularization để tránh singular matrix."""
        return C + reg * np.eye(D)

    def _log_gaussian(self, X, mu, Sigma):
        """
        Log pdf của phân phối Gaussian đa chiều N(μ, Σ).
        X: (N, D), mu: (D,), Sigma: (D, D)
        Trả về log p(x | μ, Σ): (N,)
        """
        D = X.shape[1]
        diff = X - mu  # (N, D)
        # Cholesky decomposition để tính log-det và inverse ổn định hơn
        try:
            L = np.linalg.cholesky(Sigma)
            log_det = 2 * np.sum(np.log(np.diag(L)))
            # Solve L * v = diff.T  =>  v = L^{-1} diff.T
            v = np.linalg.solve(L, diff.T)  # (D, N)
            maha = np.sum(v ** 2, axis=0)   # (N,) — Mahalanobis^2
        except np.linalg.LinAlgError:
            Sigma = self._regularize_cov(Sigma, D, reg=1e-4)
            L = np.linalg.cholesky(Sigma)
            log_det = 2 * np.sum(np.log(np.diag(L)))
            v = np.linalg.solve(L, diff.T)
            maha = np.sum(v ** 2, axis=0)

        log_p = -0.5 * (D * np.log(2 * np.pi) + log_det + maha)
        return log_p  # (N,)

    # E-step
    def _e_step(self, X):
        """
        E-step: tính trách nhiệm (responsibility) r_{nk}.
        r_{nk} = π_k * N(x_n | μ_k, Σ_k) / Σ_j π_j * N(x_n | μ_j, Σ_j)
        Trả về:
          log_resp : (N, K)  — log responsibilities
          log_lik  : float   — log likelihood trung bình
        """
        N = X.shape[0]
        log_resp = np.zeros((N, self.K))

        for k in range(self.K):
            log_resp[:, k] = (np.log(self.weights_[k] + 1e-300)
                              + self._log_gaussian(X, self.means_[k],
                                                   self.covariances_[k]))

        # Log-sum-exp trick để tránh underflow
        log_sum = np.logaddexp.reduce(log_resp, axis=1)  # (N,)
        log_resp -= log_sum[:, np.newaxis]               # normalize
        log_lik   = log_sum.mean()
        return log_resp, log_lik

    # M-step
    def _m_step(self, X, log_resp):
        """
        M-step: cập nhật π, μ, Σ từ responsibilities.
        """
        N, D = X.shape
        resp = np.exp(log_resp)          # (N, K)
        Nk   = resp.sum(axis=0)          # (K,) — effective count

        # π_k
        self.weights_ = Nk / N

        # μ_k
        self.means_ = (resp.T @ X) / Nk[:, np.newaxis]  # (K, D)

        # Σ_k
        covs = []
        for k in range(self.K):
            diff = X - self.means_[k]           # (N, D)
            rk   = resp[:, k]                   # (N,)
            # Weighted outer product: Σ_k = (1/N_k) Σ_n r_{nk} (x_n-μ_k)(x_n-μ_k)^T
            C = (rk[:, np.newaxis] * diff).T @ diff / Nk[k]  # (D, D)
            C = self._regularize_cov(C, D)
            covs.append(C)
        self.covariances_ = np.array(covs)

    # Fit
    def fit(self, X):
        """
        Chạy EM nhiều lần (n_init), giữ lần có log-likelihood cao nhất.
        """
        X = np.array(X, dtype=np.float64)
        best_ll   = -np.inf
        best_params = None

        for init in range(self.n_init):
            rng = np.random.default_rng(
                None if self.random_state is None
                else self.random_state + init * 17
            )
            self.weights_, self.means_, self.covariances_ = \
                self._init_params(X, rng)

            ll_history = []
            prev_ll = -np.inf

            for it in range(self.max_iter):
                # E-step
                log_resp, ll = self._e_step(X)
                ll_history.append(ll)
                # Kiểm tra hội tụ
                if abs(ll - prev_ll) < self.tol:
                    break
                prev_ll = ll
                # M-step
                self._m_step(X, log_resp)

            if ll > best_ll:
                best_ll = ll
                best_params = (
                    self.weights_.copy(),
                    self.means_.copy(),
                    self.covariances_.copy(),
                    it + 1,
                    ll_history,
                )

        # Lưu params tốt nhất
        (self.weights_, self.means_, self.covariances_,
         self.n_iter_, self._log_likelihood_history) = best_params
        return self

    # Predict
    def predict_proba(self, X):
        """Trả về ma trận xác suất posterior (N, K)."""
        X = np.array(X, dtype=np.float64)
        log_resp, _ = self._e_step(X)
        return np.exp(log_resp)

    def predict(self, X):
        """Trả về nhãn cụm cứng (N,)."""
        return np.argmax(self.predict_proba(X), axis=1)

    def score(self, X):
        """Log-likelihood trung bình trên X."""
        X = np.array(X, dtype=np.float64)
        _, ll = self._e_step(X)
        return ll

    # BIC / AIC
    def _n_params(self, D):
        """Số tham số tự do của model."""
        # means: K*D, covariances (full): K*D*(D+1)/2, weights: K-1
        return self.K * D + self.K * D * (D + 1) // 2 + (self.K - 1)

    def bic(self, X):
        X = np.array(X, dtype=np.float64)
        N, D = X.shape
        p = self._n_params(D)
        return -2 * self.score(X) * N + p * np.log(N)

    def aic(self, X):
        X = np.array(X, dtype=np.float64)
        N, D = X.shape
        p = self._n_params(D)
        return -2 * self.score(X) * N + 2 * p

# 1. LOAD & CHUẨN BỊ DỮ LIỆU
header("1. LOAD & CHUẨN BỊ DỮ LIỆU")

CSV_PATH = "support2_final.csv"

df = pd.read_csv(CSV_PATH)
print(f"  Đã load: {CSV_PATH}")

if df.isnull().sum().sum() > 0:
    df = df.fillna(df.median(numeric_only=True))

X_raw      = df.drop(columns=["charges"])
y_target   = df["charges"].values
feat_names = list(X_raw.columns)
N, D       = X_raw.shape

print(f"  Shape dữ liệu đầu vào (bỏ 'charges'): {N} mẫu × {D} chiều")
print(f"  Target 'charges': mean={y_target.mean():.1f}, std={y_target.std():.1f}, "
      f"min={y_target.min():.1f}, max={y_target.max():.1f}")




In [ ]:
X_scaled = X_raw

# PCA 2D & 3D để trực quan hóa
pca2      = PCA(n_components=2, random_state=42)
X_pca2    = pca2.fit_transform(X_scaled)
pca3      = PCA(n_components=3, random_state=42)
X_pca3    = pca3.fit_transform(X_scaled)
var2      = pca2.explained_variance_ratio_ * 100
var3      = pca3.explained_variance_ratio_ * 100

print(f"\n  PCA 2D giải thích: {var2.sum():.1f}%  "
      f"(PC1={var2[0]:.1f}%, PC2={var2[1]:.1f}%)")
print(f"  PCA 3D giải thích: {var3.sum():.1f}%")

header("2. CHỌN SỐ CỤM TỐI ƯU")

k_range    = range(2, min(11, N//10 + 2))
inertias   = []
bic_gmm    = []
aic_gmm    = []
sil_gmm    = []
ch_gmm     = []
db_gmm     = []
print(f"\n  {'k':>4}  {'Silhouette':>12}  {'CH Score':>10}  "
      f"{'DB Score':>10}  {'GMM BIC':>12}  {'GMM AIC':>12}")
print(f"  {'-'*76}")

for k in k_range:
    # GMM
    gmm  = GMMManual(n_components=k, random_state=42, n_init=3)
    gmm.fit(X_scaled)
    lgmm = gmm.predict(X_scaled)
    bic_gmm.append(gmm.bic(X_scaled))
    aic_gmm.append(gmm.aic(X_scaled))
    sil_gmm.append(silhouette_score(X_scaled, lgmm))
    ch_gmm.append(calinski_harabasz_score(X_scaled, lgmm))
    db_gmm.append(davies_bouldin_score(X_scaled, lgmm))
    print(f"  {k:>4}  {sil_gmm[-1]:>12.4f}  {ch_gmm[-1]:>10.1f}  "
          f"{db_gmm[-1]:>10.4f}  {bic_gmm[-1]:>12.1f}  {aic_gmm[-1]:>12.1f}")

best_k_bic = list(k_range)[np.argmin(bic_gmm)]
best_k = 2 # Chọn K = 2 (tối ưu)
print(f"  → K tối ưu (BIC GMM)           : k = {best_k_bic}")
print(f"  → K tối ưu                     : k = {best_k}")

# Biểu đồ 1: Chọn k
# Đổi thành 2 hàng, 2 cột để chứa đủ 4 loại chỉ số
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Chọn số cụm tối ưu (k) cho GMM", fontsize=13, fontweight='bold')
ks = list(k_range)

# 1. Đồ thị Silhouette (Hàng 0, Cột 0)
ax = axes[0, 0]
ax.plot(ks, sil_gmm, 's--', color='#F18F01', linewidth=2, markersize=6, label='GMM')
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score (cao hơn = tốt hơn)")
ax.legend(fontsize=9)

# 2. Đồ thị Davies-Bouldin (Hàng 0, Cột 1)
ax = axes[0, 1]
ax.plot(ks, db_gmm, 'o-', color='#E84855', linewidth=2, markersize=6, label='GMM')
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("DB Score")
ax.set_title("Davies-Bouldin Score (thấp hơn = tốt hơn)")
ax.legend(fontsize=9)

# 3. Đồ thị Calinski-Harabasz (Hàng 1, Cột 0) - THÊM MỚI VÀO ĐÂY
ax = axes[1, 0]
ax.plot(ks, ch_gmm, 'd-.', color='#3BB273', linewidth=2, markersize=6, label='GMM')
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("CH Score")
ax.set_title("Calinski-Harabasz Score (cao hơn = tốt hơn)")
ax.legend(fontsize=9)

# 4. Đồ thị BIC & AIC (Hàng 1, Cột 1)
ax = axes[1, 1]
ax.plot(ks, bic_gmm, 'o-', color='#00B4D8', linewidth=2, markersize=6, label='BIC')
ax.plot(ks, aic_gmm, 's--', color='#FF6B6B', linewidth=2, markersize=6, label='AIC')
ax.axvline(best_k_bic, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("Score")
ax.set_title("GMM: BIC & AIC (thấp hơn = tốt hơn)")
ax.legend(fontsize=9)

# 4. GMM — GAUSSIAN MIXTURE MODEL (EM)
header(f"4. GMM — GAUSSIAN MIXTURE MODEL  (k = {best_k})")

gmm_final  = GMMManual(n_components=best_k, covariance_type='full',
                               random_state=42, n_init=5, max_iter=200)
gmm_final.fit(X_scaled)
labels_gmm = gmm_final.predict(X_scaled)
proba_gmm  = gmm_final.predict_proba(X_scaled)
means_gmm  = gmm_final.means_
covs_gmm   = gmm_final.covariances_
weights_gmm = gmm_final.weights_

sub("4a. Thông tin cụm")
print(f"  {'Cụm':>5}  {'Số mẫu':>8}  {'% mẫu':>8}  {'Weight GMM':>12}  "
      f"{'charges mean':>14}  {'charges std':>13}")
print(f"  {'-'*66}")
for c in sorted(set(labels_gmm)):
    mask = labels_gmm == c
    print(f"  {c:>5}  {mask.sum():>8}  {mask.mean()*100:>7.1f}%  "
          f"{weights_gmm[c]:>12.4f}  {y_target[mask].mean():>14.1f}  "
          f"{y_target[mask].std():>13.1f}")

sub("4b. Tham số GMM (EM Algorithm)")
print(f"  Log-likelihood cuối  : {gmm_final.score(X_scaled)*N:.2f}")
print(f"  BIC                  : {gmm_final.bic(X_scaled):.2f}")
print(f"  AIC                  : {gmm_final.aic(X_scaled):.2f}")
print(f"  Số vòng lặp EM       : {gmm_final.n_iter_}")
print(f"  Covariance type      : full")
print(f"\n  Trọng số (mixing weights) các Gaussian:")
for c in range(best_k):
    print(f"    Component {c}: π = {weights_gmm[c]:.4f}")

sub("4c. Độ đo đánh giá")
sil_gmm_f = silhouette_score(X_scaled, labels_gmm)
ch_gmm_f  = calinski_harabasz_score(X_scaled, labels_gmm)
db_gmm_f  = davies_bouldin_score(X_scaled, labels_gmm)
sil_samples_gmm = silhouette_samples(X_scaled, labels_gmm)

print(f"  Silhouette Score       : {sil_gmm_f:.4f}")
print(f"  Calinski-Harabasz Score: {ch_gmm_f:.2f}")
print(f"  Davies-Bouldin Score   : {db_gmm_f:.4f}")

sub("4d. Xác suất thuộc cụm (5 mẫu đầu)")
prob_df = pd.DataFrame(proba_gmm, columns=[f'P(Cụm_{c})' for c in range(best_k)])
prob_df['Cụm gán'] = labels_gmm
prob_df['Max prob'] = proba_gmm.max(axis=1)
print(prob_df.head(10).round(4).to_string())
print(f"\n  Trung bình Max probability: {prob_df['Max prob'].mean():.4f}")
print(f"  (Giá trị gần 1.0 → mẫu nằm rõ ràng trong 1 cụm)")

sub("4e. Kiểm định ANOVA / Kruskal-Wallis")
groups_gmm = [y_target[labels_gmm == c] for c in sorted(set(labels_gmm))]
f_g, p_g   = f_oneway(*groups_gmm)
h_g, p_kg  = kruskal(*groups_gmm)
print(f"  ANOVA     : F = {f_g:.4f},  p = {p_g:.6f}  "
      f"{'→ Có sự khác biệt đáng kể' if p_g < 0.05 else '→ Không đáng kể'}")
print(f"  Kruskal-W : H = {h_g:.4f},  p = {p_kg:.6f}  "
      f"{'→ Có sự khác biệt đáng kể' if p_kg < 0.05 else '→ Không đáng kể'}")


# Biểu đồ 3: GMM
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f"GMM Clustering (k={best_k}, covariance=full)", fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)

# PCA 2D
ax = fig.add_subplot(gs[0, :2])
for c in sorted(set(labels_gmm)):
    mask = labels_gmm == c
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
               color=cluster_color(c), alpha=0.7, s=20, label=f'Cụm {c}')
means_pca = pca2.transform(means_gmm)
ax.scatter(means_pca[:, 0], means_pca[:, 1],
           marker='D', s=200, color='black', zorder=5, label='Mean')
for c, (cx, cy) in enumerate(means_pca):
    ax.annotate(f'G{c}', (cx, cy), textcoords="offset points",
                xytext=(6, 6), fontsize=9, fontweight='bold')
ax.set_xlabel(f"PC1 ({var2[0]:.1f}%)"); ax.set_ylabel(f"PC2 ({var2[1]:.1f}%)")
ax.set_title("GMM — Scatter PCA 2D + Means")
ax.legend(fontsize=8, markerscale=1.5)

# Uncertainty (1 - max proba)
ax = fig.add_subplot(gs[0, 2:])
uncertainty = 1 - proba_gmm.max(axis=1)
sc = ax.scatter(X_pca2[:, 0], X_pca2[:, 1], c=uncertainty,
                cmap='hot_r', alpha=0.8, s=20)
plt.colorbar(sc, ax=ax, label='Uncertainty (1 - max P)')
ax.set_xlabel(f"PC1"); ax.set_ylabel(f"PC2")
ax.set_title("GMM — Uncertainty (đỏ = không chắc chắn)")

# Silhouette
ax = fig.add_subplot(gs[1, :2])
y_lower = 10
for c in sorted(set(labels_gmm)):
    c_sil = np.sort(sil_samples_gmm[labels_gmm == c])
    y_upper = y_lower + len(c_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     alpha=0.75, color=cluster_color(c))
    ax.text(-0.05, (y_lower + y_upper)/2, str(c), fontsize=8)
    y_lower = y_upper + 10
ax.axvline(sil_gmm_f, color='#E84855', linestyle='--', linewidth=1.5,
           label=f'Mean Sil = {sil_gmm_f:.3f}')
ax.set_xlabel("Silhouette Coefficient")
ax.set_title("Silhouette Plot — GMM")
ax.legend(fontsize=8)

# Box plot charges
ax = fig.add_subplot(gs[1, 2:])
bp_data2 = [y_target[labels_gmm == c] for c in sorted(set(labels_gmm))]
bp2 = ax.boxplot(bp_data2, patch_artist=True, notch=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, c in zip(bp2['boxes'], sorted(set(labels_gmm))):
    patch.set_facecolor(cluster_color(c)); patch.set_alpha(0.75)
ax.set_xticklabels([f'Cụm {c}' for c in sorted(set(labels_gmm))])
ax.set_ylabel("charges")
ax.set_title("Phân phối charges theo GMM cụm")

# Probability heatmap
ax = fig.add_subplot(gs[2, :2])
prob_sorted = prob_df.sort_values('Cụm gán').reset_index(drop=True)
prob_mat    = prob_sorted[[f'P(Cụm_{c})' for c in range(best_k)]].values
im = ax.imshow(prob_mat.T, aspect='auto', cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_yticks(range(best_k))
ax.set_yticklabels([f'P(Cụm_{c})' for c in range(best_k)])
ax.set_xlabel("Mẫu dữ liệu (sắp xếp theo cụm)")
ax.set_title("GMM — Ma trận xác suất thuộc cụm (soft assignment)")

# Heatmap đặc trưng
ax = fig.add_subplot(gs[2, 2:])
gmm_heat = pd.DataFrame(means_gmm, columns=feat_names,
                          index=[f'G{c}' for c in range(best_k)])
show_feat2 = gmm_heat.std().nlargest(15).index
sns.heatmap(gmm_heat[show_feat2].T, ax=ax, cmap='RdBu_r', center=0,
            annot=True if best_k <= 5 else False,
            fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, cbar_kws={'shrink': 0.8})
ax.set_title("Heatmap đặc trưng trung bình — GMM (Z-score)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
#plt.tight_layout()
plt.show()

## 2\. Dữ liệu giảm chiều PCAmix

### 1\. KMeans

### 2\. GMM

In [ ]:
np.allclose(X_raw.values, X_mixed)
print(ALL_FEATURES)
print(X_raw.values)
print(X_mixed)

In [ ]:
pcamix_cols = [f'PCAmix{i+1}' for i in range(X_pcamix.shape[1])]

pcamix_df = pd.DataFrame(X_pcamix, columns=pcamix_cols)

print(pcamix_df.head())
print(pcamix_df.isna().sum())

In [ ]:
pcamix = PCAmix_Manual(
        n_components=25,
        discrete_features=discrete_indices,
        continuous_features=continuous_indices
    )
X_pcamix = pcamix.fit_transform(X_mixed)

header("2. CHỌN SỐ CỤM TỐI ƯU")

k_range    = range(2, min(11, N//10 + 2))
inertias   = []
bic_gmm    = []
aic_gmm    = []
sil_gmm    = []
ch_gmm     = []
db_gmm     = []
print(f"\n  {'k':>4}  {'Silhouette':>12}  {'CH Score':>10}  "
      f"{'DB Score':>10}  {'GMM BIC':>12}  {'GMM AIC':>12}")
print(f"  {'-'*76}")

for k in k_range:
    # GMM
    gmm  = GMMManual(n_components=k, random_state=42, n_init=3)
    gmm.fit(X_pcamix)
    lgmm = gmm.predict(X_pcamix)
    bic_gmm.append(gmm.bic(X_pcamix))
    aic_gmm.append(gmm.aic(X_pcamix))
    sil_gmm.append(silhouette_score(X_pcamix, lgmm))
    ch_gmm.append(calinski_harabasz_score(X_pcamix, lgmm))
    db_gmm.append(davies_bouldin_score(X_pcamix, lgmm))
    print(f"  {k:>4}  {sil_gmm[-1]:>12.4f}  {ch_gmm[-1]:>10.1f}  "
          f"{db_gmm[-1]:>10.4f}  {bic_gmm[-1]:>12.1f}  {aic_gmm[-1]:>12.1f}")

best_k_bic = list(k_range)[np.argmin(bic_gmm)]
best_k_sil = list(k_range)[np.argmax(sil_gmm)]
best_k = 6 # Chọn K = 6 (tối ưu)
print(f"  → K tối ưu (BIC GMM)           : k = {best_k_bic}")
print(f"  → K tối ưu                     : k = {best_k}")


# Biểu đồ 1: Chọn k
# Đổi thành 2 hàng, 2 cột để chứa đủ 4 loại chỉ số
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Chọn số cụm tối ưu (k) cho GMM", fontsize=13, fontweight='bold')
ks = list(k_range)

# 1. Đồ thị Silhouette (Hàng 0, Cột 0)
ax = axes[0, 0]
ax.plot(ks, sil_gmm, 's--', color='#F18F01', linewidth=2, markersize=6, label='GMM')
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score (cao hơn = tốt hơn)")
ax.legend(fontsize=9)

# 2. Đồ thị Davies-Bouldin (Hàng 0, Cột 1)
ax = axes[0, 1]
ax.plot(ks, db_gmm, 'o-', color='#E84855', linewidth=2, markersize=6, label='GMM')
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("DB Score")
ax.set_title("Davies-Bouldin Score (thấp hơn = tốt hơn)")
ax.legend(fontsize=9)

# 3. Đồ thị Calinski-Harabasz (Hàng 1, Cột 0) - THÊM MỚI VÀO ĐÂY
ax = axes[1, 0]
ax.plot(ks, ch_gmm, 'd-.', color='#3BB273', linewidth=2, markersize=6, label='GMM')
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("CH Score")
ax.set_title("Calinski-Harabasz Score (cao hơn = tốt hơn)")
ax.legend(fontsize=9)

# 4. Đồ thị BIC & AIC (Hàng 1, Cột 1)
ax = axes[1, 1]
ax.plot(ks, bic_gmm, 'o-', color='#00B4D8', linewidth=2, markersize=6, label='BIC')
ax.plot(ks, aic_gmm, 's--', color='#FF6B6B', linewidth=2, markersize=6, label='AIC')
ax.axvline(best_k_bic, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("Score")
ax.set_title("GMM: BIC & AIC (thấp hơn = tốt hơn)")
ax.legend(fontsize=9)

# 4. GMM — GAUSSIAN MIXTURE MODEL (EM)
header(f"4. GMM — GAUSSIAN MIXTURE MODEL  (k = {best_k})")

gmm_final  = GMMManual(n_components=best_k, covariance_type='full',
                               random_state=42, n_init=5, max_iter=200)
gmm_final.fit(X_pcamix)
labels_gmm = gmm_final.predict(X_pcamix)
proba_gmm  = gmm_final.predict_proba(X_pcamix)
means_gmm  = gmm_final.means_
covs_gmm   = gmm_final.covariances_
weights_gmm = gmm_final.weights_

sub("4a. Thông tin cụm")
print(f"  {'Cụm':>5}  {'Số mẫu':>8}  {'% mẫu':>8}  {'Weight GMM':>12}  "
      f"{'charges mean':>14}  {'charges std':>13}")
print(f"  {'-'*66}")
for c in sorted(set(labels_gmm)):
    mask = labels_gmm == c
    print(f"  {c:>5}  {mask.sum():>8}  {mask.mean()*100:>7.1f}%  "
          f"{weights_gmm[c]:>12.4f}  {y_target[mask].mean():>14.1f}  "
          f"{y_target[mask].std():>13.1f}")

sub("4b. Tham số GMM (EM Algorithm)")
print(f"  Log-likelihood cuối  : {gmm_final.score(X_pcamix)*N:.2f}")
print(f"  BIC                  : {gmm_final.bic(X_pcamix):.2f}")
print(f"  AIC                  : {gmm_final.aic(X_pcamix):.2f}")
print(f"  Số vòng lặp EM       : {gmm_final.n_iter_}")
print(f"  Covariance type      : full")
print(f"\n  Trọng số (mixing weights) các Gaussian:")
for c in range(best_k):
    print(f"    Component {c}: π = {weights_gmm[c]:.4f}")

sub("4c. Độ đo đánh giá")
sil_gmm_f = silhouette_score(X_pcamix, labels_gmm)
ch_gmm_f  = calinski_harabasz_score(X_pcamix, labels_gmm)
db_gmm_f  = davies_bouldin_score(X_pcamix, labels_gmm)
sil_samples_gmm = silhouette_samples(X_pcamix, labels_gmm)

print(f"  Silhouette Score       : {sil_gmm_f:.4f}")
print(f"  Calinski-Harabasz Score: {ch_gmm_f:.2f}")
print(f"  Davies-Bouldin Score   : {db_gmm_f:.4f}")

sub("4d. Xác suất thuộc cụm (5 mẫu đầu)")
prob_df = pd.DataFrame(proba_gmm, columns=[f'P(Cụm_{c})' for c in range(best_k)])
prob_df['Cụm gán'] = labels_gmm
prob_df['Max prob'] = proba_gmm.max(axis=1)
print(prob_df.head(10).round(4).to_string())
print(f"\n  Trung bình Max probability: {prob_df['Max prob'].mean():.4f}")
print(f"  (Giá trị gần 1.0 → mẫu nằm rõ ràng trong 1 cụm)")

sub("4e. Kiểm định ANOVA / Kruskal-Wallis")
groups_gmm = [y_target[labels_gmm == c] for c in sorted(set(labels_gmm))]
f_g, p_g   = f_oneway(*groups_gmm)
h_g, p_kg  = kruskal(*groups_gmm)
print(f"  ANOVA     : F = {f_g:.4f},  p = {p_g:.6f}  "
      f"{'→ Có sự khác biệt đáng kể' if p_g < 0.05 else '→ Không đáng kể'}")
print(f"  Kruskal-W : H = {h_g:.4f},  p = {p_kg:.6f}  "
      f"{'→ Có sự khác biệt đáng kể' if p_kg < 0.05 else '→ Không đáng kể'}")


# Biểu đồ 3: GMM
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f"GMM Clustering (k={best_k}, covariance=full)", fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)

# PCA 2D
ax = fig.add_subplot(gs[0, :2])
for c in sorted(set(labels_gmm)):
    mask = labels_gmm == c
    ax.scatter(X_pcamix[mask, 0], X_pcamix[mask, 1],
               color=cluster_color(c), alpha=0.7, s=20, label=f'Cụm {c}')
means_pcamix = means_gmm[:, :2]
ax.scatter(means_pcamix[:, 0], means_pcamix[:, 1],
           marker='D', s=200, color='black', zorder=5, label='Mean')
for c, (cx, cy) in enumerate(means_pcamix):
    ax.annotate(f'G{c}', (cx, cy), textcoords="offset points",
                xytext=(6, 6), fontsize=9, fontweight='bold')
ax.set_xlabel("PCAmix1"); ax.set_ylabel(f"PCAmix2")
ax.set_title("GMM — Scatter PCAmix 25D (2 chiều đầu) + Means")
ax.legend(fontsize=8, markerscale=1.5)

# Uncertainty (1 - max proba)
ax = fig.add_subplot(gs[0, 2:])
uncertainty = 1 - proba_gmm.max(axis=1)
sc = ax.scatter(X_pcamix[:, 0], X_pcamix[:, 1], c=uncertainty,
                cmap='hot_r', alpha=0.8, s=20)
plt.colorbar(sc, ax=ax, label='Uncertainty (1 - max P)')
ax.set_xlabel("PCAmix1"); ax.set_ylabel("PCAmix2")
ax.set_title("GMM — Uncertainty (đỏ = không chắc chắn)")

# Silhouette
ax = fig.add_subplot(gs[1, :2])
y_lower = 10
for c in sorted(set(labels_gmm)):
    c_sil = np.sort(sil_samples_gmm[labels_gmm == c])
    y_upper = y_lower + len(c_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     alpha=0.75, color=cluster_color(c))
    ax.text(-0.05, (y_lower + y_upper)/2, str(c), fontsize=8)
    y_lower = y_upper + 10
ax.axvline(sil_gmm_f, color='#E84855', linestyle='--', linewidth=1.5,
           label=f'Mean Sil = {sil_gmm_f:.3f}')
ax.set_xlabel("Silhouette Coefficient")
ax.set_title("Silhouette Plot — GMM")
ax.legend(fontsize=8)

# Box plot charges
ax = fig.add_subplot(gs[1, 2:])
bp_data2 = [y_target[labels_gmm == c] for c in sorted(set(labels_gmm))]
bp2 = ax.boxplot(bp_data2, patch_artist=True, notch=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, c in zip(bp2['boxes'], sorted(set(labels_gmm))):
    patch.set_facecolor(cluster_color(c)); patch.set_alpha(0.75)
ax.set_xticklabels([f'Cụm {c}' for c in sorted(set(labels_gmm))])
ax.set_ylabel("charges")
ax.set_title("Phân phối charges theo GMM cụm")

# Probability heatmap
ax = fig.add_subplot(gs[2, :2])
prob_sorted = prob_df.sort_values('Cụm gán').reset_index(drop=True)
prob_mat    = prob_sorted[[f'P(Cụm_{c})' for c in range(best_k)]].values
im = ax.imshow(prob_mat.T, aspect='auto', cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_yticks(range(best_k))
ax.set_yticklabels([f'P(Cụm_{c})' for c in range(best_k)])
ax.set_xlabel("Mẫu dữ liệu (sắp xếp theo cụm)")
ax.set_title("GMM — Ma trận xác suất thuộc cụm (soft assignment)")

# Heatmap đặc trưng
umap_cols = [f'UMAP{i+1}' for i in range(X_pcamix.shape[1])]

ax = fig.add_subplot(gs[2, 2:])
gmm_heat = pd.DataFrame(means_gmm, columns=umap_cols,
                          index=[f'G{c}' for c in range(best_k)])
show_feat2 = gmm_heat.std().nlargest(15).index
sns.heatmap(gmm_heat[show_feat2].T, ax=ax, cmap='RdBu_r', center=0,
            annot=True if best_k <= 5 else False,
            fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, cbar_kws={'shrink': 0.8})
ax.set_title("Heatmap đặc trưng trung bình — GMM (Z-score)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
plt.tight_layout()
plt.show()

## 3\. Dữ liệu giảm chiều UMAP

### 1\. KMeans

### 2\. GMM

In [ ]:
from umap import UMAP

umap_model = UMAP(
    n_neighbors=20,      # Tinh chỉnh từ 5 - 50 tùy độ thưa thớt của cụm
    min_dist=0.1,        # Tinh chỉnh từ 0.001 - 0.5 để kiểm soát độ chặt của cụm
    n_components=15,
    random_state=42,
    metric = 'cosine'
)

X_umap = umap_model.fit_transform(X_scaled)

header("2. CHỌN SỐ CỤM TỐI ƯU")

k_range    = range(2, min(11, N//10 + 2))
inertias   = []
bic_gmm    = []
aic_gmm    = []
sil_gmm    = []
ch_gmm     = []
db_gmm     = []
print(f"\n  {'k':>4}  {'Silhouette':>12}  {'CH Score':>10}  "
      f"{'DB Score':>10}  {'GMM BIC':>12}  {'GMM AIC':>12}")
print(f"  {'-'*76}")

for k in k_range:
    # GMM
    gmm  = GMMManual(n_components=k, random_state=42, n_init=3)
    gmm.fit(X_umap)
    lgmm = gmm.predict(X_umap)
    bic_gmm.append(gmm.bic(X_umap))
    aic_gmm.append(gmm.aic(X_umap))
    sil_gmm.append(silhouette_score(X_umap, lgmm))
    ch_gmm.append(calinski_harabasz_score(X_umap, lgmm))
    db_gmm.append(davies_bouldin_score(X_umap, lgmm))
    print(f"  {k:>4}  {sil_gmm[-1]:>12.4f}  {ch_gmm[-1]:>10.1f}  "
          f"{db_gmm[-1]:>10.4f}  {bic_gmm[-1]:>12.1f}  {aic_gmm[-1]:>12.1f}")

best_k_bic = list(k_range)[np.argmin(bic_gmm)]
best_k_sil = list(k_range)[np.argmax(sil_gmm)]
best_k = 6 # Chọn K = 6 (tối ưu)
print(f"  → K tối ưu (BIC GMM)           : k = {best_k_bic}")
print(f"  → K tối ưu                     : k = {best_k}")


# ─── Biểu đồ 1: Chọn k ───────────────────────────────────────────────────────
# Đổi thành 2 hàng, 2 cột để chứa đủ 4 loại chỉ số
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Chọn số cụm tối ưu (k) cho GMM", fontsize=13, fontweight='bold')
ks = list(k_range)

# 1. Đồ thị Silhouette (Hàng 0, Cột 0)
ax = axes[0, 0]
ax.plot(ks, sil_gmm, 's--', color='#F18F01', linewidth=2, markersize=6, label='GMM')
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score (cao hơn = tốt hơn)")
ax.legend(fontsize=9)

# 2. Đồ thị Davies-Bouldin (Hàng 0, Cột 1)
ax = axes[0, 1]
ax.plot(ks, db_gmm, 'o-', color='#E84855', linewidth=2, markersize=6, label='GMM')
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("DB Score")
ax.set_title("Davies-Bouldin Score (thấp hơn = tốt hơn)")
ax.legend(fontsize=9)

# 3. Đồ thị Calinski-Harabasz (Hàng 1, Cột 0) - THÊM MỚI VÀO ĐÂY
ax = axes[1, 0]
ax.plot(ks, ch_gmm, 'd-.', color='#3BB273', linewidth=2, markersize=6, label='GMM')
ax.axvline(best_k, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("CH Score")
ax.set_title("Calinski-Harabasz Score (cao hơn = tốt hơn)")
ax.legend(fontsize=9)

# 4. Đồ thị BIC & AIC (Hàng 1, Cột 1)
ax = axes[1, 1]
ax.plot(ks, bic_gmm, 'o-', color='#00B4D8', linewidth=2, markersize=6, label='BIC')
ax.plot(ks, aic_gmm, 's--', color='#FF6B6B', linewidth=2, markersize=6, label='AIC')
ax.axvline(best_k_bic, color='#E84855', linestyle='--', linewidth=1.5)
ax.set_xlabel("Số cụm k"); ax.set_ylabel("Score")
ax.set_title("GMM: BIC & AIC (thấp hơn = tốt hơn)")
ax.legend(fontsize=9)

# 4. GMM — GAUSSIAN MIXTURE MODEL (EM)
header(f"4. GMM — GAUSSIAN MIXTURE MODEL  (k = {best_k})")

gmm_final  = GMMManual(n_components=best_k, covariance_type='full',
                               random_state=42, n_init=5, max_iter=200)
gmm_final.fit(X_umap)
labels_gmm = gmm_final.predict(X_umap)
proba_gmm  = gmm_final.predict_proba(X_umap)
means_gmm  = gmm_final.means_
covs_gmm   = gmm_final.covariances_
weights_gmm = gmm_final.weights_

sub("4a. Thông tin cụm")
print(f"  {'Cụm':>5}  {'Số mẫu':>8}  {'% mẫu':>8}  {'Weight GMM':>12}  "
      f"{'charges mean':>14}  {'charges std':>13}")
print(f"  {'-'*66}")
for c in sorted(set(labels_gmm)):
    mask = labels_gmm == c
    print(f"  {c:>5}  {mask.sum():>8}  {mask.mean()*100:>7.1f}%  "
          f"{weights_gmm[c]:>12.4f}  {y_target[mask].mean():>14.1f}  "
          f"{y_target[mask].std():>13.1f}")

sub("4b. Tham số GMM (EM Algorithm)")
print(f"  Log-likelihood cuối  : {gmm_final.score(X_umap)*N:.2f}")
print(f"  BIC                  : {gmm_final.bic(X_umap):.2f}")
print(f"  AIC                  : {gmm_final.aic(X_umap):.2f}")
print(f"  Số vòng lặp EM       : {gmm_final.n_iter_}")
print(f"  Covariance type      : full")
print(f"\n  Trọng số (mixing weights) các Gaussian:")
for c in range(best_k):
    print(f"    Component {c}: π = {weights_gmm[c]:.4f}")

sub("4c. Độ đo đánh giá")
sil_gmm_f = silhouette_score(X_umap, labels_gmm)
ch_gmm_f  = calinski_harabasz_score(X_umap, labels_gmm)
db_gmm_f  = davies_bouldin_score(X_umap, labels_gmm)
sil_samples_gmm = silhouette_samples(X_umap, labels_gmm)

print(f"  Silhouette Score       : {sil_gmm_f:.4f}")
print(f"  Calinski-Harabasz Score: {ch_gmm_f:.2f}")
print(f"  Davies-Bouldin Score   : {db_gmm_f:.4f}")

sub("4d. Xác suất thuộc cụm (5 mẫu đầu)")
prob_df = pd.DataFrame(proba_gmm, columns=[f'P(Cụm_{c})' for c in range(best_k)])
prob_df['Cụm gán'] = labels_gmm
prob_df['Max prob'] = proba_gmm.max(axis=1)
print(prob_df.head(10).round(4).to_string())
print(f"\n  Trung bình Max probability: {prob_df['Max prob'].mean():.4f}")
print(f"  (Giá trị gần 1.0 → mẫu nằm rõ ràng trong 1 cụm)")

sub("4e. Kiểm định ANOVA / Kruskal-Wallis")
groups_gmm = [y_target[labels_gmm == c] for c in sorted(set(labels_gmm))]
f_g, p_g   = f_oneway(*groups_gmm)
h_g, p_kg  = kruskal(*groups_gmm)
print(f"  ANOVA     : F = {f_g:.4f},  p = {p_g:.6f}  "
      f"{'→ Có sự khác biệt đáng kể' if p_g < 0.05 else '→ Không đáng kể'}")
print(f"  Kruskal-W : H = {h_g:.4f},  p = {p_kg:.6f}  "
      f"{'→ Có sự khác biệt đáng kể' if p_kg < 0.05 else '→ Không đáng kể'}")


# ─── Biểu đồ 3: GMM ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f"GMM Clustering (k={best_k}, covariance=full)", fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)

# UMAP 2D
ax = fig.add_subplot(gs[0, :2])
for c in sorted(set(labels_gmm)):
    mask = labels_gmm == c
    ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
               color=cluster_color(c), alpha=0.7, s=20, label=f'Cụm {c}')
means_umap = means_gmm[:, :2]
ax.scatter(means_umap[:, 0], means_umap[:, 1],
           marker='D', s=200, color='black', zorder=5, label='Mean')
for c, (cx, cy) in enumerate(means_umap):
    ax.annotate(f'G{c}', (cx, cy), textcoords="offset points",
                xytext=(6, 6), fontsize=9, fontweight='bold')
ax.set_xlabel("UMAP1"); ax.set_ylabel(f"UMAP2")
ax.set_title("GMM — Scatter UMAP 15D (2 chiều đầu) + Means")
ax.legend(fontsize=8, markerscale=1.5)

# Uncertainty (1 - max proba)
ax = fig.add_subplot(gs[0, 2:])
uncertainty = 1 - proba_gmm.max(axis=1)
sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], c=uncertainty,
                cmap='hot_r', alpha=0.8, s=20)
plt.colorbar(sc, ax=ax, label='Uncertainty (1 - max P)')
ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
ax.set_title("GMM — Uncertainty (đỏ = không chắc chắn)")

# Silhouette
ax = fig.add_subplot(gs[1, :2])
y_lower = 10
for c in sorted(set(labels_gmm)):
    c_sil = np.sort(sil_samples_gmm[labels_gmm == c])
    y_upper = y_lower + len(c_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     alpha=0.75, color=cluster_color(c))
    ax.text(-0.05, (y_lower + y_upper)/2, str(c), fontsize=8)
    y_lower = y_upper + 10
ax.axvline(sil_gmm_f, color='#E84855', linestyle='--', linewidth=1.5,
           label=f'Mean Sil = {sil_gmm_f:.3f}')
ax.set_xlabel("Silhouette Coefficient")
ax.set_title("Silhouette Plot — GMM")
ax.legend(fontsize=8)

# Box plot charges
ax = fig.add_subplot(gs[1, 2:])
bp_data2 = [y_target[labels_gmm == c] for c in sorted(set(labels_gmm))]
bp2 = ax.boxplot(bp_data2, patch_artist=True, notch=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, c in zip(bp2['boxes'], sorted(set(labels_gmm))):
    patch.set_facecolor(cluster_color(c)); patch.set_alpha(0.75)
ax.set_xticklabels([f'Cụm {c}' for c in sorted(set(labels_gmm))])
ax.set_ylabel("charges")
ax.set_title("Phân phối charges theo GMM cụm")

# Probability heatmap
ax = fig.add_subplot(gs[2, :2])
prob_sorted = prob_df.sort_values('Cụm gán').reset_index(drop=True)
prob_mat    = prob_sorted[[f'P(Cụm_{c})' for c in range(best_k)]].values
im = ax.imshow(prob_mat.T, aspect='auto', cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_yticks(range(best_k))
ax.set_yticklabels([f'P(Cụm_{c})' for c in range(best_k)])
ax.set_xlabel("Mẫu dữ liệu (sắp xếp theo cụm)")
ax.set_title("GMM — Ma trận xác suất thuộc cụm (soft assignment)")

# Heatmap đặc trưng
umap_cols = [f'UMAP{i+1}' for i in range(X_umap.shape[1])]

ax = fig.add_subplot(gs[2, 2:])
gmm_heat = pd.DataFrame(means_gmm, columns=umap_cols,
                          index=[f'G{c}' for c in range(best_k)])
show_feat2 = gmm_heat.std().nlargest(15).index
sns.heatmap(gmm_heat[show_feat2].T, ax=ax, cmap='RdBu_r', center=0,
            annot=True if best_k <= 5 else False,
            fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, cbar_kws={'shrink': 0.8})
ax.set_title("Heatmap đặc trưng trung bình — GMM (Z-score)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
plt.tight_layout()
plt.show()

# IV) Phân tích Hồi quy

In [ ]:
# Thiết lập giao diện
sns.set_theme(style="whitegrid")
pastel_palette = sns.color_palette("pastel")

In [ ]:
def train_test_split(X, y, test_size=0.2, random_state=42):
    if random_state is not None:
        np.random.seed(random_state)
    indices = np.random.permutation(len(X))
    test_samples = int(len(X) * test_size)

    test_idx, train_idx = indices[:test_samples], indices[test_samples:]
    if isinstance(X, pd.DataFrame) or isinstance(X, pd.Series):
        return X.iloc[train_idx].values, X.iloc[test_idx].values, y.iloc[train_idx].values, y.iloc[test_idx].values
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [ ]:
def mean_squared_error(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mean_absolute_error(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)


In [ ]:
# ==========================================
# TRIỂN KHAI CÁC MÔ HÌNH HỒI QUY
# ==========================================

# KNNRegressor
class CustomKNNRegressor:
    def __init__(self, n_neighbors=5):
        self.k = n_neighbors

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train
        return self

    def predict(self, X_test):
        # Vectorized: tính toàn bộ distance matrix 1 lần
        # ||a-b||² = ||a||² + ||b||² - 2a·b
        X_test_sq  = np.sum(X_test ** 2, axis=1, keepdims=True)
        X_train_sq = np.sum(self.X_train ** 2, axis=1, keepdims=True)
        cross      = X_test @ self.X_train.T

        dist_matrix = np.sqrt(np.maximum(X_test_sq + X_train_sq.T - 2 * cross, 0))
        # shape: (n_test, n_train)

        k = min(self.k, len(self.X_train))
        # Lấy k chỉ số nhỏ nhất mỗi hàng
        nearest_indices = np.argpartition(dist_matrix, k, axis=1)[:, :k]

        # Tính mean của k láng giềng
        y_pred = np.mean(self.y_train[nearest_indices], axis=1)

        return y_pred

In [ ]:
# LinearRegression
class CustomLinearRegression:
    def __init__(self):
        self.theta = None

    def fit(self, X_train, y_train):
        y_train = y_train.reshape(-1)

        # Thêm bias (cột 1)
        X_b = np.c_[np.ones((X_train.shape[0], 1)), X_train]

        # Normal Equation
        self.theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y_train

        return self

    def predict(self, X_test):
        X_b = np.c_[np.ones((X_test.shape[0], 1)), X_test]
        return X_b @ self.theta

In [ ]:
# RidgeRegression
class CustomRidgeRegression:
    def __init__(self, alpha=1.0):
        self.alpha = alpha # Thành phần Regularization L2

    def fit(self, X_train, y_train):
        X_b = np.c_[np.ones((X_train.shape[0], 1)), X_train]
        I = np.eye(X_b.shape[1])
        I[0, 0] = 0
        self.theta = np.linalg.pinv(X_b.T @ X_b + self.alpha * I) @ X_b.T @ y_train
        return self

    def predict(self, X_test):
        X_b = np.c_[np.ones((X_test.shape[0], 1)), X_test]
        return X_b @ self.theta

In [ ]:
class CustomMLPRegressor:
    def __init__(self, hidden_size=32, learning_rate=0.01, epochs=500,
                 lambda_reg=0.0, batch_size=32, early_stopping=False, patience=20):
        self.hidden_size = hidden_size
        self.lr = learning_rate
        self.epochs = epochs
        self.lambda_reg = lambda_reg
        self.batch_size = batch_size
        self.early_stopping = early_stopping
        self.patience = patience

        self.loss_history = []
        self.val_loss_history = []

        self.W1, self.b1, self.W2, self.b2 = None, None, None, None

    def _init_params(self, n_features):
        np.random.seed(42)
        # He Initialization cho ReLU
        self.W1 = np.random.randn(n_features, self.hidden_size) * np.sqrt(2. / n_features)
        self.b1 = np.zeros((1, self.hidden_size))
        self.W2 = np.random.randn(self.hidden_size, 1) * np.sqrt(2. / self.hidden_size)
        self.b2 = np.zeros((1, 1))

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        y_train = y_train.reshape(-1, 1)
        if y_val is not None:
            y_val = y_val.reshape(-1, 1)

        N, d = X_train.shape
        self._init_params(d)

        best_val = np.inf
        best_weights = (self.W1.copy(), self.b1.copy(), self.W2.copy(), self.b2.copy())
        wait = 0

        # Nếu batch_size không truyền vào, mặc định lấy Mini-batch 32 (không dùng Full-batch)
        bs = self.batch_size if self.batch_size is not None else 32

        for epoch in range(self.epochs):
            # Xáo trộn dữ liệu mỗi epoch cho Mini-batch SGD chuẩn
            idx = np.random.permutation(N)
            X_shuf, y_shuf = X_train[idx], y_train[idx]

            batches = [(X_shuf[i:i+bs], y_shuf[i:i+bs]) for i in range(0, N, bs)]

            epoch_loss = 0

            for Xb, yb in batches:
                m = Xb.shape[0]

                # ==========================================
                # 1. FORWARD PASS
                # ==========================================
                Z1 = Xb.dot(self.W1) + self.b1
                A1 = np.maximum(Z1, 0)         # ReLU
                Z2 = A1.dot(self.W2) + self.b2
                Yhat = Z2                      # Linear Output

                # ==========================================
                # 2. BACKWARD PASS (Đã fix chuẩn Scale)
                # ==========================================
                E2 = (Yhat - yb) / m

                # Fix: L2 penalty scale theo N để không nuốt chửng gradient của data
                l2_term_W2 = (self.lambda_reg / N) * self.W2
                dW2 = A1.T.dot(E2) + l2_term_W2
                db2 = np.sum(E2, axis=0, keepdims=True)

                E1 = E2.dot(self.W2.T)
                E1[Z1 <= 0] = 0                # Gradient of ReLU

                l2_term_W1 = (self.lambda_reg / N) * self.W1
                dW1 = Xb.T.dot(E1) + l2_term_W1
                db1 = np.sum(E1, axis=0, keepdims=True)

                # ==========================================
                # 3. UPDATE WEIGHTS
                # ==========================================
                self.W1 -= self.lr * dW1
                self.b1 -= self.lr * db1
                self.W2 -= self.lr * dW2
                self.b2 -= self.lr * db2

                batch_loss = 0.5 * np.mean((Yhat - yb)**2) + \
                             0.5 * (self.lambda_reg / N) * (np.sum(self.W1**2) + np.sum(self.W2**2))
                epoch_loss += batch_loss * m

            self.loss_history.append(epoch_loss / N)

            # --- Validation & Early Stopping ---
            if X_val is not None and y_val is not None:
                Z1_v = X_val.dot(self.W1) + self.b1
                A1_v = np.maximum(Z1_v, 0)
                Yhat_v = A1_v.dot(self.W2) + self.b2

                val_loss = 0.5 * np.mean((Yhat_v - y_val)**2)
                self.val_loss_history.append(val_loss)

                if self.early_stopping:
                    if val_loss < best_val - 1e-7:
                        best_val = val_loss
                        best_weights = (self.W1.copy(), self.b1.copy(), self.W2.copy(), self.b2.copy())
                        wait = 0
                    else:
                        wait += 1
                        if wait >= self.patience:
                            self.W1, self.b1, self.W2, self.b2 = best_weights
                            break
        return self

    def predict(self, X_test):
        Z1 = X_test.dot(self.W1) + self.b1
        A1 = np.maximum(Z1, 0)
        Z2 = A1.dot(self.W2) + self.b2
        return Z2.flatten()

In [ ]:
# plot metrics base
def plot_metrics_bar(metrics_base, title="So sánh RMSE, MAE và R²"):
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    df_metrics = pd.DataFrame(metrics_base)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Hàm gắn label
    def add_bar_labels(ax):
        for container in ax.containers:
            ax.bar_label(container, fmt='%.3f', padding=3,
                         color="#000000", fontsize=10, fontweight='bold')

    # Vẽ từng biểu đồ
    sns.barplot(x='Tỷ lệ', y='RMSE', data=df_metrics, ax=axes[0], color='skyblue')
    axes[0].set_title('RMSE')
    add_bar_labels(axes[0])

    sns.barplot(x='Tỷ lệ', y='MAE', data=df_metrics, ax=axes[1], color='lightgreen')
    axes[1].set_title('MAE')
    add_bar_labels(axes[1])

    sns.barplot(x='Tỷ lệ', y='R2', data=df_metrics, ax=axes[2], color='salmon')
    axes[2].set_title('R² Score ')
    add_bar_labels(axes[2])

    for ax in axes:
        ax.set_xlabel('')

    plt.tight_layout()
    plt.show()

In [ ]:
# đánh giá mô hình
def evaluate_splits(X, y, test_sizes, labels, model_class, model_kwargs=None, sep = SEP):
    if model_kwargs is None:
        model_kwargs = {}

    metrics_base = []
    predictions_data = []

    print("Tỷ lệ chia       | Mẫu Train  | Mẫu Test   | RMSE     | MAE      | R2 Score")
    print(sep)

    for test_size, label in zip(test_sizes, labels):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )

        model = model_class(**model_kwargs)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)

        metrics_base.append({'Tỷ lệ': label, 'RMSE': rmse, 'MAE': mae, 'R2': r2})

        predictions_data.append({
            'label': label,
            'y_val': y_test,
            'y_pred': y_pred
        })

        print(f"{label:<16} | {len(X_train):>10,d} | {len(X_test):>10,d} "
              f"| {rmse:>8.4f} | {mae:>8.4f} | {r2:>8.4f}")

    print(sep)
    return metrics_base, predictions_data

In [ ]:
# ==========================================
# PHẦN 4: SCATTER PLOT - THỰC TẾ VS DỰ ĐOÁN
# ==========================================
def plot_actual_vs_predicted(best_models_data,
                            main_title="Scatter Plot: Thực tế vs Dự đoán",
                            palette_color='steelblue'):
    n = len(best_models_data)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
    fig.suptitle(main_title, fontsize=14, fontweight='bold')

    # Nếu chỉ có 1 plot thì axes không phải list
    if n == 1:
        axes = [axes]

    for i, model_data in enumerate(best_models_data):
        ax = axes[i]
        y_actual = model_data['y_val']
        y_predicted = model_data['y_pred']

        # Scatter
        sns.scatterplot(x=y_actual, y=y_predicted,
                        ax=ax, color=palette_color, alpha=0.6)

        # Đường y = x
        min_val = min(y_actual.min(), y_predicted.min())
        max_val = max(y_actual.max(), y_predicted.max())
        ax.plot([min_val, max_val], [min_val, max_val],
                color='red', linestyle='--', linewidth=1.5,
                label='Hoàn hảo (y=x)')

        # Tự động build subtitle dựa trên key có trong model_data
        if 'best_k' in model_data:
            subtitle = f"K tối ưu = {model_data['best_k']}"
        elif 'best_alpha' in model_data:
            subtitle = f"Alpha tối ưu = {model_data['best_alpha']}"
        else:
            subtitle = ""  # LinearRegression không có tham số

        title = f"{model_data['label']} - {subtitle}" if subtitle else model_data['label']
        ax.set_title(title, fontsize=12)

        ax.set_xlabel('Thực tế (y)')
        ax.set_ylabel('Dự đoán ($\\hat{y}$)')
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# PHẦN 5: TRỰC QUAN HÓA VÀ ĐÁNH GIÁ PHẦN DƯ (RESIDUAL PLOT)
# ==========================================

def plot_residuals(best_models_data,
                   main_title="Phân tích Phần dư (Residuals) vs Đầu vào",
                   palette_color='seagreen'):
    n = len(best_models_data)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
    fig.suptitle(main_title, fontsize=14, fontweight='bold')

    # Nếu chỉ có 1 plot thì axes không phải list
    if n == 1:
        axes = [axes]

    for i, model_data in enumerate(best_models_data):
        ax = axes[i]

        # Ép dẹp thành mảng 1D để tránh lỗi kích thước khi tính Pearson
        y_actual = np.array(model_data['y_val']).flatten()
        y_predicted = np.array(model_data['y_pred']).flatten()

        # 1. TÍNH PHẦN DƯ (Thực tế - Dự đoán)
        residuals = y_actual - y_predicted

        # 2. TÍNH HỆ SỐ TƯƠNG QUAN PEARSON
        corr, _ = pearsonr(y_predicted, residuals)

        # 3. VẼ SCATTER PLOT (X: Dự đoán, Y: Phần dư)
        sns.scatterplot(x=y_predicted, y=residuals,
                        ax=ax, color=palette_color, alpha=0.6)

        # Vẽ đường ngang y = 0 làm mốc chuẩn (Sai số bằng 0)
        ax.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Hoàn hảo (Sai số = 0)')

        # Tự động build subtitle
        if 'best_k' in model_data:
            subtitle = f"K tối ưu = {model_data['best_k']}"
        elif 'best_alpha' in model_data:
            subtitle = f"Alpha tối ưu = {model_data['best_alpha']}"
        else:
            subtitle = ""  # Các mô hình không có siêu tham số

        # Chèn thẳng hệ số Pearson vào tiêu đề của từng hình
        title = f"{model_data['label']}\n{subtitle}\nPearson r = {corr:.4f}"
        ax.set_title(title, fontsize=12)

        ax.set_xlabel('Dự đoán ($\hat{y}$) - Đại diện cho Đầu vào')
        ax.set_ylabel('Phần dư ($y - \hat{y}$)')
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# Đánh giá overfitting
# ==========================================
def analyze_overfitting(best_models_data, model_name="Model", threshold=0.1):
    print("=" * 80)
    print(f"OVERFITTING ANALYSIS - {model_name}")
    print("=" * 80)
    print(f"{'Tỉ lệ':<16} | {'R2 Train':>10} | {'R2 Val':>10} | {'ΔR2':>8} | "
          f"{'RMSE Train':>12} | {'RMSE Val':>10} | {'ΔRMSE':>8} | Kết luận")
    print("-" * 80)

    labels, delta_r2_list, delta_rmse_list = [], [], []
    r2_train_list, r2_val_list = [], []

    for m in best_models_data:
        delta_r2 = m['R2_train'] - m['R2']
        delta_rmse = m['RMSE'] - m['RMSE_train']

        labels.append(m['label'])
        delta_r2_list.append(delta_r2)
        delta_rmse_list.append(delta_rmse)
        r2_train_list.append(m['R2_train'])
        r2_val_list.append(m['R2'])

        if delta_r2 > 0.2 or delta_rmse > 0.2:
            verdict = "Overfit nặng"
        elif delta_r2 > threshold or delta_rmse > threshold:
            verdict = "Overfit nhẹ"
        else:
            verdict = "Ổn định"

        print(f"{m['label']:<16} | {m['R2_train']:>10.4f} | {m['R2']:>10.4f} | "
              f"{delta_r2:>+8.4f} | {m['RMSE_train']:>12.4f} | {m['RMSE']:>10.4f} | "
              f"{delta_rmse:>+8.4f} | {verdict}")

    print("-" * 80)

    # --- PHẦN PLOT ---
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    x = np.arange(len(labels))

    # Plot 1: R2 Train vs Val
    axes[0].bar(x - 0.2, r2_train_list, width=0.4, label="Train R2", color='skyblue')
    axes[0].bar(x + 0.2, r2_val_list, width=0.4, label="Val R2", color='orange')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=45)
    axes[0].set_ylabel("R2 Score")
    axes[0].set_title("Train vs Val R2")
    axes[0].legend()

    # Plot 2: Delta R2
    axes[1].bar(labels, delta_r2_list, color='salmon')
    axes[1].axhline(threshold, color='red', linestyle="--", label=f"Ngưỡng {threshold}")
    axes[1].axhline(0.2, color='darkred', linestyle="--", label="Overfit nặng")
    axes[1].set_xticklabels(labels, rotation=45)
    axes[1].set_ylabel("ΔR2 (Train - Val)")
    axes[1].set_title("Overfitting Gap (R2)")
    axes[1].legend()

    # Plot 3: Delta RMSE
    axes[2].bar(labels, delta_rmse_list, color='lightgreen')
    axes[2].axhline(threshold, color='red', linestyle="--")
    axes[2].axhline(0.2, color='darkred', linestyle="--")
    axes[2].set_xticklabels(labels, rotation=45)
    axes[2].set_ylabel("ΔRMSE (Val - Train)")
    axes[2].set_title("Overfitting Gap (RMSE)")

    plt.tight_layout()
    plt.show()

## 1\. Dữ liệu gốc

In [ ]:
# ==========================================
# KHỞI TẠO DỮ LIỆU & PIPELINE (Giả định X, y đã có)
# ==========================================
X = df.drop(columns=['charges']).values
y = df['charges'].values


test_sizes = [0.2, 0.3, 0.4]
labels = ['Train 80%', 'Train 70%', 'Train 60%']

models = {
    'K-NN': CustomKNNRegressor(n_neighbors=5),
    'Linear Regression': CustomLinearRegression(),
    'MLP': CustomMLPRegressor(hidden_size=32, learning_rate=0.01, epochs=300)
}

# Lưu trữ kết quả
results_list = []
prediction_data = []

### 1.1 K-NN

In [ ]:
# ==========================================
# PHẦN 3: ĐÁNH GIÁ REGULARIZATION TRÊN CẢ 3 TỈ LỆ
# ==========================================
def plot_knn_regularization(X, y, test_sizes, labels,
                           k_values=[2, 3, 5, 9, 15, 20],
                           elbow_threshold=0.01,
                           main_title="Tác động của K đến hiệu suất KNN"):

    metrics_names = ['RMSE', 'MAE', 'R2']

    fig, axes = plt.subplots(len(test_sizes), 3, figsize=(17, 5 * len(test_sizes)))
    fig.suptitle(main_title, fontsize=16, fontweight='bold')
    axes = np.array(axes).reshape(len(test_sizes), 3)

    best_models_data = []

    def find_best_k(k_values, val_r2, threshold=0.01):
        for i in range(1, len(val_r2)):
            gain = val_r2[i] - val_r2[i-1]
            if gain < threshold:
                return i-1, k_values[i-1]
        return len(val_r2)-1, k_values[-1]

    for row_idx, (test_size, label) in enumerate(zip(test_sizes, labels)):
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=test_size, random_state=42
        )

        train_metrics = {m: [] for m in metrics_names}
        val_metrics   = {m: [] for m in metrics_names}

        for k in k_values:
            knn = CustomKNNRegressor(n_neighbors=k)
            knn.fit(X_train, y_train)

            y_train_pred = knn.predict(X_train)
            y_val_pred   = knn.predict(X_val)

            train_metrics['RMSE'].append(np.sqrt(mean_squared_error(y_train, y_train_pred)))
            train_metrics['MAE'].append(mean_absolute_error(y_train, y_train_pred))
            train_metrics['R2'].append(r2_score(y_train, y_train_pred))

            val_metrics['RMSE'].append(np.sqrt(mean_squared_error(y_val, y_val_pred)))
            val_metrics['MAE'].append(mean_absolute_error(y_val, y_val_pred))
            val_metrics['R2'].append(r2_score(y_val, y_val_pred))

        # Chọn K tại điểm elbow thay vì max đơn thuần
        best_idx, best_k = find_best_k(k_values, val_metrics['R2'], elbow_threshold)

        # In để kiểm tra
        print(f"\n{label}:")
        print(f"{'K':<6} {'R2_train':>10} {'R2_val':>10} {'Gain':>8}")
        for i, k in enumerate(k_values):
            gain = val_metrics['R2'][i] - val_metrics['R2'][i-1] if i > 0 else 0
            marker = " ← elbow" if k == best_k else ""
            print(f"{k:<6} {train_metrics['R2'][i]:>10.4f} {val_metrics['R2'][i]:>10.4f} {gain:>+8.4f}{marker}")
        print(f"→ best_k={best_k} (gain tiếp theo < {elbow_threshold})")

        best_knn = CustomKNNRegressor(n_neighbors=best_k).fit(X_train, y_train)
        best_models_data.append({
            'label':      label,
            'best_k':     best_k,
            'y_val':      y_val,
            'y_pred':     best_knn.predict(X_val),
            'RMSE':       val_metrics['RMSE'][best_idx],
            'MAE':        val_metrics['MAE'][best_idx],
            'R2':         val_metrics['R2'][best_idx],
            'RMSE_train': train_metrics['RMSE'][best_idx],
            'MAE_train':  train_metrics['MAE'][best_idx],
            'R2_train':   train_metrics['R2'][best_idx]
        })

        for col_idx, metric in enumerate(metrics_names):
            ax = axes[row_idx, col_idx]

            ax.plot(k_values, train_metrics[metric],
                    marker='o', label='Train', linewidth=2)
            ax.plot(k_values, val_metrics[metric],
                    marker='s', label='Validation', linewidth=2)
            ax.axvline(best_k, color='gray', linestyle='--',
                       label=f'K tối ưu ({best_k})')

            all_vals = train_metrics[metric] + val_metrics[metric]
            ax.set_ylim(min(all_vals) * 0.95, max(all_vals) * 1.05)

            if row_idx == 0:
                ax.set_title(f'{metric}', fontsize=12)
            if col_idx == 0:
                ax.set_ylabel(label, fontweight='bold')

            ax.set_xlabel('K')
            ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    return best_models_data

In [ ]:
# ==========================================
# PHẦN 1: IN BẢNG KẾT QUẢ CƠ BẢN (K=5)
# ==========================================
metrics_knn, predictions_data = evaluate_splits(
    X, y,
    test_sizes=test_sizes,
    labels=labels,
    model_class=CustomKNNRegressor,
    model_kwargs={"n_neighbors": 5}
)

plot_metrics_bar(metrics_knn, "So sánh RMSE, MAE và R bằng phươn pháp KNN")



In [ ]:
best_knn_data = plot_knn_regularization(X, y, test_sizes, labels)



In [ ]:
analyze_overfitting(best_knn_data,   model_name="KNN")



In [ ]:
plot_actual_vs_predicted(
    best_knn_data,
    main_title="KNN - So sánh Thực tế vs Dự đoán"
)


In [ ]:
plot_residuals(
    best_knn_data,
    main_title="KNN - Phân tích phần dư theo từng tỉ lệ chia dữ liệu"
)

### 1.2 Linear Regression

In [ ]:

# ==========================================
# PHẦN 3: ĐÁNH GIÁ REGULARIZATION (Ridge Regression - Tuning Alpha)
# ==========================================

def plot_ridge_regularization(X, y, test_sizes, labels,
                               alphas=[0.01, 0.1, 1, 10, 100],
                               main_title="Tác động của Alpha đến hiệu suất Ridge Regression"):

    metrics_names = ['RMSE', 'MAE', 'R2']

    fig, axes = plt.subplots(len(test_sizes), 3, figsize=(17, 5 * len(test_sizes)))
    fig.suptitle(main_title, fontsize=16, fontweight='bold')
    axes = np.array(axes).reshape(len(test_sizes), 3)

    best_models_data = []

    for row_idx, (test_size, label) in enumerate(zip(test_sizes, labels)):
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=test_size, random_state=42
        )

        train_metrics = {m: [] for m in metrics_names}
        val_metrics   = {m: [] for m in metrics_names}

        for a in alphas:
            ridge = CustomRidgeRegression(alpha=a)
            ridge.fit(X_train, y_train)

            y_train_pred = ridge.predict(X_train)
            y_val_pred   = ridge.predict(X_val)

            train_metrics['RMSE'].append(np.sqrt(mean_squared_error(y_train, y_train_pred)))
            train_metrics['MAE'].append(mean_absolute_error(y_train, y_train_pred))
            train_metrics['R2'].append(r2_score(y_train, y_train_pred))

            val_metrics['RMSE'].append(np.sqrt(mean_squared_error(y_val, y_val_pred)))
            val_metrics['MAE'].append(mean_absolute_error(y_val, y_val_pred))
            val_metrics['R2'].append(r2_score(y_val, y_val_pred))

        max_r2 = max(val_metrics['R2'])
        # argmax vẫn đúng với Ridge vì val R2 có dạng lồi
        best_idx = next(i for i, r2 in enumerate(val_metrics['R2'])
                if r2 >= max_r2 - 1e-6)
        best_alpha = alphas[best_idx]

        # Print giải thích lý do chọn alpha
        print(f"\n{label}:")
        print(f"{'Alpha':<12} {'R2_train':>10} {'R2_val':>10} {'ΔR²(train-val)':>16}  Nhận xét")
        print("-" * 65)
        for i, a in enumerate(alphas):
            dr2    = train_metrics['R2'][i] - val_metrics['R2'][i]
            marker = " ← CHỌN (R2_val cao nhất)" if a == best_alpha else ""
            # Nhận xét mức độ regularization
            if dr2 > 0.15:
                comment = "Overfit"
            elif dr2 < -0.05:
                comment = "Underfit"
            else:
                comment = "Cân bằng"
            print(f"{a:<12} {train_metrics['R2'][i]:>10.4f} {val_metrics['R2'][i]:>10.4f} "
                  f"{dr2:>+16.4f}  {comment}{marker}")
        print(f"\n→ Chọn Alpha={best_alpha} vì R2_val={val_metrics['R2'][best_idx]:.4f} "
              f"cao nhất, ΔR²={train_metrics['R2'][best_idx]-val_metrics['R2'][best_idx]:+.4f} "
              f"(cân bằng bias-variance tốt nhất)")

        best_ridge = CustomRidgeRegression(alpha=best_alpha).fit(X_train, y_train)
        best_models_data.append({
            'label':      label,
            'best_alpha': best_alpha,
            'y_val':      y_val,
            'y_pred':     best_ridge.predict(X_val),
            'RMSE':       val_metrics['RMSE'][best_idx],
            'MAE':        val_metrics['MAE'][best_idx],
            'R2':         val_metrics['R2'][best_idx],
            'RMSE_train': train_metrics['RMSE'][best_idx],
            'MAE_train':  train_metrics['MAE'][best_idx],
            'R2_train':   train_metrics['R2'][best_idx]
        })

        for col_idx, metric in enumerate(metrics_names):
            ax = axes[row_idx, col_idx]

            ax.plot(alphas, train_metrics[metric],
                    marker='o', label='Train', color='#AEC6CF', linewidth=2)
            ax.plot(alphas, val_metrics[metric],
                    marker='s', label='Validation', color='#FFB347', linewidth=2)
            ax.axvline(best_alpha, color='gray', linestyle='--', alpha=0.5,
                       label=f'Alpha tối ưu ({best_alpha})')

            all_vals = train_metrics[metric] + val_metrics[metric]
            ax.set_ylim(min(all_vals) * 0.95, max(all_vals) * 1.05)
            ax.set_xscale('log')

            if row_idx == 0:
                ax.set_title(f'{metric}', fontsize=12)
            if col_idx == 0:
                ax.set_ylabel(label, fontweight='bold')

            ax.set_xlabel('Alpha (Log scale)')
            ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    return best_models_data

In [ ]:

metrics_linear, predictions_data = evaluate_splits(
    X, y,
    test_sizes=test_sizes,
    labels=labels,
    model_class=CustomLinearRegression,
    model_kwargs={}
)

plot_metrics_bar(metrics_linear, "So sánh RMSE, MAE và R bằng phươn pháp Linear Regression")



In [ ]:
best_linear_data = plot_ridge_regularization(X, y, test_sizes, labels)



In [ ]:
analyze_overfitting(best_linear_data, model_name="Linear")



In [ ]:
plot_actual_vs_predicted(
    best_linear_data,
    main_title="LinearRegression - So sánh Thực tế vs Dự đoán"
)


In [ ]:
plot_residuals(
    best_linear_data,
    main_title="LinearRegression - Phân tích phần dư theo từng tỉ lệ chia dữ liệu"
)

### 1.3 Multi Layers Perceptron

In [ ]:
def plot_mlp_regularization(X, y, test_sizes, labels,
                             alphas=[0.01, 0.1, 1, 10, 100],
                             hidden_size=64,
                             learning_rate=0.001,
                             epochs=200,
                             main_title="Tác động của Alpha (L2) đến hiệu suất MLP"):

    metrics_names = ['RMSE', 'MAE', 'R2']
    total = len(test_sizes) * len(alphas)
    count = 0

    fig, axes = plt.subplots(len(test_sizes), 3, figsize=(17, 5 * len(test_sizes)))
    fig.suptitle(main_title, fontsize=16, fontweight='bold')
    axes = np.array(axes).reshape(len(test_sizes), 3)

    best_models_data = []

    for row_idx, (test_size, label) in enumerate(zip(test_sizes, labels)):
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=test_size, random_state=42
        )

        train_metrics = {m: [] for m in metrics_names}
        val_metrics   = {m: [] for m in metrics_names}

        for a in alphas:
            count += 1
            print(f"  [{count}/{total}] {label} | Alpha={a} ...", end=' ')

            mlp = CustomMLPRegressor(
                hidden_size=hidden_size,
                learning_rate=learning_rate,
                epochs=epochs,
                lambda_reg=a,
                early_stopping=True
            )
            mlp.fit(X_train, y_train, X_val, y_val)
            print(f"dừng tại epoch {len(mlp.loss_history)}")

            y_train_pred = mlp.predict(X_train)
            y_val_pred   = mlp.predict(X_val)

            train_metrics['RMSE'].append(np.sqrt(mean_squared_error(y_train, y_train_pred)))
            train_metrics['MAE'].append(mean_absolute_error(y_train, y_train_pred))
            train_metrics['R2'].append(r2_score(y_train, y_train_pred))

            val_metrics['RMSE'].append(np.sqrt(mean_squared_error(y_val, y_val_pred)))
            val_metrics['MAE'].append(mean_absolute_error(y_val, y_val_pred))
            val_metrics['R2'].append(r2_score(y_val, y_val_pred))

        # argmax đúng với MLP vì val R2 có dạng lồi (tăng rồi giảm theo alpha)
        best_idx   = np.argmax(val_metrics['R2'])
        best_alpha = alphas[best_idx]

        # Print giải thích lý do chọn alpha
        print(f"\n  {label}:")
        print(f"  {'Alpha':<12} {'R2_train':>10} {'R2_val':>10} {'ΔR²(train-val)':>16}  Nhận xét")
        print("  " + "-" * 65)
        for i, a in enumerate(alphas):
            dr2    = train_metrics['R2'][i] - val_metrics['R2'][i]
            marker = " ← CHỌN (R2_val cao nhất)" if a == best_alpha else ""
            if dr2 > 0.15:
                comment = "Overfit"
            elif dr2 < -0.05:
                comment = "Underfit"
            else:
                comment = "Cân bằng"
            print(f"  {a:<12} {train_metrics['R2'][i]:>10.4f} {val_metrics['R2'][i]:>10.4f} "
                  f"{dr2:>+16.4f}  {comment}{marker}")

        dr2_best = train_metrics['R2'][best_idx] - val_metrics['R2'][best_idx]
        print(f"\n  → Chọn Alpha={best_alpha} vì R2_val={val_metrics['R2'][best_idx]:.4f} "
              f"cao nhất, ΔR²={dr2_best:+.4f}\n")

        # Train lại model tốt nhất
        print(f"  Re-train best model (Alpha={best_alpha})...", end=' ')
        best_mlp = CustomMLPRegressor(
            hidden_size=hidden_size,
            learning_rate=learning_rate,
            epochs=epochs,
            lambda_reg=best_alpha,
            early_stopping=True
        ).fit(X_train, y_train, X_val, y_val)
        print(f"dừng tại epoch {len(best_mlp.loss_history)}\n")

        y_val_pred_best = best_mlp.predict(X_val)

        best_models_data.append({
            'label':      label,
            'best_alpha': best_alpha,
            'y_val':      y_val,
            'y_pred':     y_val_pred_best,
            'RMSE':       val_metrics['RMSE'][best_idx],
            'MAE':        val_metrics['MAE'][best_idx],
            'R2':         val_metrics['R2'][best_idx],
            'RMSE_train': train_metrics['RMSE'][best_idx],
            'MAE_train':  train_metrics['MAE'][best_idx],
            'R2_train':   train_metrics['R2'][best_idx]
        })

        for col_idx, metric in enumerate(metrics_names):
            ax = axes[row_idx, col_idx]

            ax.plot(alphas, train_metrics[metric],
                    marker='o', label='Train', color='#AEC6CF', linewidth=2)
            ax.plot(alphas, val_metrics[metric],
                    marker='s', label='Validation', color='#FFB347', linewidth=2)
            ax.axvline(best_alpha, color='gray', linestyle='--', alpha=0.5,
                       label=f'Alpha tối ưu ({best_alpha})')

            all_vals = train_metrics[metric] + val_metrics[metric]
            ax.set_ylim(min(all_vals) * 0.95, max(all_vals) * 1.05)
            ax.set_xscale('log')

            if row_idx == 0:
                ax.set_title(f'{metric}', fontsize=12)
            if col_idx == 0:
                ax.set_ylabel(label, fontweight='bold')

            ax.set_xlabel('Alpha (Log scale)')
            ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    return best_models_data

In [ ]:

metrics_mlp, predictions_data = evaluate_splits(
    X, y,
    test_sizes=test_sizes,
    labels=labels,
    model_class=CustomMLPRegressor,
    model_kwargs={
        "hidden_size": 64,
        "learning_rate": 0.01,
        "epochs": 500,
        "lambda_reg": 0.01,
        "batch_size": 32
    }
)

plot_metrics_bar(metrics_mlp, "So sánh RMSE, MAE và R bằng phươn pháp Multi Layers Perceptron")




In [ ]:
best_mlp_data = plot_mlp_regularization(X, y, test_sizes, labels)



In [ ]:
analyze_overfitting(best_mlp_data,   model_name="MLP")



In [ ]:
plot_actual_vs_predicted(
    best_mlp_data,
    main_title="Multi Layers Perceptron - So sánh Thực tế vs Dự đoán"
)


In [ ]:
plot_residuals(
    best_mlp_data,
    main_title="Multi Layers Perceptron - Phân tích phần dư theo từng tỉ lệ chia dữ liệu"
)

### 1.4 Đánh giá

In [ ]:
def plot_best_models_comparison(best_knn_data, best_ridge_data, best_mlp_data,
                                 pca_palette= sns.color_palette("pastel"),
                                 main_title='Đánh giá toàn diện: So sánh các mô hình tốt nhất theo từng tỉ lệ chia'):

    if pca_palette is None:
        pca_palette = {'KNN': '#AEC6CF', 'Ridge': '#FFB347', 'MLP': '#B5EAD7'}

    # Gộp best_models_data của 3 model thành 1 DataFrame
    rows = []
    for model_name, data in [('KNN', best_knn_data), ('Ridge', best_ridge_data), ('MLP', best_mlp_data)]:
        for m in data:
            rows.append({
                'Tỉ lệ chia':  m['label'],
                'Mô hình':     model_name,
                'RMSE':        m['RMSE'],
                'MAE':         m['MAE'],
                'R2':          m['R2']
            })

    df = pd.DataFrame(rows)

    # Melt để vẽ nhóm theo chỉ số
    df_melted = df.melt(
        id_vars=['Tỉ lệ chia', 'Mô hình'],
        value_vars=['RMSE', 'MAE', 'R2'],
        var_name='Chỉ số',
        value_name='Giá trị'
    )

    ratios = df['Tỉ lệ chia'].unique()
    fig, axes = plt.subplots(1, len(ratios), figsize=(6 * len(ratios), 6))
    fig.suptitle(main_title, fontsize=16, fontweight='bold', y=1.02)

    if len(ratios) == 1:
        axes = [axes]

    def add_bar_labels(ax):
        for container in ax.containers:
            ax.bar_label(container, fmt='%.3f', padding=3, color='#444444', fontsize=9)

    for i, ratio in enumerate(ratios):
        subset = df_melted[df_melted['Tỉ lệ chia'] == ratio]

        sns.barplot(x='Chỉ số', y='Giá trị', hue='Mô hình',
                    data=subset, ax=axes[i], palette=pca_palette)

        axes[i].set_title(f'Tỉ lệ {ratio}', fontsize=14, fontweight='bold', color='#333333')
        axes[i].set_xlabel('')
        axes[i].set_ylabel('Giá trị' if i == 0 else '')

        add_bar_labels(axes[i])

        if i != len(ratios) // 2:
            axes[i].get_legend().remove()
        else:
            sns.move_legend(axes[i], "lower center",
                            bbox_to_anchor=(0.5, -0.22),
                            ncol=3, title=None, frameon=False, fontsize=11)

    plt.tight_layout()
    plt.show()

    return df

In [ ]:
# Cell này đã được lược bỏ do trùng lặp hàm plot_best_models_comparison
print('Đã tối ưu hóa: Loại bỏ định nghĩa hàm trùng lặp.')

In [ ]:
df_comparison = plot_best_models_comparison(best_knn_data, best_linear_data, best_mlp_data)

In [ ]:
# Cell này đã được lược bỏ do trùng lặp hàm plot_best_models_comparison
pass

## 2\. Dữ liệu giảm chiều

In [ ]:
import pandas as pd
import numpy as np

print("=" * 75)
print("BƯỚC 0: LẤY LẠI PCAMIX MODEL ĐÃ GIẢM CHIỀU Ở TRÊN")
print("=" * 75)

if 'mixed_features' in globals():
    # lấy số chiều thực tế
    n_pcs = mixed_features.shape[1]
    cols = [f'PC_mix_{i+1}' for i in range(n_pcs)]

    # Tạo thẳng X_pca từ mảng mixed_features và đồng bộ Index
    X_pca = pd.DataFrame(mixed_features, columns=cols, index=df_scaled.index if 'df_scaled' in globals() else None)

    print(f"█ Thành công: Khôi phục Dữ liệu PCAmix đồng thời {n_pcs} chiều.")
    print(f"█ Kích thước tập đặc trưng mới: {X_pca.shape}")
    display(X_pca.head())
else:
    print("[LỖI] Không tìm thấy mảng dữ liệu 'mixed_features'.")
    print("      Vui lòng chạy lại Ô số 2 (Ô huấn luyện PCAmix_Manual) để sinh dữ liệu trước nhé!")

### 2.1 K-NN

In [ ]:

# ==========================================
# PHẦN 1: IN BẢNG KẾT QUẢ CƠ BẢN (K=5)
# ==========================================
metrics_knn, predictions_data = evaluate_splits(
    X_pca, y,
    test_sizes=test_sizes,
    labels=labels,
    model_class=CustomKNNRegressor,
    model_kwargs={"n_neighbors": 5}
)

plot_metrics_bar(metrics_knn, "So sánh RMSE, MAE và R bằng phươn pháp KNN")



In [ ]:
best_knn_pca = plot_knn_regularization(X_pca, y, test_sizes, labels)


In [ ]:
analyze_overfitting(best_knn_pca,   model_name="KNN")



In [ ]:
plot_actual_vs_predicted(
    best_knn_pca,
    main_title="KNN - So sánh Thực tế vs Dự đoán"
)


In [ ]:
plot_residuals(
    best_knn_pca,
    main_title="KNN - Phân tích phần dư theo từng tỉ lệ chia dữ liệu"
)

### 2.2 Linear Regression

In [ ]:

metrics_linear, predictions_data = evaluate_splits(
    X_pca, y,
    test_sizes=test_sizes,
    labels=labels,
    model_class=CustomLinearRegression,
    model_kwargs={}
)

plot_metrics_bar(metrics_linear, "So sánh RMSE, MAE và R bằng phươn pháp Linear Regression")



In [ ]:
best_linear_pca = plot_ridge_regularization(X_pca, y, test_sizes, labels)


In [ ]:
analyze_overfitting(best_linear_pca, model_name="Linear")



In [ ]:
plot_actual_vs_predicted(
    best_linear_pca,
    main_title="LinearRegression - So sánh Thực tế vs Dự đoán"
)


In [ ]:
plot_residuals(
    best_linear_pca,
    main_title="LinearRegression - Phân tích phần dư theo từng tỉ lệ chia dữ liệu"
)

### 2.3 Multi Layers Perceptron  

In [ ]:
metrics_mlp, predictions_data = evaluate_splits(
    X_pca, y,
    test_sizes=test_sizes,
    labels=labels,
    model_class=CustomMLPRegressor,
    model_kwargs={
        "hidden_size": 64,
        "learning_rate": 0.01,
        "epochs": 500,
        "lambda_reg": 0.01,
        "batch_size": 32
    }
)



In [ ]:
plot_metrics_bar(metrics_mlp, "So sánh RMSE, MAE và R bằng phươn pháp Multi Layers Perceptron")



In [ ]:
best_mlp_pca = plot_mlp_regularization(X_pca, y, test_sizes, labels)





In [ ]:
analyze_overfitting(best_mlp_pca,   model_name="MLP")

In [ ]:
plot_actual_vs_predicted(
    best_mlp_pca,
    main_title="Multi Layers Perceptron - So sánh Thực tế vs Dự đoán"
)


In [ ]:
plot_residuals(
    best_mlp_pca,
    main_title="Multi Layers Perceptron - Phân tích phần dư theo từng tỉ lệ chia dữ liệu"
)

### 2.4 Đánh giá

In [ ]:
df_comparison = plot_best_models_comparison(best_knn_pca, best_linear_pca, best_mlp_pca)

## 3\. Kết quả

In [ ]:
def print_regression_summary(best_knn_data, best_linear_data, best_mlp_data,
                              best_knn_pca, best_linear_pca, best_mlp_pca):

    SEP  = "=" * 80
    SEP2 = "-" * 80

    def build_results(knn, linear, mlp):
        out = {}
        for model_name, data in [('KNN', knn), ('Linear', linear), ('MLP', mlp)]:
            out[model_name] = {}
            for m in data:
                ratio = m['label'].split()[1]
                key   = ratio.replace('%','')
                out[model_name][key]          = (m['RMSE'], m['MAE'], m['R2'])
                out[model_name][f'of_{key}']  = (m['R2_train'] - m['R2'],
                                                  m['RMSE'] - m['RMSE_train'])
        return out

    results = {
        'Gốc': build_results(best_knn_data,  best_linear_data,  best_mlp_data),
        'PCA': build_results(best_knn_pca,   best_linear_pca,   best_mlp_pca),
    }

    # -------------------------------------------------------
    # BẢNG 1: HIỆU SUẤT
    # -------------------------------------------------------
    print(SEP)
    print("  BẢNG 1: HIỆU SUẤT CÁC MÔ HÌNH (Validation Set)")
    print(SEP)
    print(f"  {'Dữ liệu':<8} {'Mô hình':<10} {'Tỉ lệ':<12} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
    print(SEP2)
    for data_type, models in results.items():
        for model, metrics in models.items():
            for ratio in ['80','70','60']:
                rmse, mae, r2 = metrics[ratio]
                print(f"  {data_type:<8} {model:<10} {'Train '+ratio+'%':<12}"
                      f" {rmse:>8.4f} {mae:>8.4f} {r2:>8.4f}")
        print(SEP2)

    # -------------------------------------------------------
    # BẢNG 2: OVERFITTING
    # -------------------------------------------------------
    print(f"\n{SEP}")
    print("  BẢNG 2: ĐÁNH GIÁ OVERFITTING (ΔR² = R²_train − R²_val)")
    print(SEP)
    print(f"  {'Dữ liệu':<8} {'Mô hình':<10} {'Tỉ lệ':<12} {'ΔR²':>8} {'ΔRMSE':>8}  Trạng thái")
    print(SEP2)
    for data_type, models in results.items():
        for model, metrics in models.items():
            for ratio in ['80','70','60']:
                dr2, drmse = metrics[f'of_{ratio}']
                if abs(dr2) > 0.2 or abs(drmse) > 0.2:
                    status = " Overfit nặng"
                elif abs(dr2) > 0.1 or abs(drmse) > 0.1:
                    status = "  Overfit nhẹ"
                else:
                    status = " Ổn định"
                print(f"  {data_type:<8} {model:<10} {'Train '+ratio+'%':<12}"
                      f" {dr2:>+8.4f} {drmse:>+8.4f}  {status}")
        print(SEP2)

    # -------------------------------------------------------
    # Tính toán để nhận xét động
    # -------------------------------------------------------
    r2  = lambda dt, m, r: results[dt][m][r][2]
    dr2 = lambda dt, m, r: results[dt][m][f'of_{r}'][0]

    r2_best = {f'{m}/{dt}': results[dt][m]['80'][2]
               for dt in ['Gốc','PCA'] for m in ['KNN','Linear','MLP']}
    best_model = max(r2_best, key=r2_best.get)

    # Mức độ nhạy cảm tỉ lệ chia (chênh R² giữa 80% và 60%)
    sens = {f'{m}/Gốc': abs(r2('Gốc',m,'80') - r2('Gốc',m,'60'))
            for m in ['KNN','Linear','MLP']}
    most_sensitive   = max(sens, key=sens.get)
    most_stable_ratio = min(sens, key=sens.get)

    # PCA stability so sánh ΔR² max
    def max_dr2(dt, m):
        return max(abs(dr2(dt, m, r)) for r in ['80','70','60'])

    print(f"""
{SEP}
  NHẬN XÉT CHI TIẾT
{SEP}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. HIỆU SUẤT CÁC MÔ HÌNH (Tỉ lệ 80/20, dữ liệu gốc)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   MLP (R²={r2('Gốc','MLP','80'):.4f}) > Linear (R²={r2('Gốc','Linear','80'):.4f}) > KNN (R²={r2('Gốc','KNN','80'):.4f})

   - KNN (R²={r2('Gốc','KNN','80'):.4f}): Đạt hiệu suất thấp nhất trong 3 mô hình. Với
     bộ dữ liệu SUPPORT2 có nhiều chiều đặc trưng, khoảng cách Euclidean kém
     phân biệt hơn trong không gian chiều cao ("curse of dimensionality"), khiến
     các láng giềng gần không thực sự mang ý nghĩa tương đồng lâm sàng.

   - Linear Regression (R²={r2('Gốc','Linear','80'):.4f}): Cải thiện đáng kể so với KNN
     (+{r2('Gốc','Linear','80')-r2('Gốc','KNN','80'):.4f} R²). Kết quả này cho thấy tồn tại mối quan hệ
     tuyến tính đáng kể giữa các chỉ số lâm sàng (huyết áp, nhịp tim, pH máu,
     tuổi tác...) và chi phí điều trị. Tuy nhiên giới hạn ở giả định tuyến tính
     khiến mô hình không nắm bắt được các tương tác phức tạp giữa các biến.

   - MLP (R²={r2('Gốc','MLP','80'):.4f}): Vượt trội rõ rệt so với 2 mô hình còn lại
     (+{r2('Gốc','MLP','80')-r2('Gốc','Linear','80'):.4f} R² so với Linear). Lớp ẩn 64 neurons với
     hàm kích hoạt ReLU cho phép MLP học được các quan hệ phi tuyến và tương tác
     giữa các chỉ số (ví dụ: pH máu thấp kết hợp nhịp tim cao → chi phí đột
     biến), điều mà Linear Regression không thể mô hình hóa được.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2. ẢNH HƯỞNG CỦA TỈ LỆ CHIA TRAIN/VALIDATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   - Cả 3 mô hình đều đạt hiệu suất tốt nhất ở tỉ lệ 80/20 — tập train lớn
     hơn cung cấp nhiều mẫu học hơn, đặc biệt quan trọng với MLP và KNN.

   - Mức độ nhạy cảm với tỉ lệ chia (chênh R² giữa 80% và 60%):
     - KNN:    ΔR²={sens['KNN/Gốc']:.4f} → Nhạy cảm nhất, cần nhiều dữ liệu train
     - MLP:    ΔR²={sens['MLP/Gốc']:.4f} → Nhạy cảm vừa
     - Linear: ΔR²={sens['Linear/Gốc']:.4f} → Ổn định nhất, ít phụ thuộc kích thước train

   - Linear Regression ổn định nhất vì normal equation tìm nghiệm tối ưu toàn
     cục — không bị ảnh hưởng bởi random initialization hay learning rate như MLP.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
3. ĐÁNH GIÁ OVERFITTING VÀ REGULARIZATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   - KNN (ΔR² max = {max_dr2('Gốc','KNN'):+.4f}): Có xu hướng overfit tự nhiên nhẹ do
     mô hình "ghi nhớ" các điểm lân cận trong tập train. Kiểm soát bằng cách
     chọn K tối ưu qua validation (K lớn hơn → smooth hơn, giảm overfit).
     ΔR² nằm trong ngưỡng ổn định (<0.1) sau regularization.

   - Linear/Ridge (ΔR² âm = {dr2('Gốc','Linear','80'):+.4f} đến {dr2('Gốc','Linear','60'):+.4f}):
     ΔR² âm nghĩa là R² validation nhỉnh hơn train — hoàn toàn không overfit.
     Đây là hiệu ứng của Ridge (L2): Alpha phạt các hệ số lớn, "kìm" mô hình
     train lại, trong khi val set không bị ràng buộc này. Alpha tối ưu được
     chọn qua dải [0.01 → 100000] trên validation.

   - MLP (ΔR² max = {max_dr2('Gốc','MLP'):+.4f}): ΔR² dương nhưng rất nhỏ, kiểm soát
     hiệu quả bằng lambda_reg=0.01 (L2 penalty) kết hợp mini-batch SGD
     (batch_size=32) — mini-batch vốn có tác dụng regularize nhẹ do gradient
     noise giữa các batch. Xu hướng ΔR² tăng khi train nhỏ hơn là bình thường.

   → Kết luận: KHÔNG có mô hình nào bị overfit đáng kể sau regularization.
     Tất cả ΔR² đều nằm trong ngưỡng ổn định (< 0.1).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
4. SO SÁNH DỮ LIỆU GỐC vs DỮ LIỆU GIẢM CHIỀU (PCA)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   Mô hình  │ R² Gốc  │ R² PCA  │  Δ R²   │ ΔR²_of Gốc │ ΔR²_of PCA
   ─────────┼─────────┼─────────┼─────────┼────────────┼───────────
   KNN      │ {r2('Gốc','KNN','80'):.4f}  │ {r2('PCA','KNN','80'):.4f}  │ {r2('Gốc','KNN','80')-r2('PCA','KNN','80'):+.4f}  │   {max_dr2('Gốc','KNN'):+.4f}    │  {max_dr2('PCA','KNN'):+.4f}
   Linear   │ {r2('Gốc','Linear','80'):.4f}  │ {r2('PCA','Linear','80'):.4f}  │ {r2('Gốc','Linear','80')-r2('PCA','Linear','80'):+.4f}  │   {max_dr2('Gốc','Linear'):+.4f}    │  {max_dr2('PCA','Linear'):+.4f}
   MLP      │ {r2('Gốc','MLP','80'):.4f}  │ {r2('PCA','MLP','80'):.4f}  │ {r2('Gốc','MLP','80')-r2('PCA','MLP','80'):+.4f}  │   {max_dr2('Gốc','MLP'):+.4f}    │  {max_dr2('PCA','MLP'):+.4f}

   - Dữ liệu gốc vượt trội ở tất cả mô hình. PCA nén dữ liệu xuống 20 chiều
     (giữ 84.1% phương sai), mất ~15.9% thông tin làm giảm R² đáng kể.

   - Linear Regression chịu ảnh hưởng nhiều nhất từ PCA
     (ΔR²={r2('Gốc','Linear','80')-r2('PCA','Linear','80'):+.4f}): hồi quy tuyến tính phụ thuộc trực tiếp
     vào các features gốc — PCA biến đổi thành các thành phần chính trực giao
     làm mất đi ý nghĩa tuyến tính ban đầu của từng biến.

   - KNN/PCA: ΔR²_of tăng từ {max_dr2('Gốc','KNN'):.4f} lên {max_dr2('PCA','KNN'):.4f} — PCA làm KNN
     kém ổn định hơn vì thay đổi cấu trúc không gian khoảng cách, khiến các
     láng giềng gần trên PCA không phản ánh đúng sự tương đồng thực tế.

   - MLP/PCA: ΔR²_of giảm từ {max_dr2('Gốc','MLP'):.4f} xuống {max_dr2('PCA','MLP'):.4f} — PCA có lợi
     cho MLP về mặt ổn định, dù đánh đổi một phần accuracy ({r2('Gốc','MLP','80')-r2('PCA','MLP','80'):+.4f} R²).
     Không gian PCA ít nhiễu hơn giúp MLP không bị overfit vào các features
     không quan trọng.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
5. ĐÁNH GIÁ PHẦN DƯ — MÔ HÌNH CÓ PHÙ HỢP KHÔNG?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   - KNN: Phần dư phân tán không đồng đều — tập trung ở vùng chi phí thấp,
     trải rộng ở vùng chi phí cao (heteroscedasticity). KNN không nội suy
     tốt ở các ca bệnh nặng có chi phí đột biến, phần dư thường lệch dương
     ở vùng giá trị lớn. Mô hình CHƯA PHÙ HỢP hoàn toàn.

   - Linear Regression: Phần dư phân tán gần đối xứng quanh đường y=0, không
     có pattern cong rõ ràng — dấu hiệu tốt cho giả định tuyến tính. Tuy nhiên
     vẫn còn outlier ở chi phí rất cao, gợi ý mối quan hệ không hoàn toàn
     tuyến tính. Mô hình PHÙ HỢP Ở MỨC VỪA.

   - MLP: Phần dư ngẫu nhiên nhất trong 3 mô hình, xu hướng (trend line) gần
     nằm ngang — cho thấy không có pattern hệ thống còn sót lại. Std residual
     thấp nhất, mean residual gần 0 nhất. Mô hình PHÙ HỢP TỐT NHẤT.

   → Cả 3 mô hình còn tồn tại outlier ở vùng chi phí điều trị cực cao — đây
     là đặc trưng của dữ liệu y tế (một số ca bệnh đặc biệt phức tạp khó dự
     đoán). Hướng cải thiện: log-transform biến 'charges' để giảm right-skew,
     hoặc xây dựng mô hình riêng cho nhóm bệnh nhân chi phí cao.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
6. KẾT LUẬN VÀ KHUYẾN NGHỊ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    Mô hình tốt nhất tổng thể: {best_model}
     R²={r2_best[best_model]:.4f}, RMSE={results[best_model.split('/')[1]][best_model.split('/')[0]]['80'][0]:.4f}

   - Ưu tiên độ chính xác    → MLP trên dữ liệu gốc, tỉ lệ 80/20
   - Ưu tiên tính giải thích → Linear Regression (hệ số hồi quy dễ trình bày)
   - Ưu tiên tài nguyên thấp → KNN (không cần train, nhưng kém chính xác)

   Hướng cải thiện tiếp theo:
   (1) Log-transform biến mục tiêu 'charges' để giảm right-skew
   (2) Tăng hidden layers hoặc neurons cho MLP
   (3) Feature selection kỹ hơn để giảm noise trước khi đưa vào KNN
{SEP}
""")

# Gọi hàm
print_regression_summary(
    best_knn_data,   best_linear_data,   best_mlp_data,
    best_knn_pca,    best_linear_pca,    best_mlp_pca
)

#V) Phân loại

## 1\. Chuyển bài toán về dạng phân loại

In [ ]:
# --- MỤC 1: CHUYỂN ĐỔI NHÃN PHÂN LOẠI ---
bin_low_log = np.log1p(11000)
bin_high_log = np.log1p(50000)
bins = [-np.inf, bin_low_log, bin_high_log, np.inf]
labels_list = [0, 1, 2]

y_classified = pd.cut(df_scaled['charges'], bins=bins, labels=labels_list).astype(int).values
X_raw_data = df_scaled.drop(columns=['charges'])
C_classes = 3

# Khởi tạo từ điển lưu kết quả tổng hợp toàn bài (Dùng cho mục 5 sau này)
global_results = {
    '80/20': {'Softmax_Raw': None, 'NB_Raw': None},
    '70/30': {'Softmax_Raw': None, 'NB_Raw': None},
    '60/40': {'Softmax_Raw': None, 'NB_Raw': None}
}                  # Số lượng lớp nhãn (Thấp, Trung bình, Cao)

# --- CẤU HÌNH THỰC NGHIỆM ĐA TỶ LỆ ---
configs = [
    (0.2, 'Train 80% / Test 20%', '80/20'),
    (0.3, 'Train 70% / Test 30%', '70/30'),
    (0.4, 'Train 60% / Test 40%', '60/40')
]

N_total = len(y_classified)

# Thống kê phân phối tỷ lệ mẫu thực tế
unique_elements, counts_elements = np.unique(y_classified, return_counts=True)
label_names = [f"Thấp (< 11,000 $)",
               f"Trung bình (11,000 $ - 50,000 $)",
               f"Cao (> 50,000 $)"]

print("\n" + "="*80)
print("MỤC 1: PHÂN PHỐI TỶ LỆ MẪU THỰC TẾ TRÊN TỪNG KHOẢNG NHÃN")
print("="*80)
for lbl_id, name, count in zip(unique_elements, label_names, counts_elements):
    pct = (count / N_total) * 100
    print(f"  Class {lbl_id} - Nhóm {name:<35}: {count:<5} mẫu  →  {pct:.2f}%")

Hàm in các độ đo cho mô hình phân loại

In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix, accuracy_score

def display_detailed_metrics(y_true, y_pred, label, len_train, len_test, title_label=""):
    """
    Hàm hiển thị All-in-One: In báo cáo chung (Accuracy) và Báo cáo chi tiết (Precision, Recall, Confusion Matrix)
    """
    acc = accuracy_score(y_true, y_pred)

    # =====================================================================
    # 1. KHỐI BÁO CÁO CHUNG TÓM TẮT
    # =====================================================================
    print("\n" + "=" * 85)
    print(f" KẾT QUẢ: {title_label}")
    print("-" * 85)
    print(f"{'Tỷ lệ chia dữ liệu':<25} | {'Mẫu Train':<10} | {'Mẫu Test':<10} | {'Độ chính xác (Accuracy)':<25}")
    print("-" * 85)
    print(f"{label:<25} | {len_train:>10,d} | {len_test:>10,d} | {acc:>25.4%}")
    print("=" * 85)

    # =====================================================================
    # 2. KHỐI BÁO CÁO CHI TIẾT
    # =====================================================================
    label_names = ["0: Thấp (< 11k$)", "1: Trung bình (11k$-50k$)", "2: Cao (> 50k$)"]

    precisions = precision_score(y_true, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    recalls = recall_score(y_true, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

    print(f"{'Nhóm nhãn Chi phí Y tế':<30} | {'Precision (Dự đoán chuẩn)':<25} | {'Recall (Tìm bắt đúng)':<25}")
    print("-" * 85)
    for idx, name in enumerate(label_names):
        print(f"  {name:<28} | {precisions[idx]:>23.2%} | {recalls[idx]:>23.2%}")
    print("-" * 85)

    print(" Ma trận nhầm lẫn (Confusion Matrix):")
    print(f"    {'Thực tế \\ Dự đoán':<22} | {'Thấp (0)':<10} | {'T.Bình (1)':<11} | {'Cao (2)':<10}")
    print("    " + "-" * 60)
    short_names = ["Thấp (0)", "Trung bình (1)", "Cao (2)"]
    for idx, name in enumerate(short_names):
        print(f"    {name:<22} | {cm[idx, 0]:>9,d} | {cm[idx, 1]:>10,d} | {cm[idx, 2]:>9,d}")
    print("=" * 85 + "\n")

##2\. Dữ liệu gốc

###2.1 Softmax Regression

Định nghĩa các hàm cần thiết

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# =====================================================================
# I) CÁC HÀM TOÁN HỌC THỦ CÔNG CÓ HIỆU CHỈNH L2 (CỦA BẠN)
# =====================================================================
def convert_labels(y, C):
    """Biến đổi nhãn 1D thành ma trận nhãn One-Hot (C x N)"""
    Y = sparse.coo_matrix((np.ones_like(y),
                           (y, np.arange(len(y)))), shape=(C, len(y))).toarray()
    return Y

def softmax_stable(Z):
    """Tính toán Softmax ổn định tránh tràn số"""
    e_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    A = e_Z / e_Z.sum(axis=0)
    return A

def softmax_regression_manual_l2(X, y, C, eta=0.01, lam=0.001, tol=1e-4, max_count=5000):
    """
    Thuật toán Softmax Regression thủ công (SGD) có hiệu chỉnh L2
    X: Ma trận đặc trưng kích thước (Số_đặc_trưng d, Số_mẫu N)
    y: Mảng nhãn 1D phân loại
    """
    d, N = X.shape
    W_current = np.random.randn(d, C) * 0.01
    Y = convert_labels(y, C)

    count = 0
    check_w_after = 20

    while count < max_count:
        mix_id = np.random.permutation(N)
        for i in mix_id:
            xi = X[:, i].reshape(d, 1)
            yi = Y[:, i].reshape(C, 1)
            ai = softmax_stable(np.dot(W_current.T, xi))

            W_old = W_current.copy()
            W_current = W_current + eta * (xi.dot((yi - ai).T) - lam * W_current)
            count += 1

            if count % check_w_after == 0:
                if np.linalg.norm(W_current - W_old) < tol:
                    return W_current

            if count >= max_count:
                break
    return W_current

def pred_manual(W, X):
    """Dự đoán nhãn lớp"""
    A = softmax_stable(W.T.dot(X))
    return np.argmax(A, axis=0)

# Hàm tự chia Train/Test thuần thủ công dựa trên xáo trộn chỉ số
def train_test_split_manual(X, y, test_size=0.2, random_state=42):
    if random_state is not None:
        np.random.seed(random_state)
    indices = np.random.permutation(len(X))
    test_samples = int(len(X) * test_size)

    test_idx, train_idx = indices[:test_samples], indices[test_samples:]

    if isinstance(X, pd.DataFrame):
        return X.iloc[train_idx].values, X.iloc[test_idx].values, y[train_idx], y[test_idx]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]



# Hàm để tự hiệu chỉnh tham số
def tune_softmax_hyperparameters(X_train_full, y_train_full, C, lambda_grid, eta_grid, val_size=0.2):
    """
    Hàm Grid Search thủ công để tìm siêu tham số tối ưu (Tránh Data Leakage)
    """
    print(f"   [Tuning] Bắt đầu tìm kiếm trên lưới: {len(lambda_grid) * len(eta_grid)} tổ hợp...")

    # 1. Trích xuất tập Validation từ tập Train hiện tại
    X_sub_train, X_val, y_sub_train, y_val = train_test_split_manual(
        X_train_full, y_train_full, test_size=val_size, random_state=42
    )

    X_sub_train_T = X_sub_train.T
    X_val_T = X_val.T

    best_acc = 0.0
    best_lam = None
    best_eta = None

    # 2. Quét qua tất cả các tổ hợp của lam và eta
    for lam in lambda_grid:
        for eta in eta_grid:
            # Huấn luyện trên tập Sub-Train với số vòng lặp nhỏ hơn để tiết kiệm thời gian (VD: 2000)
            W_temp = softmax_regression_manual_l2(
                X_sub_train_T, y_sub_train, C=C, eta=eta, lam=lam, max_count=2000
            )

            # Đánh giá trên tập Validation
            y_pred_val = pred_manual(W_temp, X_val_T)
            acc_val = accuracy_score(y_val, y_pred_val)

            # Cập nhật nếu tìm thấy cấu hình tốt hơn
            if acc_val > best_acc:
                best_acc = acc_val
                best_lam = lam
                best_eta = eta

    print(f"   [Tuning] Thành công! Tốt nhất: lam={best_lam}, eta={best_eta} (Val Acc: {best_acc:.2%})")
    return best_lam, best_eta

# định nghĩa lưới tham số cho việc tuning
LAMBDA_GRID = [0.1, 0.05, 0.01, 0.001, 0.0001]
ETA_GRID = [0.1, 0.05, 0.01, 0.005, 0.001]

Huấn luyện

In [ ]:
from sklearn.metrics import accuracy_score

print("="*85)
print("MỤC 2.1: CHẠY THỬ NGHIỆM SOFTMAX REGRESSION TRÊN DỮ LIỆU GỐC")
print("="*85)


for test_size, label, key in configs:
    # Bước 1: Chia tập Train/Test gốc (Tuyệt đối không đụng vào X_test trong quá trình tune)
    X_train, X_test, y_train, y_test = train_test_split_manual(
        X_raw_data, y_classified, test_size=test_size, random_state=42
    )

    print(f"\n Đang xử lý: {label}...")

    # Bước 2: Gọi hàm Tuning để dò tìm tham số tốt nhất bằng tập Validation nội bộ
    best_lam, best_eta = tune_softmax_hyperparameters(
        X_train, y_train, C=C_classes, lambda_grid=LAMBDA_GRID, eta_grid=ETA_GRID
    )

    # Bước 3: Đưa ma trận về dạng cột (d, N) chuẩn bị cho Softmax
    X_train_T = X_train.T
    X_test_T = X_test.T

    # Bước 4: HUẤN LUYỆN LẠI (Retrain) trên toàn bộ X_train bằng bộ tham số xịn nhất vừa tìm được
    # Lúc này ta dùng max_count lớn (5000) để đảm bảo mô hình hội tụ sâu nhất
    W_fitted = softmax_regression_manual_l2(
        X_train_T, y_train, C=C_classes, eta=best_eta, lam=best_lam, max_count=5000
    )

    # Bước 5: Dự đoán trên tập Test bị giấu kín từ đầu
    y_pred = pred_manual(W_fitted, X_test_T)
    acc = accuracy_score(y_test, y_pred)

    global_results[key]['Softmax_Raw'] = acc

    # Bước 6: Gọi hàm in báo cáo All-in-One (Cập nhật tiêu đề để show tham số xịn)
    title = f"Dữ liệu gốc | $\\lambda$={best_lam} | $\\eta$={best_eta} | Tỷ lệ {label}"
    display_detailed_metrics(y_test, y_pred, label=label, len_train=len(y_train), len_test=len(y_test), title_label=title)

print("="*85)
print("✔ HOÀN TẤT CHẠY THỰC NGHIỆM TRÊN DỮ LIỆU GỐC!")

###2.2 Hybrid Naive Bayes

In [ ]:
print("="*85)
print("MỤC 2.2: CHẠY THỬ NGHIỆM HYBRID NAIVE BAYES TRÊN DỮ LIỆU GỐC")
print("="*85)
SEP = "-" * 85
print(f"{'Tỷ lệ chia dữ liệu':<25} | {'Mẫu Train':<10} | {'Mẫu Test':<10} | {'Độ chính xác (Accuracy)':<25}")
print(SEP)


for test_size, label, key in configs:
    X_train, X_test, y_train, y_test = train_test_split_manual(
        X_raw_data, y_classified, test_size=test_size, random_state=42
    )

    # -------------------------------------------------------------------------
    # [TẠM ĐỂ TRỐNG] - KHÔNG GIAN VIẾT CODE THUẬT TOÁN HYBRID NAIVE BAYES CỦA BẠN
    # -------------------------------------------------------------------------
    # Sau này khi bạn viết xong hàm huấn luyện Naive Bayes thủ công, bạn chỉ cần
    # gọi hàm dự đoán tại đây, ví dụ:
    # y_pred_nb = naive_bayes_predict(X_train, y_train, X_test)
    # acc_nb = accuracy_score(y_test, y_pred_nb)
    # global_results[key]['NB_Raw'] = acc_nb
    # -------------------------------------------------------------------------

    acc_placeholder = "[ĐỂ TRỐNG - SẼ PHÁT TRIỂN]"
    print(f"{label:<25} | {len(y_train):>10,d} | {len(y_test):>10,d} | {acc_placeholder:>25}")
print(SEP)

### 2.3 KNN

##3\. Dữ liệu giảm chiều bằng PCAmix

###3.1 Softmax Regression

In [ ]:
print("="*85)
print("MỤC 3.1: SOFTMAX REGRESSION TRÊN DỮ LIỆU GIẢM CHIỀU BẰNG PCAMIX")
print("="*85)

total_dims = X_mixed.shape[1]
n_components_pca = max(1, int(total_dims / 3))

print(f"✔ Tổng số chiều gốc của dữ liệu: {total_dims} chiều.")
print(f"✔ Quyết định giảm chiều (giữ lại 1/3): Tiến hành giảm về {n_components_pca} chiều.")
print("-" * 85)

SEP = "-" * 85

# Duyệt qua các cấu hình đa tỷ lệ chia
for test_size, label, key in configs:

    print(f"\n⏳ Đang xử lý PCAmix: {label}...")

    # Bước 1: Chia tập dữ liệu hỗn hợp thô X_mixed trước khi giảm chiều
    X_train_raw, X_test_raw, y_train, y_test = train_test_split_manual(
        X_mixed, y_classified, test_size=test_size, random_state=42
    )

    # Bước 2: Khởi tạo đối tượng PCAmix_Manual
    pcamix_pipeline = PCAmix_Manual(
        n_components=n_components_pca,
        discrete_features=discrete_indices,
        continuous_features=continuous_indices
    )

    # Bước 3: FIT-TRANSFORM trên tập Train
    X_train_pca = pcamix_pipeline.fit_transform(X_train_raw)

    # Bước 4: TRANSFORM trên tập Test sang không gian của Train
    X_test_pca = pcamix_pipeline.transform(X_test_raw)

    # Bước 5: AUTO-TUNING TÌM THAM SỐ TỐT NHẤT TRÊN DỮ LIỆU ĐÃ GIẢM CHIỀU
    best_lam, best_eta = tune_softmax_hyperparameters(
        X_train_pca, y_train, C=C_classes, lambda_grid=LAMBDA_GRID, eta_grid=ETA_GRID
    )

    # Bước 6: Chuyển vị ma trận (.T) đưa về dạng chuẩn (d, N) của Softmax thủ công
    X_train_T = X_train_pca.T
    X_test_T = X_test_pca.T

    # Bước 7: HUẤN LUYỆN LẠI trên toàn bộ X_train_pca với cấu hình xịn nhất (max_count=5000 cho hội tụ sâu)
    W_fitted_pca = softmax_regression_manual_l2(
        X_train_T, y_train, C=C_classes, eta=best_eta, lam=best_lam, max_count=5000
    )

    # Bước 8: Dự đoán trên tập Test
    y_pred_pca = pred_manual(W_fitted_pca, X_test_T)
    acc_pca = accuracy_score(y_test, y_pred_pca)

    # Lưu kết quả
    global_results[key]['Softmax_PCAmix'] = acc_pca

    # Bước 9: Gọi hàm in báo cáo với đầy đủ 6 tham số (Cập nhật cả title để xem được tham số)
    title = f"PCAmix ({n_components_pca}D) | lam={best_lam} | eta={best_eta} | Tỷ lệ {label}"
    display_detailed_metrics(
        y_true=y_test,
        y_pred=y_pred_pca,
        label=label,
        len_train=len(y_train),
        len_test=len(y_test),
        title_label=title
    )

print("="*85)
print("✔ HOÀN TẤT CHẠY THỰC NGHIỆM PCAMIX!")

### 3.2 Hybrid Naive Bayes

### 3.3 KNN

##4\. Dữ liệu giảm chiều bằng DISMIX

Định nghĩa các hàm

In [ ]:
import numpy as np
import pandas as pd

class LDA_manual:
    """
    Linear Discriminant Analysis (LDA) tự viết từ đầu.
    Tìm kiếm các trục tuyến tính tối ưu hóa khoảng cách giữa các lớp nhãn.
    """
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.scalings_ = None # Ma trận các vector riêng (trục chiếu)
        self.classes_ = None  # Danh sách các nhãn duy nhất

    def fit(self, X, y):
        # Chuyển đổi đầu vào sang dạng numpy array để tính toán ma trận nhanh
        X_arr = np.array(X, dtype=float)
        y_arr = np.array(y).reshape(-1)

        n_samples, n_features = X_arr.shape
        self.classes_ = np.unique(y_arr)
        n_classes = len(self.classes_)

        # 1. Tính toán vector trung bình tổng thể (overall mean)
        mean_overall = np.mean(X_arr, axis=0)

        # Khởi tạo ma trận Scatter trong nhóm (Within-class) và Giữa các nhóm (Between-class)
        S_W = np.zeros((n_features, n_features))
        S_B = np.zeros((n_features, n_features))

        # 2. Tính toán ma trận Scatter từng nhóm
        for c in self.classes_:
            X_c = X_arr[y_arr == c]
            mean_c = np.mean(X_c, axis=0)

            # Tính Scatter trong nhóm c và cộng dồn vào S_W
            # S_W = sum( (X_c - mean_c)^T * (X_c - mean_c) )
            deviation_W = X_c - mean_c
            S_W += np.dot(deviation_W.T, deviation_W)

            # Tính Scatter giữa nhóm c và trung bình tổng thể rồi cộng dồn vào S_B
            # S_B = sum( n_c * (mean_c - mean_overall)^T * (mean_c - mean_overall) )
            n_c = X_c.shape[0]
            mean_diff = (mean_c - mean_overall).reshape(-1, 1)
            S_B += n_c * np.dot(mean_diff, mean_diff.T)

        # 3. Giải bài toán Trị riêng (Eigen-problem) cho ma trận: (S_W^-1) * S_B
        # Thêm một lượng nhỏ Ridge (Regularization) vào đường chéo S_W để tránh ma trận suy biến (Singular Matrix)
        S_W += np.eye(n_features) * 1e-6

        S_W_inv = np.linalg.inv(S_W)
        matrix_to_decompose = np.dot(S_W_inv, S_B)

        # Tính trị riêng và vectơ riêng
        eigenvalues, eigenvectors = np.linalg.eig(matrix_to_decompose)

        # Trị riêng tính ra có thể bị dính phần ảo cực nhỏ do sai số máy tính, ta chỉ lấy phần thực
        eigenvalues = np.real(eigenvalues)
        eigenvectors = np.real(eigenvectors)

        # 4. Sắp xếp các vectơ riêng theo thứ tự trị riêng giảm dần
        idxs = np.argsort(eigenvalues)[::-1]
        eigenvectors = eigenvectors[:, idxs]

        # Chọn số lượng thành phần chính (n_components)
        # Chú ý: Số chiều tối đa của LDA bị giới hạn bởi: min(n_features, n_classes - 1)
        max_components = min(n_features, n_classes - 1)
        actual_components = min(self.n_components, max_components)

        # Lưu trữ ma trận trọng số (trục chiếu)
        self.scalings_ = eigenvectors[:, :actual_components]
        return self

    def transform(self, X):
        X_arr = np.array(X, dtype=float)
        # Nhân ma trận dữ liệu với ma trận trục chiếu để lấy tọa độ mới
        return np.dot(X_arr, self.scalings_)

    def fit_transform(self, X, y):
        self.fit(X, y)
        return self.transform(X)

class DISMIX_manual:
    """
    Mô hình DISMIX (Discriminant Analysis on Mixed Predictors)
    Kết hợp quy trình chuỗi hai giai đoạn: PCAmix_manual và LDA_manual.
    """
    def __init__(self, pcamix_instance, lda_instance):
        """
        Parameters:
        - pcamix_instance: Đối tượng đã khởi tạo của hàm PCAmix_manual của bạn (nên đặt n_components=25).
        - lda_instance: Đối tượng đã khởi tạo của hàm LDA_manual bên trên (nên đặt n_components=2).
        """
        self.pcamix_model = pcamix_instance
        self.lda_model = lda_instance
        self.is_fitted = False

    def fit(self, X, y):
        """
        Giai đoạn học (Có giám sát):
        X: DataFrame chứa cả cột Số và cột Chữ dữ liệu gốc.
        y: Series/Array chứa nhãn bài toán mục tiêu.
        """
        print("[DISMIX_manual] Giai đoạn 1: Đang chạy PCAmix để chuyển đổi hình học hỗn hợp...")
        # Sử dụng hàm fit_transform hoặc fit/transform của hàm PCAmix của bạn
        if hasattr(self.pcamix_model, 'fit_transform'):
            X_continuous_space = self.pcamix_model.fit_transform(X)
        else:
            self.pcamix_model.fit(X)
            X_continuous_space = self.pcamix_model.transform(X)

        print(f" -> Hoàn tất PCAmix. Không gian số thực thu được: {np.shape(X_continuous_space)}")

        print("[DISMIX_manual] Giai đoạn 2: Đang chạy LDA toán học để tối ưu hóa phân lớp nhãn...")
        # Ép kiểu dữ liệu nhãn thành mảng phẳng numpy để đưa vào LDA_manual
        y_arr = np.array(y).reshape(-1)
        self.lda_model.fit(X_continuous_space, y_arr)

        print(" -> Hoàn tất huấn luyện LDA. Hệ thống DISMIX sẵn sàng chuyển đổi dữ liệu.")
        self.is_fitted = True
        return self

    def transform(self, X):
        """
        Chiếu tọa độ tập dữ liệu mới (ví dụ tập Test) qua 2 lớp bộ lọc.
        """
        if not self.is_fitted:
            raise RuntimeError("Mô hình chưa được chạy fit dữ liệu. Hãy gọi .fit() trước.")

        # Lớp 1: Ép dữ liệu mới về dạng số thực chuẩn hóa của PCAmix
        X_continuous = self.pcamix_model.transform(X)

        # Lớp 2: Chiếu không gian số thực đó xuống các trục phân biệt tuyến tính tối ưu của LDA
        X_dismix_coordinates = self.lda_model.transform(X_continuous)

        # Đóng gói dữ liệu đầu ra thành một DataFrame đẹp mắt cho bạn dễ xử lý/vẽ đồ thị
        n_dim_output = np.shape(X_dismix_coordinates)[1]
        column_names = [f"DISMIX_LD{i+1}" for i in range(n_dim_output)]

        if isinstance(X, pd.DataFrame):
            return pd.DataFrame(X_dismix_coordinates, index=X.index, columns=column_names)
        else:
            return pd.DataFrame(X_dismix_coordinates, columns=column_names)

    def fit_transform(self, X, y):
        """
        Hàm gộp thực thi nhanh đồng thời cho dữ liệu huấn luyện.
        """
        self.fit(X, y)
        return self.transform(X)

###4.1 Softmax Regression

In [ ]:
import os
import contextlib
import sys
from sklearn.metrics import accuracy_score

print("="*85)
print("MỤC 4.1: SOFTMAX REGRESSION TRÊN DỮ LIỆU GIẢM CHIỀU BẰNG DISMIX (CÓ AUTO-TUNING)")
print("="*85)

# --- BƯỚC TÍNH TOÁN SỐ CHIỀU NGOÀI VÒNG LẶP ---
total_dims = X_mixed.shape[1]
# Giai đoạn 1: PCAmix giảm về 25 chiều (giữ hơn 80% thông tin)
n_components_pca = 25
# Giai đoạn 2: LDA tối ưu phân lớp giảm về tối đa: min(n_features, n_classes - 1) = 3 - 1 = 2 chiều
n_components_lda = min(n_components_pca, C_classes - 1)

print(f"✔ Tổng số chiều gốc của dữ liệu: {total_dims} chiều.")
print(f"✔ Giai đoạn 1 (PCAmix): Nén dữ liệu hỗn hợp về {n_components_pca} chiều liên tục.")
print(f"✔ Giai đoạn 2 (LDA)   : Tối ưu phân tách lớp, đưa về {n_components_lda} chiều đích (DISMIX_LD1, DISMIX_LD2).")
print("-" * 85)


# Duyệt qua các cấu hình đa tỷ lệ chia
for test_size, label, key in configs:

    print(f"\n Đang xử lý DISMIX: {label}...")

    # Bước 1: Chia tập dữ liệu hỗn hợp thô X_mixed trước khi thực hiện chuỗi DISMIX
    X_train_raw, X_test_raw, y_train, y_test = train_test_split_manual(
        X_mixed, y_classified, test_size=test_size, random_state=42
    )

    # Bước 2: Khởi tạo các lõi thành phần với cấu hình chiều đã tính sẵn bên ngoài
    core_pcamix = PCAmix_Manual(
        n_components=n_components_pca,
        discrete_features=discrete_indices,
        continuous_features=continuous_indices
    )
    core_lda = LDA_manual(n_components=n_components_lda)

    # Bước 3: Khởi tạo đường ống DISMIX tích hợp từ 2 lõi thủ công
    dismix_pipeline = DISMIX_manual(pcamix_instance=core_pcamix, lda_instance=core_lda)

    # Bước 4 & 5: Học, chiếu trên tập TRAIN và CHIẾU trên tập TEST
    # Tắt thông báo print nhỏ bên trong hàm fit của DISMIX cho log hiển thị được sạch đẹp
    with contextlib.redirect_stdout(open(os.devnull, 'w')):
        X_train_dismix = dismix_pipeline.fit_transform(X_train_raw, y_train)
        X_test_dismix = dismix_pipeline.transform(X_test_raw)

    # Bước 6: AUTO-TUNING TÌM THAM SỐ TỐT NHẤT TRÊN KHÔNG GIAN GIẢM CHIỀU DISMIX
    # Trích xuất mảng số .values từ DataFrame trước khi đưa vào hàm tuning nội bộ
    best_lam, best_eta = tune_softmax_hyperparameters(
        X_train_dismix.values, y_train, C=C_classes, lambda_grid=LAMBDA_GRID, eta_grid=ETA_GRID
    )

    # Bước 7: Chuyển vị ma trận (.T) để đưa về kích thước (d, N) chuẩn toán học Softmax thủ công
    X_train_T = X_train_dismix.values.T
    X_test_T = X_test_dismix.values.T

    # Bước 8: HUẤN LUYỆN LẠI trên toàn bộ X_train_T với bộ tham số tối ưu vừa tìm được
    W_fitted_dismix = softmax_regression_manual_l2(
        X_train_T, y_train, C=C_classes, eta=best_eta, lam=best_lam, max_count=5000
    )

    # Bước 9: Dự đoán và tính toán độ chính xác thực nghiệm trên tập Test độc lập
    y_pred_dismix = pred_manual(W_fitted_dismix, X_test_T)
    acc_dismix = accuracy_score(y_test, y_pred_dismix)

    # Lưu kết quả thực nghiệm vào từ điển global_results
    global_results[key]['Softmax_DISMIX'] = acc_dismix

    # Bước 10: Gọi hàm in báo cáo với đầy đủ 6 tham số để tránh lỗi TypeError hoàn toàn
    title = f"DISMIX ({n_components_lda}D) | lam={best_lam} | eta={best_eta} | Tỷ lệ {label}"
    display_detailed_metrics(
        y_true=y_test,
        y_pred=y_pred_dismix,
        label=label,
        len_train=len(y_train),
        len_test=len(y_test),
        title_label=title
    )

print("="*85)
print("✔ HOÀN TẤT CHẠY THỰC NGHIỆM DISMIX!")

### 4.2 Hybrid Naive Bayes

### 4.3 KNN